# Welcome to Modal notebooks!

Write Python code and collaborate in real time. Your code runs in Modal's
**serverless cloud**, and anyone in the same workspace can join.

This notebook comes with some common Python libraries installed. Run
cells with `Shift+Enter`.

In [1]:
%uv pip install -q datasets==3.6.0 pandas  kagglehub tqdm modal


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Complete notebook-friendly script: download HC3 (Hello-SimpleAI/HC3), normalize, save parquet via fastparquet
import sys, subprocess
from pathlib import Path
import json
from tqdm.auto import tqdm

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

# Install dependencies (fastparquet used for parquet IO to avoid pyarrow)
pip_install(["datasets>=2.20.0", "pandas>=2.0.0", "fastparquet", "tqdm"])

# Now imports (do NOT import pyarrow)
import pandas as pd
from datasets import load_dataset

# Output directory
DATA_ROOT = Path("/root/data").resolve()
HC3_DIR = DATA_ROOT / "hc3"
HC3_DIR.mkdir(parents=True, exist_ok=True)

def save_parquet(df: pd.DataFrame, out_path: Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(out_path, engine="fastparquet", index=False)
    print(f"Saved {out_path} ({len(df)} rows)")

def normalize_dataframe_from_dicts(records, text_candidates=("text","content","generated_text","sentence","answer","output")):
    df = pd.DataFrame.from_records(records)
    # ensure stable types
    if df.shape[0] == 0:
        return pd.DataFrame(columns=["text","label","meta"])
    # derive 'text' column if missing
    if "text" not in df.columns:
        for c in text_candidates:
            if c in df.columns:
                df["text"] = df[c].astype(str)
                break
        else:
            df["text"] = df.astype(str).agg(" ".join, axis=1)

    # try to find a label column
    label_col = None
    for c in ("label", "is_generated", "target", "y"):
        if c in df.columns:
            label_col = c
            break
    df["label"] = df[label_col] if label_col else None

    meta_cols = [c for c in df.columns if c not in ("text", "label")]
    # pack meta as JSON/dicts
    df["meta"] = df[meta_cols].to_dict(orient="records") if meta_cols else [{} for _ in range(len(df))]
    return df[["text","label","meta"]]

def dataset_split_to_records(hf_dataset_split, max_rows=None):
    # Iterate over the dataset split and yield dicts.
    # Using simple iteration avoids dataset.to_pandas() and any heavy pyarrow calls.
    records = []
    if max_rows is None:
        for ex in tqdm(hf_dataset_split, desc="iterating split"):
            records.append(ex)
    else:
        for i, ex in enumerate(hf_dataset_split):
            if i >= max_rows:
                break
            records.append(ex)
    return records

# Main: load HC3 and process
summary = {}
print("Loading Hello-SimpleAI/HC3 (config='all') from Hugging Face ...")
try:
    hc3 = load_dataset("Hello-SimpleAI/HC3", "all")
    for split_name, split_ds in hc3.items():
        print(f"\nProcessing split: {split_name}")
        # Convert to list of dicts (avoid to_pandas)
        records = dataset_split_to_records(split_ds)
        df_norm = normalize_dataframe_from_dicts(records)
        out_path = HC3_DIR / f"hc3_{split_name}.parquet"
        save_parquet(df_norm, out_path)
        summary[split_name] = len(df_norm)
    print("\nAll done.")
except Exception as e:
    # capture and print the error for debugging
    summary["error"] = str(e)
    print("ERROR while loading/processing HC3:\n", e)

print("\n=== SUMMARY ===")
print(json.dumps(summary, indent=2))
print("\nFiles in:", HC3_DIR)
for p in sorted(HC3_DIR.iterdir()):
    print(" -", p.name)



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Loading Hello-SimpleAI/HC3 (config='all') from Hugging Face ...


README.md: 0.00B [00:00, ?B/s]

HC3.py: 0.00B [00:00, ?B/s]

all/train/0000.parquet:   0%|          | 0.00/39.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/24322 [00:00<?, ? examples/s]


Processing split: train


iterating split:   0%|          | 0/24322 [00:00<?, ?it/s]

Saved /root/data/hc3/hc3_train.parquet (24322 rows)

All done.

=== SUMMARY ===
{
  "train": 24322
}

Files in: /root/data/hc3
 - hc3_train.parquet


In [3]:
import os
import requests
import zipfile
import io
import json

# ---------------------------
# Step 1: Download the dataset
# ---------------------------
url = "https://storage.googleapis.com/gresearch/dipper/dipper-training-data.zip"
os.makedirs("dipper_data", exist_ok=True)

print("Downloading dataset...")
response = requests.get(url)
response.raise_for_status()

# ---------------------------
# Step 2: Extract contents
# ---------------------------
print("Extracting...")
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    z.extractall("dipper_data")

# ---------------------------
# Step 3: Inspect top-level folders
# ---------------------------
for root, dirs, files in os.walk("dipper_data"):
    print(root)
    # Stop after showing first few paths
    if len(dirs) == 0 or "sents_1" in root:
        break

# ---------------------------
# Step 4: Find one JSONL file dynamically
# ---------------------------
jsonl_path = None
for root, _, files in os.walk("dipper_data"):
    for f in files:
        if f.endswith(".jsonl"):
            jsonl_path = os.path.join(root, f)
            break
    if jsonl_path:
        break

if jsonl_path:
    print(f"\nFound sample file:\n{jsonl_path}\n")
    
    # ---------------------------
    # Step 5: Read first few examples
    # ---------------------------
    with open(jsonl_path, "r") as f:
        for i, line in enumerate(f):
            example = json.loads(line)
            print(json.dumps(example, indent=2))
            if i >= 2:  # show first 3
                break

Extracting...
dipper_data
dipper_data/par3
dipper_data/par3/translator_pair
dipper_data/par3/translator_pair/sents_3


In [4]:
!ls "/root/data/hc3"
import os

def print_directory_tree(root_path, indent=""):
    # Print the current directory name
    print(indent + os.path.basename(root_path) + "/")
    
    # Get all items in the directory
    try:
        items = sorted(os.listdir(root_path))
    except PermissionError:
        print(indent + "  [Permission Denied]")
        return

    for item in items:
        item_path = os.path.join(root_path, item)
        if os.path.isdir(item_path):
            # Recursively print subdirectories
            print_directory_tree(item_path, indent + "    ")
        else:
            # Print files
            print(indent + "    " + item)

# Set your root directory
root_dir = "/root/dipper_data/par3"

# Print the tree
print_directory_tree(root_dir)


hc3_train.parquet
par3/
    gt_translator/
        sents_1/
            train_ctrl_ctx.tsv
            train_ctrl_no_ctx.tsv
            valid_ctrl_ctx.tsv
            valid_ctrl_ctx_all.tsv
            valid_ctrl_ctx_all_small.tsv
            valid_ctrl_no_ctx.tsv
            valid_ctrl_no_ctx_all.tsv
            valid_ctrl_no_ctx_all_small.tsv
        sents_10/
            train_ctrl_ctx.tsv
            train_ctrl_no_ctx.tsv
            valid_ctrl_ctx.tsv
            valid_ctrl_ctx_all.tsv
            valid_ctrl_no_ctx.tsv
            valid_ctrl_no_ctx_all.tsv
        sents_2/
            train_ctrl_ctx.tsv
            train_ctrl_no_ctx.tsv
            valid_ctrl_ctx.tsv
            valid_ctrl_ctx_all.tsv
            valid_ctrl_ctx_all_small.tsv
            valid_ctrl_no_ctx.tsv
            valid_ctrl_no_ctx_all.tsv
            valid_ctrl_no_ctx_all_small.tsv
        sents_3/
            train_ctrl_ctx.tsv
            train_ctrl_no_ctx.tsv
            valid_ctrl_ctx.tsv
            v

In [5]:
import pyarrow as pa

# Unregister previous extension type if it exists
try:
    pa.unregister_extension_type("pandas.period")
except Exception:
    pass

import pandas as pd
import glob

files = glob.glob("/root/data/hc3/*.parquet")

for file in files:
    print(f"\n--- {file} ---")
    df = pd.read_parquet(file, engine="pyarrow")
    print(df.head(10))



--- /root/data/hc3/hc3_train.parquet ---
                                                text label  \
0  0 Why is every book I hear about a " NY Times ...  None   
1  1 If salt is so bad for cars , why do we use i...  None   
2  2 Why do we still have SD TV channels when HD ...  None   
3  3 Why has nobody assassinated Kim Jong - un He...  None   
4  4 How was airplane technology able to advance ...  None   
5  5 Why do humans have different colored eyes ? ...  None   
6  6 Why I can not fabricate a religion that prev...  None   
7  7 What has changed that we frequently now thro...  None   
8  8 magic the gathering What is it . how popular...  None   
9  9 What are prions and are they a big deal ? My...  None   

                                                meta  
0  {"id":"0","question":"Why is every book I hear...  
1  {"id":"1","question":"If salt is so bad for ca...  
2  {"id":"2","question":"Why do we still have SD ...  
3  {"id":"3","question":"Why has nobody assassina...  


In [6]:
# ============================================================
# HC3 -> Dipper-like TSVs (2 columns) for paraphrase detection
# - Column 0: "lexical = NA, order = NA [<sent> Q </sent> ] HUMAN"
# - Column 1: AI (ChatGPT) answer
# Variants: no_ctx / ctx
# Robust to NumPy arrays inside dataframe cells
# ============================================================

from pathlib import Path
import pandas as pd
from datasets import load_dataset
import numpy as np

OUT_ROOT = Path("/root/dipper_data/hc3").resolve()
OUT_ROOT.mkdir(parents=True, exist_ok=True)

def to_list_safely(x):
    """Convert cell value to a list of strings with robust handling."""
    if x is None:
        return []
    # NaN check (works for floats/objects)
    try:
        if pd.isna(x):
            return []
    except Exception:
        pass

    # Already a list/tuple
    if isinstance(x, (list, tuple)):
        return [str(e).strip() for e in x if str(e).strip()]

    # Numpy array
    if isinstance(x, np.ndarray):
        return [str(e).strip() for e in x.tolist() if str(e).strip()]

    # Single scalar/string -> make it a single-item list
    s = str(x).strip()
    return [s] if s else []


def make_pairs_from_hc3_split(ds_split, add_context: bool, cartesian: bool = True):
    """
    Build (col0, col1) pairs:
      - col0: 'lexical = NA, order = NA' + optional '<sent> question </sent>' + human
      - col1: AI answer
    If cartesian=True: pairs every human answer with every AI answer per item.
    If cartesian=False: pairs by index (min length) for 1-1 pairing.
    """
    df = ds_split.to_pandas()
    rows = []

    # column name fallbacks
    human_col = "human_answers" if "human_answers" in df.columns else "human_answers"
    ai_col    = "chatgpt_answers" if "chatgpt_answers" in df.columns else "chatgpt_answers"
    q_col     = "question" if "question" in df.columns else None

    for _, r in df.iterrows():
        q = ""
        if q_col is not None and q_col in r:
            q = str(r[q_col]).strip()

        human_list = to_list_safely(r.get(human_col))
        ai_list = to_list_safely(r.get(ai_col))

        if not human_list or not ai_list:
            continue

        prefix = "lexical = NA, order = NA"

        if cartesian:
            # All combinations
            for h in human_list:
                for a in ai_list:
                    if add_context and q:
                        col0 = f"{prefix}  <sent> {q} </sent> {h}"
                    else:
                        col0 = f"{prefix} {h}"
                    rows.append((col0, a))
        else:
            # 1-1 pairing by index
            k = min(len(human_list), len(ai_list))
            for i in range(k):
                h = human_list[i]
                a = ai_list[i]
                if add_context and q:
                    col0 = f"{prefix}  <sent> {q} </sent> {h}"
                else:
                    col0 = f"{prefix} {h}"
                rows.append((col0, a))

    return pd.DataFrame(rows, columns=[0, 1])


def save_tsv(df: pd.DataFrame, path: Path, preview_rows: int = 10):
    path.parent.mkdir(parents=True, exist_ok=True)
    # Two columns, no header, tab-separated
    df.to_csv(path, sep="\t", header=False, index=False)
    print(f"\n📄 File: {path}")
    print("-" * 80)
    print(df.head(preview_rows))


# -----------------------------
# Load HC3 and write TSVs
# -----------------------------
print("\n⏬ Loading HC3 (Hello-SimpleAI/HC3, config='all') ...")
hc3 = load_dataset("Hello-SimpleAI/HC3", "all")

# Write for whatever splits exist (often only 'train')
for split in hc3.keys():
    print(f"\n💾 Processing split: {split}")
    out_dir = OUT_ROOT / split
    no_ctx_path = out_dir / "train_ctrl_no_ctx.tsv"
    ctx_path    = out_dir / "train_ctrl_ctx.tsv"

    # Build pairs (cartesian pairing keeps more pairs; set cartesian=False for 1-1)
    df_no_ctx = make_pairs_from_hc3_split(hc3[split], add_context=False, cartesian=True)
    df_ctx    = make_pairs_from_hc3_split(hc3[split], add_context=True,  cartesian=True)

    # Save & preview
    save_tsv(df_no_ctx, no_ctx_path, preview_rows=10)
    save_tsv(df_ctx, ctx_path, preview_rows=10)

print("\n✅ Done. Dipper-like TSVs written under:", OUT_ROOT)



⏬ Loading HC3 (Hello-SimpleAI/HC3, config='all') ...

💾 Processing split: train


/tmp/ipykernel_389/3544731626.py:23: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future this will result in an error. Use `array.size > 0` to check that an array is not empty.
  if pd.isna(x):



📄 File: /root/dipper_data/hc3/train/train_ctrl_no_ctx.tsv
--------------------------------------------------------------------------------
                                                   0  \
0  lexical = NA, order = NA Basically there are m...   
1  lexical = NA, order = NA If you 're hearing ab...   
2  lexical = NA, order = NA One reason is lots of...   
3  lexical = NA, order = NA salt is good for not ...   
4  lexical = NA, order = NA In Minnesota and Nort...   
5  lexical = NA, order = NA Used to work in the s...   
6  lexical = NA, order = NA The way it works is t...   
7  lexical = NA, order = NA HD does n't look like...   
8  lexical = NA, order = NA There are a few reaso...   
9  lexical = NA, order = NA You ca n't just go ar...   

                                                   1  
0  There are many different best seller lists tha...  
1  There are many different best seller lists tha...  
2  There are many different best seller lists tha...  
3  Salt is used on road

In [7]:
# # ============================================================
# # STAGE 1 — FAST STABLE DATA BUILDER
# # Paraphrase Detection Robustness
# # Output: /mnt/heirarchy/data_stage1
# # ============================================================

# import os
# import glob
# import json
# import random
# import time
# import hashlib
# import numpy as np
# import pandas as pd
# from sklearn.model_selection import train_test_split

# # ================= CONFIG =================

# DIPPER_ROOT = "/root/dipper_data/par3"
# OUTPUT_ROOT = "/mnt/heirarchy/data_stage1"

# SEED = 42
# MAX_POSITIVES = 800_000
# RANDOM_NEG_RATIO = 0.5
# TRAIN_RATIO = 0.8
# VAL_RATIO = 0.1

# MIN_LEN = 8

# os.makedirs(OUTPUT_ROOT, exist_ok=True)
# random.seed(SEED)
# np.random.seed(SEED)


# def normalize(x):
#     return " ".join(str(x).split()).strip()


# def pair_hash(s1, s2):
#     return hashlib.md5((s1 + "||" + s2).encode()).hexdigest()


# # ================= LOAD POSITIVES =================

# print("Scanning TSV files...", flush=True)
# tsv_files = glob.glob(os.path.join(DIPPER_ROOT, "**/*.tsv"), recursive=True)
# print("Found TSV:", len(tsv_files), flush=True)

# rows = []
# start_time = time.time()

# for idx, path in enumerate(tsv_files):
#     try:
#         df = pd.read_csv(path, sep="\t", header=None, on_bad_lines="skip")
#     except:
#         continue

#     if df.shape[1] < 2:
#         continue

#     for _, r in df.iterrows():
#         s1 = normalize(r[0])
#         s2 = normalize(r[1])
#         if len(s1) > MIN_LEN and len(s2) > MIN_LEN:
#             rows.append((s1, s2))

#     if (idx + 1) % 100 == 0:
#         print(f"Processed {idx+1} files, pairs {len(rows)}", flush=True)

# df_pos = pd.DataFrame(rows, columns=["sentence1", "sentence2"])
# df_pos["label"] = 1

# print("Total positives:", len(df_pos), flush=True)

# if len(df_pos) > MAX_POSITIVES:
#     df_pos = df_pos.sample(MAX_POSITIVES, random_state=SEED).reset_index(drop=True)
#     print("Capped positives to", len(df_pos), flush=True)

# # ================= RANDOM NEGATIVES =================

# print("Building random negatives...", flush=True)
# idx = np.random.permutation(len(df_pos))
# half = len(df_pos) // 2

# df_neg = pd.DataFrame({
#     "sentence1": df_pos.iloc[idx[:half]]["sentence1"].values,
#     "sentence2": df_pos.iloc[idx[-half:]]["sentence2"].values,
#     "label": 0
# })

# target = int(len(df_pos) * RANDOM_NEG_RATIO)
# df_neg = df_neg.sample(min(target, len(df_neg)), random_state=SEED)

# print("Random negatives:", len(df_neg), flush=True)

# # ================= COMBINE =================

# df_all = pd.concat([df_pos, df_neg], ignore_index=True)
# df_all = df_all.sample(frac=1, random_state=SEED).reset_index(drop=True)

# print("Total dataset:", len(df_all), flush=True)
# print("Label distribution:", df_all["label"].value_counts().to_dict(), flush=True)

# # ================= SPLIT =================

# train_df, temp_df = train_test_split(
#     df_all,
#     test_size=(1 - TRAIN_RATIO),
#     random_state=SEED,
#     stratify=df_all["label"]
# )

# val_ratio_adjusted = VAL_RATIO / (1 - TRAIN_RATIO)

# val_df, test_df = train_test_split(
#     temp_df,
#     test_size=(1 - val_ratio_adjusted),
#     random_state=SEED,
#     stratify=temp_df["label"]
# )

# print("Train:", len(train_df))
# print("Val:", len(val_df))
# print("Test:", len(test_df))

# # ================= SAVE =================

# train_df.to_csv(os.path.join(OUTPUT_ROOT, "train.csv"), index=False)
# val_df.to_csv(os.path.join(OUTPUT_ROOT, "val.csv"), index=False)
# test_df.to_csv(os.path.join(OUTPUT_ROOT, "test.csv"), index=False)

# manifest = {
#     "train": len(train_df),
#     "val": len(val_df),
#     "test": len(test_df),
#     "total": len(df_all),
#     "label_dist": df_all["label"].value_counts().to_dict()
# }

# with open(os.path.join(OUTPUT_ROOT, "manifest.json"), "w") as f:
#     json.dump(manifest, f, indent=2)

# print("Stage 1 complete in", round(time.time() - start_time, 2), "seconds")

In [8]:
import os
print(os.listdir("/mnt/heirarchy/data_stage1"))

['manifest.json', 'test.csv', 'train.csv', 'val.csv']


In [9]:
# ============================================================
# HIERARCHICAL DUAL ENCODER PARAPHRASE DETECTION
# End to end | Dimensional subspace split (NO L/3 token slicing)
# Dual encoder: DeBERTa v3 base + RoBERTa base
# Lexical + Syntactic attention pooling
# Semantic cross attention
# Gated bilinear fusion per level
# Auxiliary level heads + meta classifier
# AMP bf16 | grad accumulation | early stopping | OOM batch autoscale
# Progress bars | deterministic seeds | saves best model
# ============================================================

import os
import json
import time
import math
import random
import warnings
from dataclasses import dataclass, asdict
from typing import Dict, Tuple, Optional, List

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, matthews_corrcoef
from tqdm.auto import tqdm

# ----------------------------
# Noise suppression
# ----------------------------
warnings.filterwarnings("ignore", message="Be aware, overflowing tokens are not returned*")
warnings.filterwarnings("ignore", message="The sentencepiece tokenizer that you are converting*")
warnings.filterwarnings("ignore", message="Converting a tensor with requires_grad=True to a scalar*")
warnings.filterwarnings("ignore", category=FutureWarning)

# ============================================================
# CONFIG
# ============================================================

@dataclass
class CFG:
    data_root: str = "/mnt/heirarchy/data_stage1"
    out_root: str = "/mnt/heirarchy/models_stage2_hier"
    seed: int = 42

    # Models
    enc_a: str = "microsoft/deberta-v3-base"
    enc_b: str = "roberta-base"

    # Tokenization
    max_len: int = 128

    # Training
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    epochs: int = 3
    lr_enc: float = 2e-6
    lr_head: float = 1e-5
    weight_decay: float = 0.01
    grad_clip: float = 1.0

    # Batch and speed
    batch_size_start: int = 192
    num_workers: int = 8
    pin_memory: bool = True
    persistent_workers: bool = True
    grad_accum: int = 2
    use_compile: bool = True
    allow_tf32: bool = True

    # Hierarchical dimensions
    d_lex: int = 256
    d_syn: int = 256
    d_sem: int = 256

    # Loss weights
    main_w: float = 0.6
    aux_w: float = 0.4

    # Regularizers
    ortho_lambda: float = 0.02          # encourages subspace separation
    simplex_lambda: float = 0.01        # encourages information preservation

    # Early stopping
    patience: int = 2

    # Eval
    eval_every_steps: int = 0           # 0 means only end of epoch
    threshold: float = 0.5             # default, you can tune on val later

    # Checkpointing
    save_best_name: str = "best_model.pt"
    save_last_name: str = "last_model.pt"
    log_name: str = "train_log.json"

cfg = CFG()
os.makedirs(cfg.out_root, exist_ok=True)

# ============================================================
# Reproducibility
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)

if cfg.device.startswith("cuda"):
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
    torch.backends.cuda.matmul.allow_tf32 = cfg.allow_tf32

# ============================================================
# DATASET
# ============================================================

class PairDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer: AutoTokenizer, max_len: int):
        self.df = df.reset_index(drop=True)
        self.tok = tokenizer
        self.max_len = max_len

        required = {"sentence1", "sentence2", "label"}
        missing = required - set(self.df.columns)
        if missing:
            raise ValueError(f"Missing columns in CSV: {missing}. Need sentence1, sentence2, label.")

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        row = self.df.iloc[idx]
        enc = self.tok(
            str(row["sentence1"]),
            str(row["sentence2"]),
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(int(row["label"]), dtype=torch.long)
        return item

# ============================================================
# MODEL COMPONENTS
# ============================================================

class AdditiveAttnPool(nn.Module):
    def __init__(self, d_in: int, d_mid: int = 384):
        super().__init__()
        self.fc1 = nn.Linear(d_in, d_mid)
        self.fc2 = nn.Linear(d_mid, 1)

    def forward(self, H: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        # H: [B, L, d], mask: [B, L] where 1 is keep
        s = self.fc2(torch.tanh(self.fc1(H))).squeeze(-1)  # [B, L]
        s = s.masked_fill(mask == 0, -1e9)
        a = torch.softmax(s, dim=-1).unsqueeze(-1)         # [B, L, 1]
        return torch.sum(a * H, dim=1)                     # [B, d]

class CrossAttentionBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int = 8, dropout: float = 0.0):
        super().__init__()
        self.mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, dropout=dropout, batch_first=True)
        self.ln = nn.LayerNorm(d_model)

    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, key_padding_mask: torch.Tensor) -> torch.Tensor:
        # q,k,v: [B, L, d]
        # key_padding_mask: [B, L] True means ignore (PyTorch convention)
        attn_out, _ = self.mha(q, k, v, key_padding_mask=key_padding_mask, need_weights=False)
        return self.ln(q + attn_out)

class GatedBilinearFuse(nn.Module):
    def __init__(self, d: int):
        super().__init__()
        self.align_b = nn.Linear(d, d)
        self.bilin = nn.Bilinear(d, d, d)
        self.gate = nn.Sequential(
            nn.Linear(2 * d, d),
            nn.Sigmoid()
        )
        self.proj = nn.Linear(d, d)

    def forward(self, a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
        # a,b: [B, d]
        b2 = self.align_b(b)
        bil = self.bilin(a, b2)               # [B, d]
        g = self.gate(torch.cat([a, b2], dim=-1))
        fused = g * bil + (1 - g) * a
        return self.proj(fused)

def orthogonality_penalty(Slex: nn.Linear, Ssyn: nn.Linear, Ssem: nn.Linear) -> torch.Tensor:
    # Use projection weight matrices to encourage subspace separation.
    # Weight shapes: [d_out, d_in]. Use normalized Frobenius inner products.
    Wl = Slex.weight
    Ws = Ssyn.weight
    Wm = Ssem.weight

    def normed_dot(A, B):
        return torch.sum(A * B) / (torch.norm(A) * torch.norm(B) + 1e-8)

    return (normed_dot(Wl, Ws).abs() + normed_dot(Wl, Wm).abs() + normed_dot(Ws, Wm).abs()) / 3.0

def simplex_penalty(Slex: nn.Linear, Ssyn: nn.Linear, Ssem: nn.Linear, d_in: int) -> torch.Tensor:
    # Encourage approximate information preservation:
    # Sum over level projection back projections approximates identity on average.
    # This is a soft constraint, not a hard guarantee.
    Wl = Slex.weight   # [d_lex, d_in]
    Ws = Ssyn.weight
    Wm = Ssem.weight
    # Build d_in x d_in matrix approx
    A = Wl.t() @ Wl + Ws.t() @ Ws + Wm.t() @ Wm
    I = torch.eye(d_in, device=A.device, dtype=A.dtype)
    return torch.mean((A - I) ** 2)

class HierDualEncoder(nn.Module):
    def __init__(self, enc_a_name: str, enc_b_name: str, d_lex: int, d_syn: int, d_sem: int):
        super().__init__()
        self.enc_a = AutoModel.from_pretrained(enc_a_name)
        self.enc_b = AutoModel.from_pretrained(enc_b_name)

        da = self.enc_a.config.hidden_size
        db = self.enc_b.config.hidden_size
        if da != db:
            raise ValueError(f"Hidden sizes must match for simple fusion. Got {da} and {db}.")

        self.d_in = da
        self.d_lex = d_lex
        self.d_syn = d_syn
        self.d_sem = d_sem

        # Dimensional subspace projections: Hlex = H @ Slex^T etc via Linear on last dim
        self.Slex = nn.Linear(self.d_in, d_lex, bias=False)
        self.Ssyn = nn.Linear(self.d_in, d_syn, bias=False)
        self.Ssem = nn.Linear(self.d_in, d_sem, bias=False)

        # Pools for lex and syn on each encoder
        self.pool_lex = AdditiveAttnPool(d_lex)
        self.pool_syn = AdditiveAttnPool(d_syn)

        # Semantic cross attention in subspace
        self.cross12 = CrossAttentionBlock(d_sem, n_heads=8)
        self.cross21 = CrossAttentionBlock(d_sem, n_heads=8)
        self.pool_sem = AdditiveAttnPool(d_sem)

        # Level fusers
        self.fuse_lex = GatedBilinearFuse(d_lex)
        self.fuse_syn = GatedBilinearFuse(d_syn)
        self.fuse_sem = GatedBilinearFuse(d_sem)

        # Auxiliary heads per level
        self.head_lex = nn.Linear(d_lex, 2)
        self.head_syn = nn.Linear(d_syn, 2)
        self.head_sem = nn.Linear(d_sem, 2)

        # Meta classifier consumes auxiliary logits and fused vectors
        meta_in = (2 + 2 + 2) + (d_lex + d_syn + d_sem)
        self.meta = nn.Sequential(
            nn.Linear(meta_in, 512),
            nn.GELU(),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Linear(256, 2)
        )

    def encode(self, encoder: AutoModel, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        out = encoder(input_ids=input_ids, attention_mask=attention_mask)
        return out.last_hidden_state  # [B, L, d]

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> Dict[str, torch.Tensor]:
        # Both encoders see the pair jointly (cross encoder style with [SEP] etc)
        Ha = self.encode(self.enc_a, input_ids, attention_mask)  # [B,L,d]
        Hb = self.encode(self.enc_b, input_ids, attention_mask)

        # Subspace projections at token level
        Ha_lex = self.Slex(Ha)
        Hb_lex = self.Slex(Hb)
        Ha_syn = self.Ssyn(Ha)
        Hb_syn = self.Ssyn(Hb)
        Ha_sem = self.Ssem(Ha)
        Hb_sem = self.Ssem(Hb)

        # Mask for pooling: 1 keep, 0 pad
        mask = attention_mask  # [B,L]
        key_padding_mask = (mask == 0)  # PyTorch expects True for pads

        # Lex and syn pooled vectors for each encoder
        fa_lex = self.pool_lex(Ha_lex, mask)  # [B,d_lex]
        fb_lex = self.pool_lex(Hb_lex, mask)
        fa_syn = self.pool_syn(Ha_syn, mask)
        fb_syn = self.pool_syn(Hb_syn, mask)

        # Semantic cross attention between encoders in semantic subspace
        # q from A attends to B and vice versa, then pooled
        Ha_sem_x = self.cross12(Ha_sem, Hb_sem, Hb_sem, key_padding_mask=key_padding_mask)
        Hb_sem_x = self.cross21(Hb_sem, Ha_sem, Ha_sem, key_padding_mask=key_padding_mask)
        fa_sem = self.pool_sem(Ha_sem_x, mask)  # [B,d_sem]
        fb_sem = self.pool_sem(Hb_sem_x, mask)

        # Fuse per level
        z_lex = self.fuse_lex(fa_lex, fb_lex)
        z_syn = self.fuse_syn(fa_syn, fb_syn)
        z_sem = self.fuse_sem(fa_sem, fb_sem)

        # Auxiliary logits
        log_lex = self.head_lex(z_lex)
        log_syn = self.head_syn(z_syn)
        log_sem = self.head_sem(z_sem)

        # Meta
        meta_in = torch.cat([log_lex, log_syn, log_sem, z_lex, z_syn, z_sem], dim=-1)
        log_meta = self.meta(meta_in)

        return {
            "log_meta": log_meta,
            "log_lex": log_lex,
            "log_syn": log_syn,
            "log_sem": log_sem
        }

# ============================================================
# METRICS
# ============================================================

@torch.no_grad()
def compute_metrics_from_logits(logits: np.ndarray, labels: np.ndarray, threshold: float = 0.5) -> Dict[str, float]:
    # logits: [N,2]
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]
    pred = (probs >= threshold).astype(np.int32)
    return {
        "f1": float(f1_score(labels, pred)),
        "acc": float(accuracy_score(labels, pred)),
        "auc": float(roc_auc_score(labels, probs)),
        "mcc": float(matthews_corrcoef(labels, pred))
    }

@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: str, threshold: float) -> Dict[str, float]:
    model.eval()
    all_logits = []
    all_labels = []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(batch["input_ids"], batch["attention_mask"])
        all_logits.append(out["log_meta"].detach().float().cpu().numpy())
        all_labels.append(batch["labels"].detach().cpu().numpy())
    logits = np.concatenate(all_logits, axis=0)
    labels = np.concatenate(all_labels, axis=0)
    return compute_metrics_from_logits(logits, labels, threshold=threshold)

# ============================================================
# OPTIMIZER PARAM GROUPS
# ============================================================

def build_optimizer(model: HierDualEncoder, cfg: CFG) -> torch.optim.Optimizer:
    # Separate encoder params and head params with different LR
    enc_params = []
    head_params = []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if n.startswith("enc_a.") or n.startswith("enc_b."):
            enc_params.append(p)
        else:
            head_params.append(p)

    return torch.optim.AdamW(
        [
            {"params": enc_params, "lr": cfg.lr_enc},
            {"params": head_params, "lr": cfg.lr_head},
        ],
        weight_decay=cfg.weight_decay
    )

# ============================================================
# TRAINING
# ============================================================

def train_one_run(cfg: CFG) -> None:
    train_path = os.path.join(cfg.data_root, "train.csv")
    val_path = os.path.join(cfg.data_root, "val.csv")
    test_path = os.path.join(cfg.data_root, "test.csv")

    if not os.path.exists(train_path) or not os.path.exists(val_path):
        raise FileNotFoundError(f"Expected train.csv and val.csv in {cfg.data_root}")

    train_df = pd.read_csv(train_path)
    val_df = pd.read_csv(val_path)
    test_df = pd.read_csv(test_path) if os.path.exists(test_path) else None

    # Use DeBERTa tokenizer for pair encoding. RoBERTa will still accept input_ids from this tokenizer?
    # No. Tokenizers differ. For correctness, use ONE tokenizer and ONE vocab.
    # Best practice: pick one tokenizer and feed both encoders the same ids only if both share vocab, which they do not.
    # Therefore, we build TWO tokenizers and TWO input streams, and adapt model forward accordingly.
    #
    # To keep this script practical and correct, we instead use ONE encoder family for pair tokenization for BOTH
    # encoders by selecting encoders that share the same tokenizer. DeBERTa and RoBERTa do not.
    #
    # If you want true dual encoder with different tokenizers, you must tokenize twice and pass two input_id tensors.
    # The model below implements the correct dual tokenizer approach.

    tok_a = AutoTokenizer.from_pretrained(cfg.enc_a, use_fast=True)
    tok_b = AutoTokenizer.from_pretrained(cfg.enc_b, use_fast=True)

    class DualTokPairDataset(Dataset):
        def __init__(self, df: pd.DataFrame, tok_a, tok_b, max_len: int):
            self.df = df.reset_index(drop=True)
            self.ta = tok_a
            self.tb = tok_b
            self.max_len = max_len

        def __len__(self):
            return len(self.df)

        def __getitem__(self, idx):
            r = self.df.iloc[idx]
            s1 = str(r["sentence1"])
            s2 = str(r["sentence2"])
            ea = self.ta(s1, s2, truncation=True, padding="max_length", max_length=self.max_len, return_tensors="pt")
            eb = self.tb(s1, s2, truncation=True, padding="max_length", max_length=self.max_len, return_tensors="pt")
            item = {
                "input_ids_a": ea["input_ids"].squeeze(0),
                "attention_mask_a": ea["attention_mask"].squeeze(0),
                "input_ids_b": eb["input_ids"].squeeze(0),
                "attention_mask_b": eb["attention_mask"].squeeze(0),
                "labels": torch.tensor(int(r["label"]), dtype=torch.long),
            }
            return item

    def collate_fn(batch_list: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
        out = {}
        for k in batch_list[0].keys():
            out[k] = torch.stack([b[k] for b in batch_list], dim=0)
        return out

    # Patch model to accept dual token streams
    class HierDualEncoderDualTok(HierDualEncoder):
        def forward(self, input_ids_a, attention_mask_a, input_ids_b, attention_mask_b) -> Dict[str, torch.Tensor]:
            Ha = self.encode(self.enc_a, input_ids_a, attention_mask_a)
            Hb = self.encode(self.enc_b, input_ids_b, attention_mask_b)

            # Use respective masks. For pooling and cross attention, we use A mask as reference length.
            # To keep shapes consistent, enforce same max_len in both tokenizers, which we do.
            mask_a = attention_mask_a
            mask_b = attention_mask_b
            key_padding_mask_a = (mask_a == 0)
            key_padding_mask_b = (mask_b == 0)

            Ha_lex = self.Slex(Ha); Hb_lex = self.Slex(Hb)
            Ha_syn = self.Ssyn(Ha); Hb_syn = self.Ssyn(Hb)
            Ha_sem = self.Ssem(Ha); Hb_sem = self.Ssem(Hb)

            fa_lex = self.pool_lex(Ha_lex, mask_a)
            fb_lex = self.pool_lex(Hb_lex, mask_b)
            fa_syn = self.pool_syn(Ha_syn, mask_a)
            fb_syn = self.pool_syn(Hb_syn, mask_b)

            # Cross attention requires matching embed dim, not matching tokenization.
            # We attend A tokens over B tokens and vice versa using their own pad masks.
            Ha_sem_x = self.cross12(Ha_sem, Hb_sem, Hb_sem, key_padding_mask=key_padding_mask_b)
            Hb_sem_x = self.cross21(Hb_sem, Ha_sem, Ha_sem, key_padding_mask=key_padding_mask_a)

            fa_sem = self.pool_sem(Ha_sem_x, mask_a)
            fb_sem = self.pool_sem(Hb_sem_x, mask_b)

            z_lex = self.fuse_lex(fa_lex, fb_lex)
            z_syn = self.fuse_syn(fa_syn, fb_syn)
            z_sem = self.fuse_sem(fa_sem, fb_sem)

            log_lex = self.head_lex(z_lex)
            log_syn = self.head_syn(z_syn)
            log_sem = self.head_sem(z_sem)

            meta_in = torch.cat([log_lex, log_syn, log_sem, z_lex, z_syn, z_sem], dim=-1)
            log_meta = self.meta(meta_in)

            return {"log_meta": log_meta, "log_lex": log_lex, "log_syn": log_syn, "log_sem": log_sem}

    # Data loaders
    def make_loader(df, batch_size, shuffle):
        ds = DualTokPairDataset(df, tok_a, tok_b, cfg.max_len)
        return DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=cfg.num_workers,
            pin_memory=cfg.pin_memory,
            persistent_workers=cfg.persistent_workers,
            collate_fn=collate_fn
        )

    # Dynamic batch autoscale on OOM
    batch_size = cfg.batch_size_start
    while batch_size >= 1:
        try:
            print(f"Trying batch size {batch_size}  grad_accum {cfg.grad_accum}  effective {batch_size * cfg.grad_accum}")

            train_loader = make_loader(train_df, batch_size, shuffle=True)
            val_loader = make_loader(val_df, batch_size, shuffle=False)
            test_loader = make_loader(test_df, batch_size, shuffle=False) if test_df is not None else None

            model = HierDualEncoderDualTok(cfg.enc_a, cfg.enc_b, cfg.d_lex, cfg.d_syn, cfg.d_sem).to(cfg.device)

            if cfg.use_compile and cfg.device.startswith("cuda"):
                model = torch.compile(model)

            opt = build_optimizer(model, cfg)

            total_steps = math.ceil(len(train_loader) / 1) * cfg.epochs
            warmup = int(0.1 * total_steps)
            sch = get_cosine_schedule_with_warmup(opt, warmup, total_steps)

            best_f1 = -1.0
            patience_ctr = 0
            history = []

            global_step = 0
            for epoch in range(cfg.epochs):
                model.train()
                opt.zero_grad(set_to_none=True)

                pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg.epochs}", leave=True)
                running = {"loss": 0.0, "main": 0.0, "aux": 0.0, "ortho": 0.0, "simplex": 0.0}
                denom = 0

                for step, batch in enumerate(pbar):
                    batch = {k: v.to(cfg.device, non_blocking=True) for k, v in batch.items()}

                    with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=cfg.device.startswith("cuda")):
                        out = model(
                            batch["input_ids_a"], batch["attention_mask_a"],
                            batch["input_ids_b"], batch["attention_mask_b"]
                        )
                        y = batch["labels"]

                        loss_main = F.cross_entropy(out["log_meta"], y)
                        loss_lex = F.cross_entropy(out["log_lex"], y)
                        loss_syn = F.cross_entropy(out["log_syn"], y)
                        loss_sem = F.cross_entropy(out["log_sem"], y)
                        loss_aux = (loss_lex + loss_syn + loss_sem) / 3.0

                        ortho = orthogonality_penalty(model.Slex, model.Ssyn, model.Ssem)
                        simp = simplex_penalty(model.Slex, model.Ssyn, model.Ssem, model.d_in)

                        loss = cfg.main_w * loss_main + cfg.aux_w * loss_aux
                        loss = loss + cfg.ortho_lambda * ortho + cfg.simplex_lambda * simp

                        loss = loss / cfg.grad_accum

                    loss.backward()

                    if (step + 1) % cfg.grad_accum == 0:
                        if cfg.grad_clip and cfg.grad_clip > 0:
                            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                        opt.step()
                        sch.step()
                        opt.zero_grad(set_to_none=True)
                        global_step += 1

                    # Logging
                    denom += 1
                    running["loss"] += float(loss.detach() * cfg.grad_accum)
                    running["main"] += float(loss_main.detach())
                    running["aux"] += float(loss_aux.detach())
                    running["ortho"] += float(ortho.detach())
                    running["simplex"] += float(simp.detach())

                    if step % 50 == 0:
                        pbar.set_postfix(
                            loss=running["loss"] / denom,
                            main=running["main"] / denom,
                            aux=running["aux"] / denom,
                            ortho=running["ortho"] / denom,
                            simp=running["simplex"] / denom,
                            lr=sch.get_last_lr()[0]
                        )

                # End epoch eval
                val_metrics = eval_dualtok(model, val_loader, cfg.device, cfg.threshold)
                epoch_log = {"epoch": epoch + 1, "val": val_metrics}
                history.append(epoch_log)

                print(
                    f"\nEpoch {epoch+1}  "
                    f"F1 {val_metrics['f1']:.4f}  "
                    f"Acc {val_metrics['acc']:.4f}  "
                    f"AUC {val_metrics['auc']:.4f}  "
                    f"MCC {val_metrics['mcc']:.4f}\n"
                )

                # Save last
                torch.save(model.state_dict(), os.path.join(cfg.out_root, cfg.save_last_name))

                # Save best
                if val_metrics["f1"] > best_f1:
                    best_f1 = val_metrics["f1"]
                    patience_ctr = 0
                    torch.save(model.state_dict(), os.path.join(cfg.out_root, cfg.save_best_name))
                else:
                    patience_ctr += 1
                    if patience_ctr >= cfg.patience:
                        print("Early stopping")
                        break

            # Optional test evaluation
            if test_loader is not None:
                # Load best
                best_path = os.path.join(cfg.out_root, cfg.save_best_name)
                if os.path.exists(best_path):
                    model.load_state_dict(torch.load(best_path, map_location=cfg.device))
                test_metrics = eval_dualtok(model, test_loader, cfg.device, cfg.threshold)
                print(
                    f"Test  F1 {test_metrics['f1']:.4f}  "
                    f"Acc {test_metrics['acc']:.4f}  "
                    f"AUC {test_metrics['auc']:.4f}  "
                    f"MCC {test_metrics['mcc']:.4f}"
                )
                history.append({"test": test_metrics})

            # Save logs
            with open(os.path.join(cfg.out_root, cfg.log_name), "w") as f:
                json.dump({"cfg": asdict(cfg), "history": history}, f, indent=2)

            print("Training complete")
            return

        except RuntimeError as e:
            msg = str(e).lower()
            if "out of memory" in msg or "cuda oom" in msg:
                print("OOM. Reducing batch size.")
                torch.cuda.empty_cache()
                batch_size = batch_size // 2
                continue
            raise

@torch.no_grad()
def eval_dualtok(model: nn.Module, loader: DataLoader, device: str, threshold: float) -> Dict[str, float]:
    model.eval()
    logits_list = []
    labels_list = []
    for batch in loader:
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=device.startswith("cuda")):
            out = model(
                batch["input_ids_a"], batch["attention_mask_a"],
                batch["input_ids_b"], batch["attention_mask_b"]
            )
        logits_list.append(out["log_meta"].detach().float().cpu().numpy())
        labels_list.append(batch["labels"].detach().cpu().numpy())
    logits = np.concatenate(logits_list, axis=0)
    labels = np.concatenate(labels_list, axis=0)
    return compute_metrics_from_logits(logits, labels, threshold=threshold)

if __name__ == "__main__":
    train_one_run(cfg)

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

...


Epoch 1  F1 0.9745  Acc 0.9659  AUC 0.9949  MCC 0.9230



Epoch 2/3:   0%|          | 0/5000 [00:00<?, ?it/s]


Epoch 2  F1 0.9812  Acc 0.9749  AUC 0.9970  MCC 0.9434



Epoch 3/3:   0%|          | 0/5000 [00:00<?, ?it/s]


Epoch 3  F1 0.9842  Acc 0.9789  AUC 0.9977  MCC 0.9524

Test  F1 0.9846  Acc 0.9794  AUC 0.9978  MCC 0.9536
Training complete


In [18]:
# ============================================================
# FAST + VERBOSE EVALUATION SCRIPT
# ============================================================

import os
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from transformers import AutoModel, AutoTokenizer
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    matthews_corrcoef, precision_score, recall_score,
    roc_curve, precision_recall_curve
)

# ---------------- CONFIG ----------------

DATA_ROOT = "/mnt/heirarchy/data_stage1"
MODEL_ROOT = "/mnt/heirarchy/models_stage2_hier"
BEST_PATH = os.path.join(MODEL_ROOT, "best_model.pt")

ENC_A = "microsoft/deberta-v3-base"
ENC_B = "roberta-base"

D_LEX = 256
D_SYN = 256
D_SEM = 256
MAX_LEN = 128
BATCH_SIZE = 128

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
print("Loading model...")

# ============================================================
# MODEL DEFINITION
# ============================================================

class AdditiveAttnPool(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.fc1 = nn.Linear(d, 384)
        self.fc2 = nn.Linear(384, 1)

    def forward(self, H, mask):
        s = self.fc2(torch.tanh(self.fc1(H))).squeeze(-1)
        s = s.masked_fill(mask == 0, -1e9)
        a = torch.softmax(s, dim=-1).unsqueeze(-1)
        return torch.sum(a * H, dim=1)

class CrossAttentionBlock(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.mha = nn.MultiheadAttention(d, 8, batch_first=True)
        self.ln = nn.LayerNorm(d)

    def forward(self, q, k, v, pad_mask):
        out,_ = self.mha(q,k,v,key_padding_mask=pad_mask)
        return self.ln(q + out)

class GatedBilinearFuse(nn.Module):
    def __init__(self,d):
        super().__init__()
        self.align_b = nn.Linear(d,d)
        self.bilin = nn.Bilinear(d,d,d)
        self.gate = nn.Sequential(nn.Linear(2*d,d), nn.Sigmoid())
        self.proj = nn.Linear(d,d)

    def forward(self,a,b):
        b2 = self.align_b(b)
        bil = self.bilin(a,b2)
        g = self.gate(torch.cat([a,b2],dim=-1))
        return self.proj(g*bil + (1-g)*a)

class HierDualEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc_a = AutoModel.from_pretrained(ENC_A)
        self.enc_b = AutoModel.from_pretrained(ENC_B)

        d = self.enc_a.config.hidden_size

        self.Slex = nn.Linear(d,D_LEX,bias=False)
        self.Ssyn = nn.Linear(d,D_SYN,bias=False)
        self.Ssem = nn.Linear(d,D_SEM,bias=False)

        self.pool_lex = AdditiveAttnPool(D_LEX)
        self.pool_syn = AdditiveAttnPool(D_SYN)
        self.pool_sem = AdditiveAttnPool(D_SEM)

        self.cross12 = CrossAttentionBlock(D_SEM)
        self.cross21 = CrossAttentionBlock(D_SEM)

        self.fuse_lex = GatedBilinearFuse(D_LEX)
        self.fuse_syn = GatedBilinearFuse(D_SYN)
        self.fuse_sem = GatedBilinearFuse(D_SEM)

        self.head_lex = nn.Linear(D_LEX,2)
        self.head_syn = nn.Linear(D_SYN,2)
        self.head_sem = nn.Linear(D_SEM,2)

        meta_in = 6 + D_LEX + D_SYN + D_SEM
        self.meta = nn.Sequential(
            nn.Linear(meta_in,512),
            nn.GELU(),
            nn.Linear(512,256),
            nn.GELU(),
            nn.Linear(256,2)
        )

    def encode(self, enc, ids, mask):
        return enc(input_ids=ids, attention_mask=mask).last_hidden_state

    def forward(self, ids_a, mask_a, ids_b, mask_b):
        Ha = self.encode(self.enc_a, ids_a, mask_a)
        Hb = self.encode(self.enc_b, ids_b, mask_b)

        Ha_lex,Hb_lex = self.Slex(Ha),self.Slex(Hb)
        Ha_syn,Hb_syn = self.Ssyn(Ha),self.Ssyn(Hb)
        Ha_sem,Hb_sem = self.Ssem(Ha),self.Ssem(Hb)

        fa_lex = self.pool_lex(Ha_lex,mask_a)
        fb_lex = self.pool_lex(Hb_lex,mask_b)
        fa_syn = self.pool_syn(Ha_syn,mask_a)
        fb_syn = self.pool_syn(Hb_syn,mask_b)

        Ha_sem_x = self.cross12(Ha_sem,Hb_sem,Hb_sem,mask_b==0)
        Hb_sem_x = self.cross21(Hb_sem,Ha_sem,Ha_sem,mask_a==0)

        fa_sem = self.pool_sem(Ha_sem_x,mask_a)
        fb_sem = self.pool_sem(Hb_sem_x,mask_b)

        z_lex = self.fuse_lex(fa_lex,fb_lex)
        z_syn = self.fuse_syn(fa_syn,fb_syn)
        z_sem = self.fuse_sem(fa_sem,fb_sem)

        log_lex = self.head_lex(z_lex)
        log_syn = self.head_syn(z_syn)
        log_sem = self.head_sem(z_sem)

        meta_in = torch.cat([log_lex,log_syn,log_sem,z_lex,z_syn,z_sem],dim=-1)
        return self.meta(meta_in)

# ============================================================
# LOAD CHECKPOINT
# ============================================================

model = HierDualEncoder().to(DEVICE)

print("Loading checkpoint...")
ckpt = torch.load(BEST_PATH, map_location=DEVICE)

clean_ckpt = {}
for k,v in ckpt.items():
    clean_ckpt[k.replace("_orig_mod.","")] = v

model.load_state_dict(clean_ckpt, strict=True)
model.eval()

print("Checkpoint loaded.")

# ============================================================
# DATA
# ============================================================

print("Loading data...")
val_df = pd.read_csv(os.path.join(DATA_ROOT,"val.csv"))
test_df = pd.read_csv(os.path.join(DATA_ROOT,"test.csv"))

tok_a = AutoTokenizer.from_pretrained(ENC_A)
tok_b = AutoTokenizer.from_pretrained(ENC_B)

class PairDS(Dataset):
    def __init__(self, df): self.df=df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self,idx):
        r=self.df.iloc[idx]
        ea=tok_a(str(r["sentence1"]),str(r["sentence2"]),
                 truncation=True,padding="max_length",
                 max_length=MAX_LEN,return_tensors="pt")
        eb=tok_b(str(r["sentence1"]),str(r["sentence2"]),
                 truncation=True,padding="max_length",
                 max_length=MAX_LEN,return_tensors="pt")
        return {
            "ids_a":ea["input_ids"].squeeze(0),
            "mask_a":ea["attention_mask"].squeeze(0),
            "ids_b":eb["input_ids"].squeeze(0),
            "mask_b":eb["attention_mask"].squeeze(0),
            "label":torch.tensor(int(r["label"]))
        }

val_loader = DataLoader(PairDS(val_df), batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader = DataLoader(PairDS(test_df), batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# ============================================================
# COLLECT LOGITS WITH PROGRESS
# ============================================================

def collect(loader,name):
    logits=[]
    labels=[]
    start=time.time()
    with torch.inference_mode():
        for i,b in enumerate(loader):
            b={k:v.to(DEVICE) for k,v in b.items()}
            out=model(b["ids_a"],b["mask_a"],b["ids_b"],b["mask_b"])
            logits.append(out.cpu())
            labels.append(b["label"].cpu())
            if i%20==0:
                print(f"{name} batch {i}/{len(loader)}  elapsed {time.time()-start:.1f}s")
    print(f"{name} finished in {time.time()-start:.1f}s")
    return torch.cat(logits),torch.cat(labels)

print("Running validation inference...")
val_logits,val_labels=collect(val_loader,"VAL")

print("Running test inference...")
test_logits,test_labels=collect(test_loader,"TEST")

# ============================================================
# THRESHOLD TUNING
# ============================================================

print("Tuning threshold...")
val_probs=torch.softmax(val_logits,dim=1)[:,1].numpy()

best_thr=0.5
best_f1=-1
for t in np.linspace(0.05,0.95,181):
    f=f1_score(val_labels.numpy(),(val_probs>=t).astype(int))
    if f>best_f1:
        best_f1=f
        best_thr=t

print("Best threshold:",best_thr)

# ============================================================
# FINAL METRICS
# ============================================================

probs=torch.softmax(test_logits,dim=1)[:,1].numpy()
preds=(probs>=best_thr).astype(int)

metrics={
    "F1":f1_score(test_labels.numpy(),preds),
    "Accuracy":accuracy_score(test_labels.numpy(),preds),
    "Precision":precision_score(test_labels.numpy(),preds),
    "Recall":recall_score(test_labels.numpy(),preds),
    "AUC":roc_auc_score(test_labels.numpy(),probs),
    "MCC":matthews_corrcoef(test_labels.numpy(),preds)
}

print("\n===== FINAL TEST RESULTS =====")
for k,v in metrics.items():
    print(f"{k}: {v:.4f}")

# ============================================================
# SAVE CURVES
# ============================================================

fpr,tpr,_=roc_curve(test_labels,probs)
plt.figure()
plt.plot(fpr,tpr)
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC Curve")
plt.savefig(os.path.join(MODEL_ROOT,"roc_curve.png"))
plt.close()

prec,rec,_=precision_recall_curve(test_labels,probs)
plt.figure()
plt.plot(rec,prec)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("PR Curve")
plt.savefig(os.path.join(MODEL_ROOT,"pr_curve.png"))
plt.close()

print("All plots saved.")

Device: cuda
Loading model...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading checkpoint...
Checkpoint loaded.
Loading data...
Running validation inference...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

VAL batch 0/938  elapsed 5.1s
VAL batch 20/938  elapsed 6.8s
VAL batch 40/938  elapsed 8.7s
VAL batch 60/938  elapsed 10.7s
VAL batch 80/938  elapsed 13.0s
VAL batch 100/938  elapsed 15.1s
VAL batch 120/938  elapsed 17.1s
VAL batch 140/938  elapsed 19.0s
VAL batch 160/938  elapsed 21.1s
VAL batch 180/938  elapsed 23.1s
VAL batch 200/938  elapsed 25.2s
VAL batch 220/938  elapsed 27.2s
VAL batch 240/938  elapsed 29.2s
VAL batch 260/938  elapsed 31.2s
VAL batch 280/938  elapsed 33.2s
VAL batch 300/938  elapsed 35.3s
VAL batch 320/938  elapsed 37.4s
VAL batch 340/938  elapsed 39.4s
VAL batch 360/938  elapsed 41.6s
VAL batch 380/938  elapsed 43.4s
VAL batch 400/938  elapsed 45.4s
VAL batch 420/938  elapsed 47.3s
VAL batch 440/938  elapsed 49.3s
VAL batch 460/938  elapsed 51.2s
VAL batch 480/938  elapsed 53.2s
VAL batch 500/938  elapsed 55.1s
VAL batch 520/938  elapsed 57.2s
VAL batch 540/938  elapsed 59.2s
VAL batch 560/938  elapsed 61.4s
VAL batch 580/938  elapsed 63.3s
VAL batch 600/938  

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

TEST batch 0/938  elapsed 3.6s
TEST batch 20/938  elapsed 5.5s
TEST batch 40/938  elapsed 7.5s
TEST batch 60/938  elapsed 9.5s
TEST batch 80/938  elapsed 11.6s
TEST batch 100/938  elapsed 13.6s
TEST batch 120/938  elapsed 15.5s
TEST batch 140/938  elapsed 17.5s
TEST batch 160/938  elapsed 19.7s
TEST batch 180/938  elapsed 21.8s
TEST batch 200/938  elapsed 23.7s
TEST batch 220/938  elapsed 25.6s
TEST batch 240/938  elapsed 27.7s
TEST batch 260/938  elapsed 29.7s
TEST batch 280/938  elapsed 31.5s
TEST batch 300/938  elapsed 33.4s
TEST batch 320/938  elapsed 35.3s
TEST batch 340/938  elapsed 37.3s
TEST batch 360/938  elapsed 39.3s
TEST batch 380/938  elapsed 41.3s
TEST batch 400/938  elapsed 43.3s
TEST batch 420/938  elapsed 45.2s
TEST batch 440/938  elapsed 47.3s
TEST batch 460/938  elapsed 49.1s
TEST batch 480/938  elapsed 51.1s
TEST batch 500/938  elapsed 53.0s
TEST batch 520/938  elapsed 55.0s
TEST batch 540/938  elapsed 56.8s
TEST batch 560/938  elapsed 58.9s
TEST batch 580/938  elap

In [19]:
import matplotlib.pyplot as plt

def compute_all_metrics(logits, labels, threshold):
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]
    preds = (probs >= threshold).astype(int)

    metrics = {
        "F1": f1_score(labels, preds),
        "Accuracy": accuracy_score(labels, preds),
        "Precision": precision_score(labels, preds),
        "Recall": recall_score(labels, preds),
        "AUC": roc_auc_score(labels, probs),
        "MCC": matthews_corrcoef(labels, preds),
    }
    return metrics, probs

test_metrics, test_probs = compute_all_metrics(test_logits, test_labels, best_thr)

print("\n===== FINAL TEST RESULTS =====")
for k, v in test_metrics.items():
    print(f"{k}: {v:.4f}")

# ROC
fpr, tpr, _ = roc_curve(test_labels, test_probs)
plt.figure()
plt.plot(fpr, tpr)
plt.title("ROC Curve")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.savefig(os.path.join(MODEL_DIR, "roc_curve.png"))
plt.close()

# PR
prec, rec, _ = precision_recall_curve(test_labels, test_probs)
plt.figure()
plt.plot(rec, prec)
plt.title("Precision Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.savefig(os.path.join(MODEL_DIR, "pr_curve.png"))
plt.close()


===== FINAL TEST RESULTS =====
F1: 0.9846
Accuracy: 0.9794
Precision: 0.9830
Recall: 0.9862
AUC: 0.9978
MCC: 0.9536


In [21]:
%uv pip install umap-learn

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Using Python 3.12.6 environment at: /usr/local
Audited 1 package in 9ms
Note: you may need to restart the kernel to use updated packages.


In [23]:
# ============================================================
# COMPLETE ACL READY ANALYSIS PIPELINE (SELF CONTAINED)
# ============================================================

import os
import time
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import AutoModel, AutoTokenizer
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    matthews_corrcoef, precision_recall_curve,
    roc_curve, confusion_matrix
)
from sklearn.calibration import calibration_curve
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

try:
    import umap
    UMAP_AVAILABLE = True
except:
    UMAP_AVAILABLE = False

# ============================================================
# CONFIG
# ============================================================

DATA_ROOT = "/mnt/heirarchy/data_stage1"
MODEL_ROOT = "/mnt/heirarchy/models_stage2_hier"
FIG_ROOT = os.path.join(MODEL_ROOT, "figures")
os.makedirs(FIG_ROOT, exist_ok=True)

ENC_A = "microsoft/deberta-v3-base"
ENC_B = "roberta-base"

D_LEX = 256
D_SYN = 256
D_SEM = 256
MAX_LEN = 128
BATCH_SIZE = 128

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# ============================================================
# MODEL DEFINITION (matches training)
# ============================================================

class AdditiveAttnPool(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.fc1 = nn.Linear(d, 384)
        self.fc2 = nn.Linear(384, 1)

    def forward(self, H, mask):
        s = self.fc2(torch.tanh(self.fc1(H))).squeeze(-1)
        s = s.masked_fill(mask == 0, -1e9)
        a = torch.softmax(s, dim=-1).unsqueeze(-1)
        return torch.sum(a * H, dim=1)

class CrossAttentionBlock(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.mha = nn.MultiheadAttention(d, 8, batch_first=True)
        self.ln = nn.LayerNorm(d)

    def forward(self, q, k, v, pad_mask):
        out,_ = self.mha(q,k,v,key_padding_mask=pad_mask)
        return self.ln(q + out)

class GatedBilinearFuse(nn.Module):
    def __init__(self,d):
        super().__init__()
        self.align_b = nn.Linear(d,d)
        self.bilin = nn.Bilinear(d,d,d)
        self.gate = nn.Sequential(nn.Linear(2*d,d), nn.Sigmoid())
        self.proj = nn.Linear(d,d)

    def forward(self,a,b):
        b2 = self.align_b(b)
        bil = self.bilin(a,b2)
        g = self.gate(torch.cat([a,b2],dim=-1))
        return self.proj(g*bil + (1-g)*a)

class HierDualEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc_a = AutoModel.from_pretrained(ENC_A)
        self.enc_b = AutoModel.from_pretrained(ENC_B)

        d = self.enc_a.config.hidden_size

        self.Slex = nn.Linear(d,D_LEX,bias=False)
        self.Ssyn = nn.Linear(d,D_SYN,bias=False)
        self.Ssem = nn.Linear(d,D_SEM,bias=False)

        self.pool_lex = AdditiveAttnPool(D_LEX)
        self.pool_syn = AdditiveAttnPool(D_SYN)
        self.pool_sem = AdditiveAttnPool(D_SEM)

        self.cross12 = CrossAttentionBlock(D_SEM)
        self.cross21 = CrossAttentionBlock(D_SEM)

        self.fuse_lex = GatedBilinearFuse(D_LEX)
        self.fuse_syn = GatedBilinearFuse(D_SYN)
        self.fuse_sem = GatedBilinearFuse(D_SEM)

        self.head_lex = nn.Linear(D_LEX,2)
        self.head_syn = nn.Linear(D_SYN,2)
        self.head_sem = nn.Linear(D_SEM,2)

        meta_in = 6 + D_LEX + D_SYN + D_SEM
        self.meta = nn.Sequential(
            nn.Linear(meta_in,512),
            nn.GELU(),
            nn.Linear(512,256),
            nn.GELU(),
            nn.Linear(256,2)
        )

    def encode(self, enc, ids, mask):
        return enc(input_ids=ids, attention_mask=mask).last_hidden_state

    def forward(self, ids_a, mask_a, ids_b, mask_b, return_features=False):

        Ha = self.encode(self.enc_a, ids_a, mask_a)
        Hb = self.encode(self.enc_b, ids_b, mask_b)

        Ha_lex,Hb_lex = self.Slex(Ha),self.Slex(Hb)
        Ha_syn,Hb_syn = self.Ssyn(Ha),self.Ssyn(Hb)
        Ha_sem,Hb_sem = self.Ssem(Ha),self.Ssem(Hb)

        fa_lex = self.pool_lex(Ha_lex,mask_a)
        fb_lex = self.pool_lex(Hb_lex,mask_b)
        fa_syn = self.pool_syn(Ha_syn,mask_a)
        fb_syn = self.pool_syn(Hb_syn,mask_b)

        Ha_sem_x = self.cross12(Ha_sem,Hb_sem,Hb_sem,mask_b==0)
        Hb_sem_x = self.cross21(Hb_sem,Ha_sem,Ha_sem,mask_a==0)

        fa_sem = self.pool_sem(Ha_sem_x,mask_a)
        fb_sem = self.pool_sem(Hb_sem_x,mask_b)

        z_lex = self.fuse_lex(fa_lex,fb_lex)
        z_syn = self.fuse_syn(fa_syn,fb_syn)
        z_sem = self.fuse_sem(fa_sem,fb_sem)

        log_lex = self.head_lex(z_lex)
        log_syn = self.head_syn(z_syn)
        log_sem = self.head_sem(z_sem)

        meta_in = torch.cat([log_lex,log_syn,log_sem,z_lex,z_syn,z_sem],dim=-1)
        log_meta = self.meta(meta_in)

        if return_features:
            return {
                "p_meta": F.softmax(log_meta,dim=1)[:,1],
                "z_lex": z_lex,
                "z_syn": z_syn,
                "z_sem": z_sem
            }

        return log_meta

# ============================================================
# LOAD CHECKPOINT (handle torch.compile)
# ============================================================

print("Loading checkpoint...")
model = HierDualEncoder().to(DEVICE)
ckpt = torch.load(os.path.join(MODEL_ROOT,"best_model.pt"), map_location=DEVICE)
clean = {k.replace("_orig_mod.",""):v for k,v in ckpt.items()}
model.load_state_dict(clean, strict=True)
model.eval()
print("Checkpoint loaded.")

# ============================================================
# DATA
# ============================================================

tok_a = AutoTokenizer.from_pretrained(ENC_A)
tok_b = AutoTokenizer.from_pretrained(ENC_B)

class DualDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        s1, s2 = str(r["sentence1"]), str(r["sentence2"])
        ea = tok_a(s1,s2,truncation=True,padding="max_length",
                   max_length=MAX_LEN,return_tensors="pt")
        eb = tok_b(s1,s2,truncation=True,padding="max_length",
                   max_length=MAX_LEN,return_tensors="pt")
        return {
            "ids_a":ea["input_ids"].squeeze(0),
            "mask_a":ea["attention_mask"].squeeze(0),
            "ids_b":eb["input_ids"].squeeze(0),
            "mask_b":eb["attention_mask"].squeeze(0),
            "label":torch.tensor(int(r["label"]))
        }

def loader(df):
    return DataLoader(DualDataset(df),batch_size=BATCH_SIZE,shuffle=False)

val_df = pd.read_csv(os.path.join(DATA_ROOT,"val.csv"))
test_df = pd.read_csv(os.path.join(DATA_ROOT,"test.csv"))

val_loader = loader(val_df)
test_loader = loader(test_df)

# ============================================================
# INFERENCE WITH PROGRESS
# ============================================================

def run(loader,name):
    ys=[];ps=[];z_sem=[]
    start=time.time()
    with torch.inference_mode():
        for i,b in enumerate(loader):
            b={k:v.to(DEVICE) for k,v in b.items()}
            out=model(b["ids_a"],b["mask_a"],b["ids_b"],b["mask_b"],return_features=True)
            ys.append(b["label"].cpu().numpy())
            ps.append(out["p_meta"].cpu().numpy())
            z_sem.append(out["z_sem"].cpu().numpy())
            if i%20==0:
                print(f"{name} batch {i}/{len(loader)}  elapsed {time.time()-start:.1f}s")
    return np.concatenate(ys),np.concatenate(ps),np.concatenate(z_sem)

print("Running VAL inference")
y_val,p_val,z_val=run(val_loader,"VAL")

print("Running TEST inference")
y_test,p_test,z_test=run(test_loader,"TEST")

# ============================================================
# THRESHOLD
# ============================================================

ts=np.linspace(0.01,0.99,99)
best_t=0.5
best_f=-1
for t in ts:
    f=f1_score(y_val,(p_val>=t).astype(int))
    if f>best_f:
        best_f=f
        best_t=t

print("Best threshold:",best_t)

# ============================================================
# METRICS
# ============================================================

def compute(y,p):
    return {
        "F1":f1_score(y,(p>=best_t).astype(int)),
        "Accuracy":accuracy_score(y,(p>=best_t).astype(int)),
        "AUC":roc_auc_score(y,p),
        "MCC":matthews_corrcoef(y,(p>=best_t).astype(int))
    }

metrics={"val":compute(y_val,p_val),
         "test":compute(y_test,p_test),
         "threshold":float(best_t)}

json.dump(metrics,open(os.path.join(FIG_ROOT,"metrics.json"),"w"),indent=2)

# ============================================================
# PLOTS
# ============================================================

def save(fig,name):
    fig.tight_layout()
    fig.savefig(os.path.join(FIG_ROOT,name),dpi=300)
    plt.close(fig)

# ROC
for split,y,p in [("val",y_val,p_val),("test",y_test,p_test)]:
    fpr,tpr,_=roc_curve(y,p)
    fig=plt.figure();plt.plot(fpr,tpr);plt.plot([0,1],[0,1])
    plt.title(split.upper()+" ROC")
    save(fig,f"{split}_roc.png")

# PR
for split,y,p in [("val",y_val,p_val),("test",y_test,p_test)]:
    prec,rec,_=precision_recall_curve(y,p)
    fig=plt.figure();plt.plot(rec,prec)
    plt.title(split.upper()+" PR")
    save(fig,f"{split}_pr.png")

# PCA
for split,y,z in [("val",y_val,z_val),("test",y_test,z_test)]:
    pca=PCA(n_components=2)
    emb=pca.fit_transform(z)
    fig=plt.figure()
    plt.scatter(emb[y==0,0],emb[y==0,1],s=2,label="non")
    plt.scatter(emb[y==1,0],emb[y==1,1],s=2,label="para")
    plt.legend();plt.title(split.upper()+" PCA Semantic")
    save(fig,f"{split}_pca_sem.png")

# UMAP
if UMAP_AVAILABLE:
    reducer=umap.UMAP(n_neighbors=30,min_dist=0.1,metric="cosine")
    emb=reducer.fit_transform(z_val[:40000])
    fig=plt.figure()
    plt.scatter(emb[y_val[:40000]==0,0],emb[y_val[:40000]==0,1],s=2,label="non")
    plt.scatter(emb[y_val[:40000]==1,0],emb[y_val[:40000]==1,1],s=2,label="para")
    plt.legend();plt.title("VAL UMAP Semantic")
    save(fig,"val_umap_sem.png")

print("All analysis complete. Figures saved in:",FIG_ROOT)

Device: cuda
Loading checkpoint...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Checkpoint loaded.
Running VAL inference
VAL batch 0/938  elapsed 1.7s
VAL batch 20/938  elapsed 37.3s
VAL batch 40/938  elapsed 72.6s
VAL batch 60/938  elapsed 108.7s
VAL batch 80/938  elapsed 145.5s
VAL batch 100/938  elapsed 180.4s
VAL batch 120/938  elapsed 217.4s
VAL batch 140/938  elapsed 253.1s
VAL batch 160/938  elapsed 290.3s
VAL batch 180/938  elapsed 326.2s
VAL batch 200/938  elapsed 363.0s
VAL batch 220/938  elapsed 399.6s
VAL batch 240/938  elapsed 433.4s
VAL batch 260/938  elapsed 469.5s
VAL batch 280/938  elapsed 505.2s
VAL batch 300/938  elapsed 540.9s
VAL batch 320/938  elapsed 578.5s
VAL batch 340/938  elapsed 613.9s
VAL batch 360/938  elapsed 650.5s
VAL batch 380/938  elapsed 686.7s
VAL batch 400/938  elapsed 722.3s
VAL batch 420/938  elapsed 758.1s
VAL batch 440/938  elapsed 793.5s
VAL batch 460/938  elapsed 829.6s
VAL batch 480/938  elapsed 867.7s
VAL batch 500/938  elapsed 904.8s
VAL batch 520/938  elapsed 941.7s
VAL batch 540/938  elapsed 975.3s
VAL batch 560/938

In [26]:
# ============================================================
# PLOTS WITH AXIS TITLES (NO RETRAINING)
# ============================================================

FIG_ROOT_TITLED = os.path.join(MODEL_ROOT, "figures_titled")
os.makedirs(FIG_ROOT_TITLED, exist_ok=True)

def save_titled(fig, name):
    fig.tight_layout()
    fig.savefig(os.path.join(FIG_ROOT_TITLED, name), dpi=300)
    plt.close(fig)

# ------------------------
# ROC
# ------------------------
for split, y, p in [("val", y_val, p_val), ("test", y_test, p_test)]:
    fpr, tpr, _ = roc_curve(y, p)
    fig = plt.figure()
    plt.plot(fpr, tpr)
    plt.plot([0, 1], [0, 1])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(split.upper() + " ROC")
    save_titled(fig, f"{split}_roc.png")

# ------------------------
# PR
# ------------------------
for split, y, p in [("val", y_val, p_val), ("test", y_test, p_test)]:
    prec, rec, _ = precision_recall_curve(y, p)
    fig = plt.figure()
    plt.plot(rec, prec)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(split.upper() + " PR")
    save_titled(fig, f"{split}_pr.png")

# ------------------------
# PCA
# ------------------------
for split, y, z in [("val", y_val, z_val), ("test", y_test, z_test)]:
    pca = PCA(n_components=2)
    emb = pca.fit_transform(z)
    fig = plt.figure()
    plt.scatter(emb[y == 0, 0], emb[y == 0, 1], s=2, label="non")
    plt.scatter(emb[y == 1, 0], emb[y == 1, 1], s=2, label="para")
    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")
    plt.legend()
    plt.title(split.upper() + " PCA Semantic")
    save_titled(fig, f"{split}_pca_sem.png")

# ------------------------
# UMAP
# ------------------------
if UMAP_AVAILABLE:
    reducer = umap.UMAP(n_neighbors=30, min_dist=0.1, metric="cosine")
    emb = reducer.fit_transform(z_val[:40000])
    fig = plt.figure()
    plt.scatter(emb[y_val[:40000] == 0, 0],
                emb[y_val[:40000] == 0, 1], s=2, label="non")
    plt.scatter(emb[y_val[:40000] == 1, 0],
                emb[y_val[:40000] == 1, 1], s=2, label="para")
    plt.xlabel("UMAP Dimension 1")
    plt.ylabel("UMAP Dimension 2")
    plt.legend()
    plt.title("VAL UMAP Semantic")
    save_titled(fig, "val_umap_sem.png")

print("Titled figures saved in:", FIG_ROOT_TITLED)

Titled figures saved in: /mnt/heirarchy/models_stage2_hier/figures_titled


In [31]:
import os
import json
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    matthews_corrcoef, precision_score, recall_score,
    roc_curve, precision_recall_curve, confusion_matrix
)
from sklearn.decomposition import PCA

try:
    import umap
    UMAP_AVAILABLE = True
except Exception:
    UMAP_AVAILABLE = False

os.environ["TOKENIZERS_PARALLELISM"] = "false"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ROOT = "/mnt/heirarchy/models_stage2_hier"
DATA_ROOT = "/mnt/heirarchy/data_stage1"
OUT_DIR = os.path.join(MODEL_ROOT, "acl_analysis_final")
os.makedirs(OUT_DIR, exist_ok=True)

print("Device:", DEVICE)
print("Saving to:", OUT_DIR)

# ============================================================
# Patch CrossAttentionBlock.forward to accept key_padding_mask
# ============================================================

def patched_cross_forward(self, q, k, v, key_padding_mask=None, **kwargs):
    attn_out, _ = self.mha(q, k, v, key_padding_mask=key_padding_mask, need_weights=False)
    return self.ln(q + attn_out)

CrossAttentionBlock.forward = patched_cross_forward
print("CrossAttentionBlock patched.")

# ============================================================
# Load model checkpoint
# ============================================================

best_path = os.path.join(MODEL_ROOT, "best_model.pt")

model = HierDualEncoderDualTok().to(DEVICE)
state = torch.load(best_path, map_location=DEVICE)
state = {k.replace("_orig_mod.", ""): v for k, v in state.items()}
model.load_state_dict(state, strict=True)
model.eval()
print("Model loaded successfully.")

# ============================================================
# Tokenizers inferred from loaded encoders
# ============================================================

from transformers import AutoTokenizer

tok_a = AutoTokenizer.from_pretrained(model.enc_a.name_or_path, use_fast=True)
tok_b = AutoTokenizer.from_pretrained(model.enc_b.name_or_path, use_fast=True)
MAX_LEN = 128

# ============================================================
# Dataset and loader
# ============================================================

class EvalDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        s1 = str(r["sentence1"])
        s2 = str(r["sentence2"])

        ea = tok_a(s1, s2, truncation=True, padding="max_length", max_length=MAX_LEN, return_tensors="pt")
        eb = tok_b(s1, s2, truncation=True, padding="max_length", max_length=MAX_LEN, return_tensors="pt")

        return {
            "input_ids_a": ea["input_ids"].squeeze(0),
            "attention_mask_a": ea["attention_mask"].squeeze(0),
            "input_ids_b": eb["input_ids"].squeeze(0),
            "attention_mask_b": eb["attention_mask"].squeeze(0),
            "label": torch.tensor(int(r["label"]), dtype=torch.long),
        }

def make_loader(df, batch_size=128):
    return torch.utils.data.DataLoader(
        EvalDataset(df),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )

val_df = pd.read_csv(os.path.join(DATA_ROOT, "val.csv"))
test_df = pd.read_csv(os.path.join(DATA_ROOT, "test.csv"))
val_loader = make_loader(val_df, batch_size=128)
test_loader = make_loader(test_df, batch_size=128)
print("Data loaded.")

# ============================================================
# Robust output parsing
# ============================================================

def get_log_meta(out):
    # training style: dict
    if isinstance(out, dict):
        if "log_meta" in out:
            return out["log_meta"]
        # fallback if dict but different key
        for k in ("logits", "meta", "out"):
            if k in out:
                return out[k]
        raise KeyError(f"Model returned dict keys {list(out.keys())} but no log_meta or logits-like key found")

    # tuple or list
    if isinstance(out, (tuple, list)):
        if len(out) == 0:
            raise ValueError("Model returned empty tuple or list")
        return out[0]

    # tensor
    if torch.is_tensor(out):
        return out

    raise TypeError(f"Unsupported model output type: {type(out)}")

def pos_prob_from_logits(logits: torch.Tensor) -> torch.Tensor:
    x = logits
    if not torch.is_tensor(x):
        x = torch.as_tensor(x, device=DEVICE)

    if x.dtype not in (torch.float16, torch.bfloat16, torch.float32, torch.float64):
        x = x.float()

    # [B,2]
    if x.ndim == 2 and x.shape[1] == 2:
        return torch.softmax(x, dim=-1)[:, 1]

    # [B] logits
    if x.ndim == 1:
        return torch.sigmoid(x)

    # [B,1]
    if x.ndim == 2 and x.shape[1] == 1:
        return torch.sigmoid(x[:, 0])

    raise ValueError(f"Unsupported logits shape for probability extraction: {tuple(x.shape)}")

# ============================================================
# Inference
# ============================================================

def run(loader, name):
    ys, ps, zs = [], [], []
    start = time.time()

    with torch.inference_mode():
        for i, batch in enumerate(loader):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}

            out = model(
                batch["input_ids_a"], batch["attention_mask_a"],
                batch["input_ids_b"], batch["attention_mask_b"]
            )

            log_meta = get_log_meta(out)
            prob = pos_prob_from_logits(log_meta)

            if i == 0:
                print(name, "first batch out type:", type(out))
                if isinstance(out, dict):
                    print(name, "dict keys:", list(out.keys()))
                print(name, "log_meta shape:", tuple(log_meta.shape))
                print(name, "prob shape:", tuple(prob.shape))

            ys.append(batch["label"].detach().cpu().numpy())
            ps.append(prob.detach().float().cpu().numpy())

            # semantic embedding for PCA and UMAP
            Ha = model.encode(model.enc_a, batch["input_ids_a"], batch["attention_mask_a"])
            z = model.Ssem(Ha).mean(dim=1)
            zs.append(z.detach().float().cpu().numpy())

            if i % 20 == 0:
                print(f"{name} batch {i}/{len(loader)} elapsed {time.time()-start:.1f}s", flush=True)

    return np.concatenate(ys), np.concatenate(ps), np.concatenate(zs)

print("Running VAL inference")
y_val, p_val, z_val = run(val_loader, "VAL")

print("Running TEST inference")
y_test, p_test, z_test = run(test_loader, "TEST")

# ============================================================
# Threshold tuning on VAL
# ============================================================

ts = np.linspace(0.01, 0.99, 99)
best_t = 0.5
best_f = -1.0
for t in ts:
    f = f1_score(y_val, (p_val >= t).astype(int))
    if f > best_f:
        best_f = f
        best_t = t

print("Best threshold:", float(best_t), "val f1:", float(best_f))

# ============================================================
# Metrics
# ============================================================

def compute_metrics(y, p, thr):
    pred = (p >= thr).astype(int)
    return {
        "F1": float(f1_score(y, pred)),
        "Accuracy": float(accuracy_score(y, pred)),
        "Precision": float(precision_score(y, pred)),
        "Recall": float(recall_score(y, pred)),
        "AUC": float(roc_auc_score(y, p)),
        "MCC": float(matthews_corrcoef(y, pred)),
        "Threshold": float(thr),
    }

val_metrics = compute_metrics(y_val, p_val, best_t)
test_metrics = compute_metrics(y_test, p_test, best_t)

print("\nVAL METRICS")
for k, v in val_metrics.items():
    print(k, ":", v)

print("\nTEST METRICS")
for k, v in test_metrics.items():
    print(k, ":", v)

json.dump({"val": val_metrics, "test": test_metrics},
          open(os.path.join(OUT_DIR, "metrics.json"), "w"),
          indent=2)

# ============================================================
# Plots
# ============================================================

def save_fig(fig, name):
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, name), dpi=300)
    plt.close(fig)

def plot_roc(y, p, title, fname):
    fpr, tpr, _ = roc_curve(y, p)
    fig = plt.figure()
    plt.plot(fpr, tpr)
    plt.plot([0, 1], [0, 1])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    save_fig(fig, fname)

def plot_pr(y, p, title, fname):
    prec, rec, _ = precision_recall_curve(y, p)
    fig = plt.figure()
    plt.plot(rec, prec)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(title)
    save_fig(fig, fname)

def plot_confusion(y, p, thr, title, fname):
    cm = confusion_matrix(y, (p >= thr).astype(int))
    fig = plt.figure()
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    save_fig(fig, fname)

def plot_score_hist(y, p, title, fname):
    fig = plt.figure()
    sns.histplot(p[y == 0], stat="density", label="neg")
    sns.histplot(p[y == 1], stat="density", label="pos")
    plt.xlabel("Predicted probability")
    plt.ylabel("Density")
    plt.legend()
    plt.title(title)
    save_fig(fig, fname)

def plot_pca(z, y, title, fname):
    pca = PCA(n_components=2)
    emb = pca.fit_transform(z)
    fig = plt.figure()
    plt.scatter(emb[y == 0, 0], emb[y == 0, 1], s=2, label="neg")
    plt.scatter(emb[y == 1, 0], emb[y == 1, 1], s=2, label="pos")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.legend()
    plt.title(title)
    save_fig(fig, fname)

def plot_umap(z, y, title, fname, n=30000):
    if not UMAP_AVAILABLE:
        return
    n = min(n, len(z))
    reducer = umap.UMAP(n_neighbors=30, min_dist=0.1, metric="cosine")
    emb = reducer.fit_transform(z[:n])
    yy = y[:n]
    fig = plt.figure()
    plt.scatter(emb[yy == 0, 0], emb[yy == 0, 1], s=2, label="neg")
    plt.scatter(emb[yy == 1, 0], emb[yy == 1, 1], s=2, label="pos")
    plt.xlabel("UMAP1")
    plt.ylabel("UMAP2")
    plt.legend()
    plt.title(title)
    save_fig(fig, fname)

plot_roc(y_val, p_val, "VAL ROC", "val_roc.png")
plot_roc(y_test, p_test, "TEST ROC", "test_roc.png")

plot_pr(y_val, p_val, "VAL PR", "val_pr.png")
plot_pr(y_test, p_test, "TEST PR", "test_pr.png")

plot_confusion(y_val, p_val, best_t, "VAL Confusion", "val_confusion.png")
plot_confusion(y_test, p_test, best_t, "TEST Confusion", "test_confusion.png")

plot_score_hist(y_val, p_val, "VAL Score Distribution", "val_score_hist.png")
plot_score_hist(y_test, p_test, "TEST Score Distribution", "test_score_hist.png")

plot_pca(z_val, y_val, "VAL PCA Semantic", "val_pca_sem.png")
plot_pca(z_test, y_test, "TEST PCA Semantic", "test_pca_sem.png")

plot_umap(z_val, y_val, "VAL UMAP Semantic", "val_umap_sem.png", n=30000)
plot_umap(z_test, y_test, "TEST UMAP Semantic", "test_umap_sem.png", n=30000)

summary = pd.DataFrame([
    {"Split": "val", **val_metrics},
    {"Split": "test", **test_metrics},
])
print("\nSUMMARY TABLE")
print(summary)

summary.to_csv(os.path.join(OUT_DIR, "summary_table.csv"), index=False)

print("\nDone. Outputs in:", OUT_DIR)

Device: cuda
Saving to: /mnt/heirarchy/models_stage2_hier/acl_analysis_final
CrossAttentionBlock patched.


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded successfully.
Data loaded.
Running VAL inference
VAL first batch out type: <class 'torch.Tensor'>
VAL log_meta shape: (128, 2)
VAL prob shape: (128,)
VAL batch 0/938 elapsed 0.5s
VAL batch 20/938 elapsed 10.4s
VAL batch 40/938 elapsed 20.4s
VAL batch 60/938 elapsed 30.6s
VAL batch 80/938 elapsed 41.0s
VAL batch 100/938 elapsed 50.9s
VAL batch 120/938 elapsed 61.1s
VAL batch 140/938 elapsed 71.1s
VAL batch 160/938 elapsed 81.8s
VAL batch 180/938 elapsed 92.1s
VAL batch 200/938 elapsed 102.7s
VAL batch 220/938 elapsed 112.8s
VAL batch 240/938 elapsed 122.6s
VAL batch 260/938 elapsed 132.8s
VAL batch 280/938 elapsed 143.3s
VAL batch 300/938 elapsed 153.4s
VAL batch 320/938 elapsed 163.8s
VAL batch 340/938 elapsed 173.7s
VAL batch 360/938 elapsed 184.1s
VAL batch 380/938 elapsed 194.0s
VAL batch 400/938 elapsed 204.2s
VAL batch 420/938 elapsed 214.3s
VAL batch 440/938 elapsed 224.2s
VAL batch 460/938 elapsed 234.3s
VAL batch 480/938 elapsed 244.9s
VAL batch 500/938 elapsed 254

In [32]:
# ============================================================
# EXTRA ACL-STYLE IMPROVEMENTS (NO TRAINING)
# Adds: calibration, ECE, Brier, threshold sweeps, ROC/PR AUC,
# AUPRC, subgroup analyses, bootstrap CIs, decision curves,
# reliability, score quantiles, error analysis dumps,
# publication-ready tables + all plots saved.
# Assumes you already have in session:
#   y_val, p_val, z_val, y_test, p_test, z_test, best_t
# If not, it will load metrics.json from acl_analysis_final.
# ============================================================

import os, json, time, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
    roc_auc_score, matthews_corrcoef,
    roc_curve, precision_recall_curve, auc,
    confusion_matrix, brier_score_loss
)
from sklearn.calibration import calibration_curve
from sklearn.decomposition import PCA

try:
    import umap
    UMAP_AVAILABLE = True
except Exception:
    UMAP_AVAILABLE = False

MODEL_ROOT = "/mnt/heirarchy/models_stage2_hier"
BASE_OUT = os.path.join(MODEL_ROOT, "acl_analysis_final")
OUT_DIR = os.path.join(MODEL_ROOT, "acl_analysis_plus")
os.makedirs(OUT_DIR, exist_ok=True)

print("Saving extra outputs to:", OUT_DIR)

# ------------------------------------------------------------
# If variables are missing, try to load from BASE_OUT artifacts
# ------------------------------------------------------------
_missing = []
for v in ["y_val","p_val","z_val","y_test","p_test","z_test","best_t"]:
    if v not in globals():
        _missing.append(v)

if _missing:
    print("Warning: missing in-session variables:", _missing)
    # best effort: load threshold from metrics.json
    metrics_path = os.path.join(BASE_OUT, "metrics.json")
    if os.path.exists(metrics_path):
        m = json.load(open(metrics_path, "r"))
        best_t = float(m["test"]["Threshold"])
        print("Loaded threshold from metrics.json:", best_t)
    else:
        best_t = 0.55
        print("metrics.json not found, using best_t=0.55")

# ============================================================
# Utilities
# ============================================================

def save_fig(fig, name):
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, name), dpi=300)
    plt.close(fig)

def metric_pack(y, p, thr):
    pred = (p >= thr).astype(int)
    pr_prec, pr_rec, _ = precision_recall_curve(y, p)
    fpr, tpr, _ = roc_curve(y, p)
    return {
        "F1": float(f1_score(y, pred)),
        "Accuracy": float(accuracy_score(y, pred)),
        "Precision": float(precision_score(y, pred)),
        "Recall": float(recall_score(y, pred)),
        "AUC_ROC": float(roc_auc_score(y, p)),
        "AUC_PR": float(auc(pr_rec, pr_prec)),
        "MCC": float(matthews_corrcoef(y, pred)),
        "Brier": float(brier_score_loss(y, p)),
        "Threshold": float(thr),
        "PosRate": float(np.mean(y)),
        "PredPosRate": float(np.mean(pred)),
    }

def expected_calibration_error(y, p, n_bins=15):
    # equal-width bins on [0,1]
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        m = (p >= lo) & (p < hi) if i < n_bins - 1 else (p >= lo) & (p <= hi)
        if not np.any(m):
            continue
        acc = np.mean((p[m] >= best_t).astype(int) == y[m])
        conf = np.mean(p[m])
        ece += (np.sum(m) / len(y)) * abs(acc - conf)
    return float(ece)

def threshold_sweep(y, p, grid=None):
    if grid is None:
        grid = np.linspace(0.01, 0.99, 99)
    rows = []
    for t in grid:
        pred = (p >= t).astype(int)
        rows.append({
            "t": float(t),
            "f1": float(f1_score(y, pred)),
            "acc": float(accuracy_score(y, pred)),
            "prec": float(precision_score(y, pred, zero_division=0)),
            "rec": float(recall_score(y, pred, zero_division=0)),
            "mcc": float(matthews_corrcoef(y, pred)),
        })
    return pd.DataFrame(rows)

def bootstrap_ci(y, p, thr, B=2000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y)
    stats = {"F1": [], "Accuracy": [], "Precision": [], "Recall": [], "AUC_ROC": [], "MCC": [], "Brier": []}
    for _ in range(B):
        idx = rng.integers(0, n, size=n)
        yy = y[idx]
        pp = p[idx]
        pred = (pp >= thr).astype(int)
        stats["F1"].append(f1_score(yy, pred))
        stats["Accuracy"].append(accuracy_score(yy, pred))
        stats["Precision"].append(precision_score(yy, pred, zero_division=0))
        stats["Recall"].append(recall_score(yy, pred, zero_division=0))
        # AUC requires both classes
        if len(np.unique(yy)) == 2:
            stats["AUC_ROC"].append(roc_auc_score(yy, pp))
        else:
            stats["AUC_ROC"].append(np.nan)
        stats["MCC"].append(matthews_corrcoef(yy, pred))
        stats["Brier"].append(brier_score_loss(yy, pp))
    out = {}
    for k, arr in stats.items():
        arr = np.asarray(arr, dtype=float)
        arr = arr[~np.isnan(arr)]
        if arr.size == 0:
            out[k] = {"mean": np.nan, "lo": np.nan, "hi": np.nan}
        else:
            out[k] = {
                "mean": float(np.mean(arr)),
                "lo": float(np.quantile(arr, 0.025)),
                "hi": float(np.quantile(arr, 0.975)),
            }
    return out

def decision_curve_net_benefit(y, p, thresholds=None):
    # Net Benefit = TP/N - FP/N * (t/(1-t))
    if thresholds is None:
        thresholds = np.linspace(0.01, 0.99, 99)
    N = len(y)
    rows = []
    for t in thresholds:
        pred = (p >= t).astype(int)
        tp = np.sum((pred == 1) & (y == 1))
        fp = np.sum((pred == 1) & (y == 0))
        nb = (tp / N) - (fp / N) * (t / (1 - t))
        rows.append({"t": float(t), "net_benefit": float(nb)})
    return pd.DataFrame(rows)

# ============================================================
# Core packs
# ============================================================

val_pack = metric_pack(y_val, p_val, best_t)
test_pack = metric_pack(y_test, p_test, best_t)
val_pack["ECE"] = expected_calibration_error(y_val, p_val, n_bins=15)
test_pack["ECE"] = expected_calibration_error(y_test, p_test, n_bins=15)

summary = pd.DataFrame([
    {"Split": "val", **val_pack},
    {"Split": "test", **test_pack},
])
print("\n=== EXTENDED SUMMARY ===")
print(summary)
summary.to_csv(os.path.join(OUT_DIR, "extended_summary.csv"), index=False)

# ============================================================
# Bootstrap confidence intervals (test)
# ============================================================

print("\nBootstrapping CIs on TEST (this can take a bit)...", flush=True)
ci_test = bootstrap_ci(y_test, p_test, best_t, B=2000, seed=7)
json.dump(ci_test, open(os.path.join(OUT_DIR, "bootstrap_ci_test.json"), "w"), indent=2)
print("Saved bootstrap_ci_test.json")

# Make a nice table
ci_rows = []
for k, d in ci_test.items():
    ci_rows.append({"Metric": k, "Mean": d["mean"], "CI_2.5": d["lo"], "CI_97.5": d["hi"]})
ci_df = pd.DataFrame(ci_rows)
ci_df.to_csv(os.path.join(OUT_DIR, "bootstrap_ci_test.csv"), index=False)
print("\n=== TEST BOOTSTRAP CI TABLE ===")
print(ci_df)

# ============================================================
# Reliability diagram + calibration histogram
# ============================================================

def plot_reliability(y, p, title, fname, n_bins=15):
    frac_pos, mean_pred = calibration_curve(y, p, n_bins=n_bins, strategy="uniform")
    fig = plt.figure()
    plt.plot(mean_pred, frac_pos, marker="o")
    plt.plot([0, 1], [0, 1])
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Fraction positive")
    plt.title(title)
    save_fig(fig, fname)

def plot_prob_hist(y, p, title, fname, bins=30):
    fig = plt.figure()
    plt.hist(p[y == 0], bins=bins, alpha=0.7, label="neg")
    plt.hist(p[y == 1], bins=bins, alpha=0.7, label="pos")
    plt.xlabel("Predicted probability")
    plt.ylabel("Count")
    plt.legend()
    plt.title(title)
    save_fig(fig, fname)

plot_reliability(y_val, p_val, "VAL Reliability Diagram", "val_reliability.png")
plot_reliability(y_test, p_test, "TEST Reliability Diagram", "test_reliability.png")
plot_prob_hist(y_val, p_val, "VAL Probability Histogram", "val_prob_hist.png")
plot_prob_hist(y_test, p_test, "TEST Probability Histogram", "test_prob_hist.png")

# ============================================================
# Threshold sweeps: F1, MCC, Acc, Prec, Rec
# ============================================================

sweep_val = threshold_sweep(y_val, p_val)
sweep_test = threshold_sweep(y_test, p_test)
sweep_val.to_csv(os.path.join(OUT_DIR, "threshold_sweep_val.csv"), index=False)
sweep_test.to_csv(os.path.join(OUT_DIR, "threshold_sweep_test.csv"), index=False)

def plot_sweep(df, metric, title, fname):
    fig = plt.figure()
    plt.plot(df["t"].values, df[metric].values)
    plt.axvline(best_t)
    plt.xlabel("Threshold")
    plt.ylabel(metric.upper())
    plt.title(title)
    save_fig(fig, fname)

for m in ["f1", "mcc", "acc", "prec", "rec"]:
    plot_sweep(sweep_val, m, f"VAL {m.upper()} vs Threshold", f"val_{m}_vs_threshold.png")
    plot_sweep(sweep_test, m, f"TEST {m.upper()} vs Threshold", f"test_{m}_vs_threshold.png")

# ============================================================
# ROC AUC and PR AUC with labeled axes
# ============================================================

def plot_roc(y, p, title, fname):
    fpr, tpr, _ = roc_curve(y, p)
    fig = plt.figure()
    plt.plot(fpr, tpr)
    plt.plot([0, 1], [0, 1])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title + f" (AUC={roc_auc_score(y,p):.4f})")
    save_fig(fig, fname)

def plot_pr(y, p, title, fname):
    prec, rec, _ = precision_recall_curve(y, p)
    auprc = auc(rec, prec)
    fig = plt.figure()
    plt.plot(rec, prec)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(title + f" (AUPRC={auprc:.4f})")
    save_fig(fig, fname)

plot_roc(y_val, p_val, "VAL ROC", "val_roc_labeled.png")
plot_roc(y_test, p_test, "TEST ROC", "test_roc_labeled.png")
plot_pr(y_val, p_val, "VAL PR", "val_pr_labeled.png")
plot_pr(y_test, p_test, "TEST PR", "test_pr_labeled.png")

# ============================================================
# Confusion matrices at best_t and at 0.5
# ============================================================

def plot_cm(y, p, thr, title, fname):
    cm = confusion_matrix(y, (p >= thr).astype(int))
    fig = plt.figure()
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title + f" (t={thr:.2f})")
    save_fig(fig, fname)

plot_cm(y_val, p_val, best_t, "VAL Confusion", "val_confusion_best.png")
plot_cm(y_test, p_test, best_t, "TEST Confusion", "test_confusion_best.png")
plot_cm(y_val, p_val, 0.5, "VAL Confusion", "val_confusion_050.png")
plot_cm(y_test, p_test, 0.5, "TEST Confusion", "test_confusion_050.png")

# ============================================================
# Decision curve analysis (net benefit)
# ============================================================

dc_val = decision_curve_net_benefit(y_val, p_val)
dc_test = decision_curve_net_benefit(y_test, p_test)
dc_val.to_csv(os.path.join(OUT_DIR, "decision_curve_val.csv"), index=False)
dc_test.to_csv(os.path.join(OUT_DIR, "decision_curve_test.csv"), index=False)

def plot_dc(df, title, fname):
    fig = plt.figure()
    plt.plot(df["t"], df["net_benefit"])
    plt.axvline(best_t)
    plt.xlabel("Threshold probability")
    plt.ylabel("Net benefit")
    plt.title(title)
    save_fig(fig, fname)

plot_dc(dc_val, "VAL Decision Curve", "val_decision_curve.png")
plot_dc(dc_test, "TEST Decision Curve", "test_decision_curve.png")

# ============================================================
# Representation geometry extras: explained variance, UMAP, PCA
# ============================================================

def plot_pca_variance(z, title, fname, k=50):
    k = min(k, z.shape[1], 200)
    pca = PCA(n_components=k)
    pca.fit(z)
    ev = pca.explained_variance_ratio_
    fig = plt.figure()
    plt.plot(np.arange(1, len(ev)+1), np.cumsum(ev))
    plt.xlabel("Number of components")
    plt.ylabel("Cumulative explained variance")
    plt.title(title)
    save_fig(fig, fname)

plot_pca_variance(z_val, "VAL PCA Cumulative Variance", "val_pca_cumvar.png")
plot_pca_variance(z_test, "TEST PCA Cumulative Variance", "test_pca_cumvar.png")

def plot_pca_scatter(z, y, title, fname):
    pca = PCA(n_components=2)
    emb = pca.fit_transform(z)
    fig = plt.figure()
    plt.scatter(emb[y==0,0], emb[y==0,1], s=2, label="neg")
    plt.scatter(emb[y==1,0], emb[y==1,1], s=2, label="pos")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.legend()
    plt.title(title)
    save_fig(fig, fname)

plot_pca_scatter(z_val, y_val, "VAL PCA Scatter (Semantic)", "val_pca_scatter.png")
plot_pca_scatter(z_test, y_test, "TEST PCA Scatter (Semantic)", "test_pca_scatter.png")

if UMAP_AVAILABLE:
    def plot_umap_scatter(z, y, title, fname, n=30000, seed=42):
        n = min(n, len(z))
        reducer = umap.UMAP(n_neighbors=30, min_dist=0.1, metric="cosine", random_state=seed)
        emb = reducer.fit_transform(z[:n])
        yy = y[:n]
        fig = plt.figure()
        plt.scatter(emb[yy==0,0], emb[yy==0,1], s=2, label="neg")
        plt.scatter(emb[yy==1,0], emb[yy==1,1], s=2, label="pos")
        plt.xlabel("UMAP1")
        plt.ylabel("UMAP2")
        plt.legend()
        plt.title(title)
        save_fig(fig, fname)

    plot_umap_scatter(z_val, y_val, "VAL UMAP Scatter (Semantic)", "val_umap_scatter.png")
    plot_umap_scatter(z_test, y_test, "TEST UMAP Scatter (Semantic)", "test_umap_scatter.png")

# ============================================================
# Error analysis dumps: hardest false positives and false negatives
# Requires original CSV rows for interpretation
# ============================================================

val_df = pd.read_csv(os.path.join("/mnt/heirarchy/data_stage1", "val.csv"))
test_df = pd.read_csv(os.path.join("/mnt/heirarchy/data_stage1", "test.csv"))

def dump_hard_cases(df, y, p, thr, split, k=100):
    pred = (p >= thr).astype(int)
    y = np.asarray(y)
    p = np.asarray(p)

    # false positives: y=0, pred=1, highest p
    fp_idx = np.where((y == 0) & (pred == 1))[0]
    fp_sorted = fp_idx[np.argsort(-p[fp_idx])] if fp_idx.size else np.array([], dtype=int)

    # false negatives: y=1, pred=0, lowest p
    fn_idx = np.where((y == 1) & (pred == 0))[0]
    fn_sorted = fn_idx[np.argsort(p[fn_idx])] if fn_idx.size else np.array([], dtype=int)

    fp_df = df.iloc[fp_sorted[:k]].copy()
    fp_df["prob"] = p[fp_sorted[:k]]
    fp_df.to_csv(os.path.join(OUT_DIR, f"{split}_hard_false_positives_top{k}.csv"), index=False)

    fn_df = df.iloc[fn_sorted[:k]].copy()
    fn_df["prob"] = p[fn_sorted[:k]]
    fn_df.to_csv(os.path.join(OUT_DIR, f"{split}_hard_false_negatives_top{k}.csv"), index=False)

    # also easiest cases
    easy_pos_idx = np.where(y == 1)[0]
    easy_neg_idx = np.where(y == 0)[0]
    easy_pos = easy_pos_idx[np.argsort(-p[easy_pos_idx])][:k] if easy_pos_idx.size else np.array([], dtype=int)
    easy_neg = easy_neg_idx[np.argsort(p[easy_neg_idx])][:k] if easy_neg_idx.size else np.array([], dtype=int)

    ep = df.iloc[easy_pos].copy(); ep["prob"] = p[easy_pos]
    en = df.iloc[easy_neg].copy(); en["prob"] = p[easy_neg]
    ep.to_csv(os.path.join(OUT_DIR, f"{split}_easiest_positives_top{k}.csv"), index=False)
    en.to_csv(os.path.join(OUT_DIR, f"{split}_easiest_negatives_top{k}.csv"), index=False)

dump_hard_cases(val_df, y_val, p_val, best_t, "val", k=100)
dump_hard_cases(test_df, y_test, p_test, best_t, "test", k=100)
print("Saved hard case CSVs.")

# ============================================================
# Quantile table: how probabilities distribute by class
# ============================================================

def quantile_table(y, p, split):
    qs = [0.0, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1.0]
    def qstats(arr):
        return {f"q{int(q*100):02d}": float(np.quantile(arr, q)) for q in qs}
    rows = []
    rows.append({"split": split, "class": "neg", **qstats(p[y==0])})
    rows.append({"split": split, "class": "pos", **qstats(p[y==1])})
    return pd.DataFrame(rows)

qt = pd.concat([quantile_table(y_val, p_val, "val"), quantile_table(y_test, p_test, "test")], ignore_index=True)
qt.to_csv(os.path.join(OUT_DIR, "prob_quantiles_by_class.csv"), index=False)
print("\n=== PROB QUANTILES BY CLASS ===")
print(qt)

# ============================================================
# Final note
# ============================================================

artifact_index = {
    "extended_summary": "extended_summary.csv",
    "bootstrap_ci_test": ["bootstrap_ci_test.json", "bootstrap_ci_test.csv"],
    "threshold_sweeps": ["threshold_sweep_val.csv", "threshold_sweep_test.csv"],
    "decision_curves": ["decision_curve_val.csv", "decision_curve_test.csv"],
    "hard_cases": [
        "val_hard_false_positives_top100.csv",
        "val_hard_false_negatives_top100.csv",
        "test_hard_false_positives_top100.csv",
        "test_hard_false_negatives_top100.csv",
        "val_easiest_positives_top100.csv",
        "val_easiest_negatives_top100.csv",
        "test_easiest_positives_top100.csv",
        "test_easiest_negatives_top100.csv",
    ],
    "plots": "all .png files in this folder",
}
json.dump(artifact_index, open(os.path.join(OUT_DIR, "artifact_index.json"), "w"), indent=2)

print("\nAll extras done.")
print("Artifacts index saved to:", os.path.join(OUT_DIR, "artifact_index.json"))
print("Folder:", OUT_DIR)

Saving extra outputs to: /mnt/heirarchy/models_stage2_hier/acl_analysis_plus

=== EXTENDED SUMMARY ===
  Split        F1  Accuracy  Precision    Recall   AUC_ROC    AUC_PR  \
0   val  0.984383  0.979125   0.981904  0.986875  0.997698  0.998836   
1  test  0.984609  0.979433   0.982416  0.986812  0.997801  0.998875   

        MCC     Brier  Threshold   PosRate  PredPosRate       ECE  
0  0.952940  0.016329       0.55  0.666667     0.670042  0.319081  
1  0.953643  0.015928       0.55  0.666667     0.669650  0.319544  

Bootstrapping CIs on TEST (this can take a bit)...
Saved bootstrap_ci_test.json

=== TEST BOOTSTRAP CI TABLE ===
      Metric      Mean    CI_2.5   CI_97.5
0         F1  0.984599  0.983972  0.985195
1   Accuracy  0.979418  0.978583  0.980217
2  Precision  0.982406  0.981514  0.983291
3     Recall  0.986803  0.985989  0.987563
4    AUC_ROC  0.997800  0.997643  0.997945
5        MCC  0.953604  0.951738  0.955399
6      Brier  0.015934  0.015381  0.016528
Saved hard case CS

In [33]:
# ============================================================
# POST-HOC TEMPERATURE SCALING (NO RETRAINING)
# Fits temperature on validation logits only
# Applies to test logits
# Saves calibrated metrics + plots
# ============================================================

import os
import json
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
    roc_auc_score, matthews_corrcoef,
    roc_curve, precision_recall_curve,
    brier_score_loss
)
from sklearn.calibration import calibration_curve

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CAL_OUT = "/mnt/heirarchy/models_stage2_hier/acl_analysis_calibrated"
os.makedirs(CAL_OUT, exist_ok=True)

print("Calibration output folder:", CAL_OUT)

# ------------------------------------------------------------
# REQUIREMENTS:
# You must already have:
#   val_logits, val_labels
#   test_logits, test_labels
# If not available, reconstruct from saved logits or rerun
# inference that stores raw logits.
# ------------------------------------------------------------

if "val_logits" not in globals() or "test_logits" not in globals():
    raise RuntimeError("val_logits and test_logits must exist in session.")


# ============================================================
# TEMPERATURE SCALER
# ============================================================

class TemperatureScaler(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = torch.nn.Parameter(torch.ones(1))

    def forward(self, logits):
        return logits / self.temperature

def fit_temperature(logits, labels):
    logits = torch.tensor(logits, dtype=torch.float32, device=DEVICE)
    labels = torch.tensor(labels, dtype=torch.long, device=DEVICE)

    scaler = TemperatureScaler().to(DEVICE)

    optimizer = torch.optim.LBFGS([scaler.temperature], lr=0.01, max_iter=100)

    def closure():
        optimizer.zero_grad()
        scaled_logits = scaler(logits)
        loss = F.cross_entropy(scaled_logits, labels)
        loss.backward()
        return loss

    optimizer.step(closure)

    return scaler

print("Fitting temperature on validation set...")
scaler = fit_temperature(val_logits, val_labels)
T = scaler.temperature.item()
print("Optimal temperature:", T)


# ============================================================
# APPLY CALIBRATION
# ============================================================

def calibrated_probs(logits, scaler):
    logits = torch.tensor(logits, dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        scaled = scaler(logits)
        probs = F.softmax(scaled, dim=1)[:, 1]
    return probs.cpu().numpy()

val_probs_cal = calibrated_probs(val_logits, scaler)
test_probs_cal = calibrated_probs(test_logits, scaler)


# ============================================================
# METRICS FUNCTION
# ============================================================

def compute_metrics(y, p, threshold=0.55):
    pred = (p >= threshold).astype(int)
    return {
        "F1": float(f1_score(y, pred)),
        "Accuracy": float(accuracy_score(y, pred)),
        "Precision": float(precision_score(y, pred)),
        "Recall": float(recall_score(y, pred)),
        "AUC": float(roc_auc_score(y, p)),
        "MCC": float(matthews_corrcoef(y, pred)),
        "Brier": float(brier_score_loss(y, p)),
    }

print("\n=== CALIBRATED METRICS ===")
val_cal_metrics = compute_metrics(val_labels, val_probs_cal)
test_cal_metrics = compute_metrics(test_labels, test_probs_cal)

print("\nValidation (Calibrated)")
for k, v in val_cal_metrics.items():
    print(k, ":", v)

print("\nTest (Calibrated)")
for k, v in test_cal_metrics.items():
    print(k, ":", v)

json.dump({
    "temperature": T,
    "val": val_cal_metrics,
    "test": test_cal_metrics
}, open(os.path.join(CAL_OUT, "calibrated_metrics.json"), "w"), indent=2)


# ============================================================
# RELIABILITY DIAGRAM
# ============================================================

def plot_reliability(y, p, name):
    frac_pos, mean_pred = calibration_curve(y, p, n_bins=15)

    plt.figure()
    plt.plot(mean_pred, frac_pos, marker="o")
    plt.plot([0, 1], [0, 1])
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Fraction positive")
    plt.title(name)
    plt.tight_layout()
    plt.savefig(os.path.join(CAL_OUT, f"{name}.png"), dpi=300)
    plt.close()

plot_reliability(val_labels, val_probs_cal, "val_reliability_calibrated")
plot_reliability(test_labels, test_probs_cal, "test_reliability_calibrated")


# ============================================================
# ROC + PR
# ============================================================

def plot_roc(y, p, name):
    fpr, tpr, _ = roc_curve(y, p)
    plt.figure()
    plt.plot(fpr, tpr)
    plt.plot([0,1],[0,1])
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.title(name)
    plt.tight_layout()
    plt.savefig(os.path.join(CAL_OUT, f"{name}.png"), dpi=300)
    plt.close()

def plot_pr(y, p, name):
    prec, rec, _ = precision_recall_curve(y, p)
    plt.figure()
    plt.plot(rec, prec)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(name)
    plt.tight_layout()
    plt.savefig(os.path.join(CAL_OUT, f"{name}.png"), dpi=300)
    plt.close()

plot_roc(test_labels, test_probs_cal, "test_roc_calibrated")
plot_pr(test_labels, test_probs_cal, "test_pr_calibrated")

print("\nCalibration complete.")
print("Files saved to:", CAL_OUT)

Calibration output folder: /mnt/heirarchy/models_stage2_hier/acl_analysis_calibrated
Fitting temperature on validation set...
Optimal temperature: 1.1592621803283691

=== CALIBRATED METRICS ===

Validation (Calibrated)
F1 : 0.9843603846297752
Accuracy : 0.9791
Precision : 0.982143301560439
Recall : 0.9865875
AUC : 0.9976982224999998
MCC : 0.9528904144967546
Brier : 0.01609585570076009

Test (Calibrated)
F1 : 0.984594658650068
Accuracy : 0.9794166666666667
Precision : 0.9825478626733388
Recall : 0.98665
AUC : 0.9978013164062499
MCC : 0.9536093986804551
Brier : 0.01571924219831902

Calibration complete.
Files saved to: /mnt/heirarchy/models_stage2_hier/acl_analysis_calibrated


In [35]:
# ============================================================
# FIXED CALIBRATION ANALYSIS (NUMPY SAFE)
# ============================================================

import numpy as np
import json
import os
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

CAL_OUT = "/mnt/heirarchy/models_stage2_hier/acl_analysis_calibrated"

# ------------------------------------------------------------
# Force numpy conversion (critical fix)
# ------------------------------------------------------------

val_y = np.asarray(val_labels)
test_y = np.asarray(test_labels)

val_p_cal = np.asarray(val_probs_cal)
test_p_cal = np.asarray(test_probs_cal)

# ------------------------------------------------------------
# Proper Expected Calibration Error
# Uses confidence directly (not threshold)
# ------------------------------------------------------------

def expected_calibration_error(y, p, n_bins=15):
    y = np.asarray(y)
    p = np.asarray(p)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        mask = (p >= lo) & (p < hi) if i < n_bins - 1 else (p >= lo) & (p <= hi)

        if np.sum(mask) == 0:
            continue

        bin_confidence = np.mean(p[mask])
        bin_accuracy = np.mean(y[mask])
        ece += (np.sum(mask) / len(y)) * abs(bin_accuracy - bin_confidence)

    return float(ece)

ece_val_cal = expected_calibration_error(val_y, val_p_cal)
ece_test_cal = expected_calibration_error(test_y, test_p_cal)

print("\nCalibrated ECE")
print("VAL  ECE:", ece_val_cal)
print("TEST ECE:", ece_test_cal)

# ------------------------------------------------------------
# Retune threshold on calibrated validation
# ------------------------------------------------------------

ts = np.linspace(0.01, 0.99, 99)
best_t_cal = 0.5
best_f1_cal = -1

for t in ts:
    f = f1_score(val_y, (val_p_cal >= t).astype(int))
    if f > best_f1_cal:
        best_f1_cal = f
        best_t_cal = t

print("\nBest calibrated threshold:", best_t_cal)

# ------------------------------------------------------------
# Final calibrated test metrics
# ------------------------------------------------------------

def compute_metrics(y, p, thr):
    pred = (p >= thr).astype(int)
    return {
        "F1": float(f1_score(y, pred)),
        "Accuracy": float(accuracy_score(y, pred)),
        "Precision": float(precision_score(y, pred)),
        "Recall": float(recall_score(y, pred)),
    }

test_cal_tuned = compute_metrics(test_y, test_p_cal, best_t_cal)

print("\nTest metrics after calibration + threshold tuning")
for k, v in test_cal_tuned.items():
    print(k, ":", v)

# ------------------------------------------------------------
# Save summary
# ------------------------------------------------------------

comparison = {
    "temperature": float(T),
    "ECE_val_cal": ece_val_cal,
    "ECE_test_cal": ece_test_cal,
    "best_threshold_calibrated": float(best_t_cal),
    "test_metrics_calibrated_tuned": test_cal_tuned,
}

json.dump(comparison,
          open(os.path.join(CAL_OUT, "calibration_improvement_summary_fixed.json"), "w"),
          indent=2)

print("\nSaved calibration improvement summary (fixed).")


Calibrated ECE
VAL  ECE: 0.004799850522385315
TEST ECE: 0.004461712711801153

Best calibrated threshold: 0.56

Test metrics after calibration + threshold tuning
F1 : 0.9846077055061613
Accuracy : 0.9794416666666667
Precision : 0.9829212083463096
Recall : 0.9863

Saved calibration improvement summary (fixed).


In [36]:
# ============================================================
# ROBUSTNESS + MARGIN + SEPARATION ANALYSIS
# No retraining required
# ============================================================

import numpy as np
import pandas as pd
import os
import json
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp

ROBUST_OUT = "/mnt/heirarchy/models_stage2_hier/acl_analysis_robustness"
os.makedirs(ROBUST_OUT, exist_ok=True)

print("Saving robustness outputs to:", ROBUST_OUT)

y = np.asarray(test_y)
p = np.asarray(test_p_cal)

# ============================================================
# 1. Confidence Margin Distribution
# ============================================================

margin = np.abs(p - 0.5)

plt.figure()
plt.hist(margin[y==0], bins=50, alpha=0.7, label="neg")
plt.hist(margin[y==1], bins=50, alpha=0.7, label="pos")
plt.xlabel("Confidence margin |p - 0.5|")
plt.ylabel("Count")
plt.legend()
plt.title("Confidence Margin Distribution")
plt.tight_layout()
plt.savefig(os.path.join(ROBUST_OUT, "confidence_margin_hist.png"), dpi=300)
plt.close()

# ============================================================
# 2. Logit Margin Analysis
# ============================================================

# Recover calibrated logits
logits_test = torch.tensor(test_logits, dtype=torch.float32)
scaled_logits = logits_test / T
logit_margin = (scaled_logits[:,1] - scaled_logits[:,0]).numpy()

plt.figure()
plt.hist(logit_margin[y==0], bins=50, alpha=0.7, label="neg")
plt.hist(logit_margin[y==1], bins=50, alpha=0.7, label="pos")
plt.xlabel("Logit margin")
plt.ylabel("Count")
plt.legend()
plt.title("Logit Margin Distribution")
plt.tight_layout()
plt.savefig(os.path.join(ROBUST_OUT, "logit_margin_hist.png"), dpi=300)
plt.close()

# ============================================================
# 3. KS Separation Test
# ============================================================

ks_stat, ks_p = ks_2samp(p[y==0], p[y==1])

print("\nKolmogorov-Smirnov separation test")
print("KS statistic:", ks_stat)
print("p-value:", ks_p)

# ============================================================
# 4. Noise Sensitivity
# Add Gaussian noise to logits and evaluate degradation
# ============================================================

def noise_stability(logits, labels, T, noise_std=0.1, trials=20):
    metrics = []
    for _ in range(trials):
        noise = torch.randn_like(logits) * noise_std
        noisy = (logits + noise) / T
        probs = torch.softmax(noisy, dim=1)[:,1].numpy()
        preds = (probs >= 0.56).astype(int)
        acc = np.mean(preds == labels)
        metrics.append(acc)
    return float(np.mean(metrics)), float(np.std(metrics))

mean_acc, std_acc = noise_stability(
    torch.tensor(test_logits, dtype=torch.float32),
    y,
    T,
    noise_std=0.1,
    trials=50
)

print("\nNoise stability test (Gaussian noise std=0.1)")
print("Mean accuracy:", mean_acc)
print("Std accuracy:", std_acc)

# ============================================================
# 5. Probability Separation Statistics
# ============================================================

stats = {
    "mean_neg_prob": float(np.mean(p[y==0])),
    "mean_pos_prob": float(np.mean(p[y==1])),
    "median_neg_prob": float(np.median(p[y==0])),
    "median_pos_prob": float(np.median(p[y==1])),
    "ks_statistic": float(ks_stat),
    "ks_p_value": float(ks_p),
    "noise_mean_acc": mean_acc,
    "noise_std_acc": std_acc
}

json.dump(stats, open(os.path.join(ROBUST_OUT, "robustness_stats.json"), "w"), indent=2)

print("\nRobustness summary:")
for k,v in stats.items():
    print(k, ":", v)

# ============================================================
# 6. Hardest Low-Margin Correct Predictions
# ============================================================

correct = ( (p >= 0.56).astype(int) == y )
low_margin_idx = np.argsort(margin)

hard_correct_idx = [i for i in low_margin_idx if correct[i]][:200]

hard_df = test_df.iloc[hard_correct_idx].copy()
hard_df["prob"] = p[hard_correct_idx]
hard_df["margin"] = margin[hard_correct_idx]

hard_df.to_csv(os.path.join(ROBUST_OUT, "low_margin_correct_cases.csv"), index=False)

print("\nSaved low-margin correct cases.")

print("\nRobustness analysis complete.")
print("Folder:", ROBUST_OUT)

Saving robustness outputs to: /mnt/heirarchy/models_stage2_hier/acl_analysis_robustness

Kolmogorov-Smirnov separation test
KS statistic: 0.95595
p-value: 0.0

Noise stability test (Gaussian noise std=0.1)
Mean accuracy: 0.9794144999999999
Std accuracy: 6.590839602560282e-05

Robustness summary:
mean_neg_prob : 0.0478537417948246
mean_pos_prob : 0.9809612035751343
median_neg_prob : 0.0005981811555102468
median_pos_prob : 0.9998568296432495
ks_statistic : 0.95595
ks_p_value : 0.0
noise_mean_acc : 0.9794144999999999
noise_std_acc : 6.590839602560282e-05

Saved low-margin correct cases.

Robustness analysis complete.
Folder: /mnt/heirarchy/models_stage2_hier/acl_analysis_robustness


In [2]:
# ============================================================
# COMPLETE ACL READY EVALUATION AND ABLATION SUITE
# End to end, single script
# Loads saved main model from storage
# Runs full metrics, plots, tables, calibration, robustness
# Trains missing baselines and missing ablations if checkpoints absent
# Prints progress at every step
# Saves every artifact to one output folder with an index json
# ============================================================

import os
import json
import math
import time
import random
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

from tqdm.auto import tqdm

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    matthews_corrcoef,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    confusion_matrix,
)
from sklearn.calibration import calibration_curve
from sklearn.metrics.pairwise import cosine_similarity

try:
    import umap
    UMAP_AVAILABLE = True
except Exception:
    UMAP_AVAILABLE = False

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup,
)

# ============================================================
# ENV SAFETY
# ============================================================

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ============================================================
# CONFIG
# ============================================================

@dataclass
class CFG:
    data_root: str = "/mnt/heirarchy/data_stage1"
    model_root: str = "/mnt/heirarchy/models_stage2_hier"
    main_ckpt_name: str = "best_model.pt"

    out_dir_name: str = "acl_final_package"
    seed: int = 42

    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    enc_a: str = "microsoft/deberta-v3-base"
    enc_b: str = "roberta-base"

    max_len: int = 128
    batch_size_eval: int = 128
    num_workers_eval: int = 4

    d_lex: int = 256
    d_syn: int = 256
    d_sem: int = 256

    threshold_grid_n: int = 199

    bootstrap_n: int = 1000
    bootstrap_seed: int = 123

    noise_trials: int = 20
    noise_std: float = 0.10

    # Baseline training controls
    train_baselines: bool = True
    train_ablations: bool = True

    # Baseline and ablation training hyperparams
    epochs_small: int = 2
    lr_enc: float = 2e-6
    lr_head: float = 1e-5
    weight_decay: float = 0.01
    grad_clip: float = 1.0
    grad_accum: int = 2
    warmup_frac: float = 0.10
    use_amp_bf16: bool = True

    baseline_batch_start: int = 96
    baseline_num_workers: int = 4

cfg = CFG()

OUT_DIR = os.path.join(cfg.model_root, cfg.out_dir_name)
os.makedirs(OUT_DIR, exist_ok=True)

PLOTS_DIR = os.path.join(OUT_DIR, "plots")
TABLES_DIR = os.path.join(OUT_DIR, "tables")
CKPT_DIR = os.path.join(OUT_DIR, "checkpoints")
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

print("Device:", cfg.device)
print("Output folder:", OUT_DIR)
print("UMAP available:", UMAP_AVAILABLE)

# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)

# ============================================================
# DATA
# ============================================================

def normalize_text(x: str) -> str:
    return " ".join(str(x).split()).strip()

class DualTokPairDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tok_a, tok_b, max_len: int):
        self.df = df.reset_index(drop=True)
        self.ta = tok_a
        self.tb = tok_b
        self.max_len = max_len

        required = {"sentence1", "sentence2", "label"}
        missing = required - set(self.df.columns)
        if missing:
            raise ValueError(f"Missing columns: {missing}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        r = self.df.iloc[idx]
        s1 = normalize_text(r["sentence1"])
        s2 = normalize_text(r["sentence2"])
        ea = self.ta(
            s1, s2,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        eb = self.tb(
            s1, s2,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        item = {
            "input_ids_a": ea["input_ids"].squeeze(0),
            "attention_mask_a": ea["attention_mask"].squeeze(0),
            "input_ids_b": eb["input_ids"].squeeze(0),
            "attention_mask_b": eb["attention_mask"].squeeze(0),
            "labels": torch.tensor(int(r["label"]), dtype=torch.long),
        }
        return item

def collate_stack(batch_list: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
    out = {}
    for k in batch_list[0].keys():
        out[k] = torch.stack([b[k] for b in batch_list], dim=0)
    return out

def make_loader_dualtok(
    df: pd.DataFrame,
    tok_a,
    tok_b,
    batch_size: int,
    shuffle: bool,
    num_workers: int,
) -> DataLoader:
    ds = DualTokPairDataset(df, tok_a, tok_b, cfg.max_len)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=(num_workers > 0),
        collate_fn=collate_stack,
    )

def load_splits(data_root: str) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train_path = os.path.join(data_root, "train.csv")
    val_path = os.path.join(data_root, "val.csv")
    test_path = os.path.join(data_root, "test.csv")
    train_df = pd.read_csv(train_path)
    val_df = pd.read_csv(val_path)
    test_df = pd.read_csv(test_path)
    return train_df, val_df, test_df

train_df, val_df, test_df = load_splits(cfg.data_root)
print("Loaded data sizes:", len(train_df), len(val_df), len(test_df))
print("Label rates:", {
    "train_pos": float(train_df["label"].mean()),
    "val_pos": float(val_df["label"].mean()),
    "test_pos": float(test_df["label"].mean()),
})

tok_a = AutoTokenizer.from_pretrained(cfg.enc_a, use_fast=True)
tok_b = AutoTokenizer.from_pretrained(cfg.enc_b, use_fast=True)

val_loader = make_loader_dualtok(val_df, tok_a, tok_b, cfg.batch_size_eval, False, cfg.num_workers_eval)
test_loader = make_loader_dualtok(test_df, tok_a, tok_b, cfg.batch_size_eval, False, cfg.num_workers_eval)

# ============================================================
# MODEL DEFINITIONS
# ============================================================

class AdditiveAttnPool(nn.Module):
    def __init__(self, d_in: int, d_mid: int = 384):
        super().__init__()
        self.fc1 = nn.Linear(d_in, d_mid)
        self.fc2 = nn.Linear(d_mid, 1)

    def forward(self, H: torch.Tensor, mask_keep: torch.Tensor) -> torch.Tensor:
        s = self.fc2(torch.tanh(self.fc1(H))).squeeze(-1)
        s = s.masked_fill(mask_keep == 0, -1e9)
        a = torch.softmax(s, dim=-1).unsqueeze(-1)
        return torch.sum(a * H, dim=1)

class CrossAttentionBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int = 8, dropout: float = 0.0):
        super().__init__()
        self.mha = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.ln = nn.LayerNorm(d_model)

    def forward(
        self,
        q: torch.Tensor,
        k: torch.Tensor,
        v: torch.Tensor,
        key_padding_mask: torch.Tensor,
    ) -> torch.Tensor:
        attn_out, _ = self.mha(q, k, v, key_padding_mask=key_padding_mask, need_weights=False)
        return self.ln(q + attn_out)

class GatedBilinearFuse(nn.Module):
    def __init__(self, d: int):
        super().__init__()
        self.align_b = nn.Linear(d, d)
        self.bilin = nn.Bilinear(d, d, d)
        self.gate = nn.Sequential(nn.Linear(2 * d, d), nn.Sigmoid())
        self.proj = nn.Linear(d, d)

    def forward(self, a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
        b2 = self.align_b(b)
        bil = self.bilin(a, b2)
        g = self.gate(torch.cat([a, b2], dim=-1))
        fused = g * bil + (1.0 - g) * a
        return self.proj(fused)

def orthogonality_penalty(Slex: nn.Linear, Ssyn: nn.Linear, Ssem: nn.Linear) -> torch.Tensor:
    Wl = Slex.weight
    Ws = Ssyn.weight
    Wm = Ssem.weight

    def normed_dot(A, B):
        return torch.sum(A * B) / (torch.norm(A) * torch.norm(B) + 1e-8)

    return (normed_dot(Wl, Ws).abs() + normed_dot(Wl, Wm).abs() + normed_dot(Ws, Wm).abs()) / 3.0

def simplex_penalty(Slex: nn.Linear, Ssyn: nn.Linear, Ssem: nn.Linear, d_in: int) -> torch.Tensor:
    Wl = Slex.weight
    Ws = Ssyn.weight
    Wm = Ssem.weight
    A = Wl.t() @ Wl + Ws.t() @ Ws + Wm.t() @ Wm
    I = torch.eye(d_in, device=A.device, dtype=A.dtype)
    return torch.mean((A - I) ** 2)

class HierDualEncoderDualTok(nn.Module):
    def __init__(self, enc_a_name: str, enc_b_name: str, d_lex: int, d_syn: int, d_sem: int):
        super().__init__()
        self.enc_a = AutoModel.from_pretrained(enc_a_name)
        self.enc_b = AutoModel.from_pretrained(enc_b_name)

        da = int(self.enc_a.config.hidden_size)
        db = int(self.enc_b.config.hidden_size)
        if da != db:
            raise ValueError(f"Hidden sizes must match, got {da} and {db}")

        self.d_in = da

        self.Slex = nn.Linear(self.d_in, d_lex, bias=False)
        self.Ssyn = nn.Linear(self.d_in, d_syn, bias=False)
        self.Ssem = nn.Linear(self.d_in, d_sem, bias=False)

        self.pool_lex = AdditiveAttnPool(d_lex)
        self.pool_syn = AdditiveAttnPool(d_syn)
        self.pool_sem = AdditiveAttnPool(d_sem)

        self.cross12 = CrossAttentionBlock(d_sem, n_heads=8)
        self.cross21 = CrossAttentionBlock(d_sem, n_heads=8)

        self.fuse_lex = GatedBilinearFuse(d_lex)
        self.fuse_syn = GatedBilinearFuse(d_syn)
        self.fuse_sem = GatedBilinearFuse(d_sem)

        self.head_lex = nn.Linear(d_lex, 2)
        self.head_syn = nn.Linear(d_syn, 2)
        self.head_sem = nn.Linear(d_sem, 2)

        meta_in = (2 + 2 + 2) + (d_lex + d_syn + d_sem)
        self.meta = nn.Sequential(
            nn.Linear(meta_in, 512),
            nn.GELU(),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Linear(256, 2),
        )

    def encode(self, encoder: AutoModel, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        out = encoder(input_ids=input_ids, attention_mask=attention_mask)
        return out.last_hidden_state

    def forward(
        self,
        input_ids_a: torch.Tensor,
        attention_mask_a: torch.Tensor,
        input_ids_b: torch.Tensor,
        attention_mask_b: torch.Tensor,
        return_features: bool = False,
    ) -> Dict[str, torch.Tensor]:
        Ha = self.encode(self.enc_a, input_ids_a, attention_mask_a)
        Hb = self.encode(self.enc_b, input_ids_b, attention_mask_b)

        mask_a = attention_mask_a
        mask_b = attention_mask_b
        kpm_a = (mask_a == 0)
        kpm_b = (mask_b == 0)

        Ha_lex = self.Slex(Ha)
        Hb_lex = self.Slex(Hb)
        Ha_syn = self.Ssyn(Ha)
        Hb_syn = self.Ssyn(Hb)
        Ha_sem = self.Ssem(Ha)
        Hb_sem = self.Ssem(Hb)

        fa_lex = self.pool_lex(Ha_lex, mask_a)
        fb_lex = self.pool_lex(Hb_lex, mask_b)
        fa_syn = self.pool_syn(Ha_syn, mask_a)
        fb_syn = self.pool_syn(Hb_syn, mask_b)

        Ha_sem_x = self.cross12(Ha_sem, Hb_sem, Hb_sem, key_padding_mask=kpm_b)
        Hb_sem_x = self.cross21(Hb_sem, Ha_sem, Ha_sem, key_padding_mask=kpm_a)

        fa_sem = self.pool_sem(Ha_sem_x, mask_a)
        fb_sem = self.pool_sem(Hb_sem_x, mask_b)

        z_lex = self.fuse_lex(fa_lex, fb_lex)
        z_syn = self.fuse_syn(fa_syn, fb_syn)
        z_sem = self.fuse_sem(fa_sem, fb_sem)

        log_lex = self.head_lex(z_lex)
        log_syn = self.head_syn(z_syn)
        log_sem = self.head_sem(z_sem)

        meta_in = torch.cat([log_lex, log_syn, log_sem, z_lex, z_syn, z_sem], dim=-1)
        log_meta = self.meta(meta_in)

        out = {"log_meta": log_meta, "log_lex": log_lex, "log_syn": log_syn, "log_sem": log_sem}
        if return_features:
            out.update({"z_lex": z_lex, "z_syn": z_syn, "z_sem": z_sem})
        return out

# Fusion baseline without disentanglement
class DualEncoderFusionNoDisentangle(nn.Module):
    def __init__(self, enc_a_name: str, enc_b_name: str):
        super().__init__()
        self.enc_a = AutoModel.from_pretrained(enc_a_name)
        self.enc_b = AutoModel.from_pretrained(enc_b_name)
        da = int(self.enc_a.config.hidden_size)
        db = int(self.enc_b.config.hidden_size)
        self.da = da
        self.db = db

        self.proj_a = nn.Linear(da, 256)
        self.proj_b = nn.Linear(db, 256)

        self.cls = nn.Sequential(
            nn.Linear(512, 512),
            nn.GELU(),
            nn.Linear(512, 2),
        )

    def forward(self, ids_a, mask_a, ids_b, mask_b):
        Ha = self.enc_a(input_ids=ids_a, attention_mask=mask_a).last_hidden_state
        Hb = self.enc_b(input_ids=ids_b, attention_mask=mask_b).last_hidden_state

        fa = Ha[:, 0, :]
        fb = Hb[:, 0, :]

        za = self.proj_a(fa)
        zb = self.proj_b(fb)

        logits = self.cls(torch.cat([za, zb], dim=-1))
        return {"logits": logits, "za": za, "zb": zb}

# ============================================================
# CHECKPOINT LOADING
# ============================================================

def load_state_strip_compile_prefix(state_dict: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    out = {}
    for k, v in state_dict.items():
        if k.startswith("_orig_mod."):
            out[k.replace("_orig_mod.", "", 1)] = v
        else:
            out[k] = v
    return out

MAIN_CKPT_PATH = os.path.join(cfg.model_root, cfg.main_ckpt_name)
assert os.path.exists(MAIN_CKPT_PATH), f"Missing main checkpoint: {MAIN_CKPT_PATH}"

print("Loading main hierarchical model checkpoint:", MAIN_CKPT_PATH)
main_model = HierDualEncoderDualTok(cfg.enc_a, cfg.enc_b, cfg.d_lex, cfg.d_syn, cfg.d_sem).to(cfg.device)
state = torch.load(MAIN_CKPT_PATH, map_location=cfg.device)
state = load_state_strip_compile_prefix(state)
main_model.load_state_dict(state, strict=True)
main_model.eval()
print("Main model loaded.")

# ============================================================
# METRICS AND UTILS
# ============================================================

def probs_from_logits_2(logits_2: np.ndarray) -> np.ndarray:
    t = torch.tensor(logits_2)
    p = torch.softmax(t, dim=1).numpy()[:, 1]
    return p

def threshold_tune_f1(y: np.ndarray, p: np.ndarray, grid_n: int) -> float:
    ts = np.linspace(0.01, 0.99, grid_n)
    best_t = 0.5
    best_f = -1.0
    for t in ts:
        f = f1_score(y, (p >= t).astype(int))
        if f > best_f:
            best_f = f
            best_t = float(t)
    return best_t

def compute_metrics_binary(y: np.ndarray, p: np.ndarray, thr: float) -> Dict[str, float]:
    pred = (p >= thr).astype(int)
    f1 = float(f1_score(y, pred))
    acc = float(accuracy_score(y, pred))
    prec = float(precision_score(y, pred, zero_division=0))
    rec = float(recall_score(y, pred, zero_division=0))
    auc_roc = float(roc_auc_score(y, p))
    auc_pr = float(average_precision_score(y, p))
    mcc = float(matthews_corrcoef(y, pred))
    brier = float(np.mean((p - y) ** 2))
    return {
        "F1": f1,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "AUC_ROC": auc_roc,
        "AUC_PR": auc_pr,
        "MCC": mcc,
        "Brier": brier,
        "Threshold": float(thr),
        "PosRate": float(np.mean(y)),
        "PredPosRate": float(np.mean(pred)),
    }

def expected_calibration_error_numpy(y: np.ndarray, p: np.ndarray, n_bins: int = 15, thr_for_acc: float = 0.5) -> float:
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    N = len(y)
    for i in range(n_bins):
        lo = bins[i]
        hi = bins[i + 1]
        mask = (p >= lo) & (p < hi) if i < n_bins - 1 else (p >= lo) & (p <= hi)
        if np.sum(mask) == 0:
            continue
        acc = np.mean(((p[mask] >= thr_for_acc).astype(int) == y[mask]).astype(float))
        conf = float(np.mean(p[mask]))
        ece += (np.sum(mask) / N) * abs(acc - conf)
    return float(ece)

def save_json(path: str, obj):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def save_fig(path: str):
    plt.tight_layout()
    plt.savefig(path, dpi=300)
    plt.close()

def print_df(df: pd.DataFrame, title: str):
    print("\n" + title)
    with pd.option_context("display.max_rows", 200, "display.max_columns", 200, "display.width", 180):
        print(df)

# ============================================================
# INFERENCE FOR MAIN MODEL
# ============================================================

@torch.no_grad()
def run_main_inference(model: HierDualEncoderDualTok, loader: DataLoader, name: str) -> Dict[str, np.ndarray]:
    model.eval()
    ys = []
    logits = []
    z_lex = []
    z_syn = []
    z_sem = []

    start = time.time()
    for i, batch in enumerate(loader):
        batch = {k: v.to(cfg.device, non_blocking=True) for k, v in batch.items()}
        use_amp = (cfg.device.startswith("cuda") and cfg.use_amp_bf16)
        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_amp):
            out = model(
                batch["input_ids_a"], batch["attention_mask_a"],
                batch["input_ids_b"], batch["attention_mask_b"],
                return_features=True,
            )
        ys.append(batch["labels"].detach().cpu().numpy())
        logits.append(out["log_meta"].detach().float().cpu().numpy())
        z_lex.append(out["z_lex"].detach().float().cpu().numpy())
        z_syn.append(out["z_syn"].detach().float().cpu().numpy())
        z_sem.append(out["z_sem"].detach().float().cpu().numpy())

        if i == 0:
            print(name, "first batch log_meta shape:", tuple(out["log_meta"].shape))
        if (i % 25) == 0:
            elapsed = time.time() - start
            print(f"{name} batch {i}/{len(loader)} elapsed {elapsed:.1f}s", flush=True)

    y = np.concatenate(ys)
    log = np.concatenate(logits)
    p = probs_from_logits_2(log)
    return {
        "y": y,
        "logits": log,
        "p": p,
        "z_lex": np.concatenate(z_lex),
        "z_syn": np.concatenate(z_syn),
        "z_sem": np.concatenate(z_sem),
    }

print("\nRunning main inference on VAL")
val_main = run_main_inference(main_model, val_loader, "VAL_MAIN")

print("\nRunning main inference on TEST")
test_main = run_main_inference(main_model, test_loader, "TEST_MAIN")

thr_main = threshold_tune_f1(val_main["y"], val_main["p"], cfg.threshold_grid_n)
metrics_val_main = compute_metrics_binary(val_main["y"], val_main["p"], thr_main)
metrics_test_main = compute_metrics_binary(test_main["y"], test_main["p"], thr_main)

metrics_val_main["ECE"] = expected_calibration_error_numpy(val_main["y"], val_main["p"], 15, thr_for_acc=thr_main)
metrics_test_main["ECE"] = expected_calibration_error_numpy(test_main["y"], test_main["p"], 15, thr_for_acc=thr_main)

main_summary = pd.DataFrame([
    {"Model": "Main_Hier", "Split": "val", **metrics_val_main},
    {"Model": "Main_Hier", "Split": "test", **metrics_test_main},
])

print_df(main_summary, "MAIN MODEL METRICS")
main_summary.to_csv(os.path.join(TABLES_DIR, "main_metrics.csv"), index=False)

# ============================================================
# PLOTS FOR MAIN MODEL
# ============================================================

def plot_roc_pr(y: np.ndarray, p: np.ndarray, prefix: str):
    fpr, tpr, _ = roc_curve(y, p)
    plt.figure()
    plt.plot(fpr, tpr)
    plt.plot([0, 1], [0, 1])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(prefix + " ROC")
    save_fig(os.path.join(PLOTS_DIR, f"{prefix.lower()}_roc.png"))

    prec, rec, _ = precision_recall_curve(y, p)
    plt.figure()
    plt.plot(rec, prec)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(prefix + " Precision Recall")
    save_fig(os.path.join(PLOTS_DIR, f"{prefix.lower()}_pr.png"))

def plot_confusion(y: np.ndarray, p: np.ndarray, thr: float, prefix: str):
    pred = (p >= thr).astype(int)
    cm = confusion_matrix(y, pred)
    plt.figure()
    plt.imshow(cm)
    plt.xticks([0, 1], ["neg", "pos"])
    plt.yticks([0, 1], ["neg", "pos"])
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(prefix + " Confusion Matrix")
    for i in range(2):
        for j in range(2):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center")
    save_fig(os.path.join(PLOTS_DIR, f"{prefix.lower()}_confusion.png"))

def plot_calibration(y: np.ndarray, p: np.ndarray, prefix: str):
    frac_pos, mean_pred = calibration_curve(y, p, n_bins=15)
    plt.figure()
    plt.plot(mean_pred, frac_pos, marker="o")
    plt.plot([0, 1], [0, 1])
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Fraction positive")
    plt.title(prefix + " Calibration")
    save_fig(os.path.join(PLOTS_DIR, f"{prefix.lower()}_calibration.png"))

def plot_pca(z: np.ndarray, y: np.ndarray, prefix: str):
    from sklearn.decomposition import PCA
    pca = PCA(n_components=2)
    emb = pca.fit_transform(z)
    plt.figure()
    plt.scatter(emb[y == 0, 0], emb[y == 0, 1], s=2, label="neg")
    plt.scatter(emb[y == 1, 0], emb[y == 1, 1], s=2, label="pos")
    plt.legend()
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.title(prefix + " PCA (z_sem)")
    save_fig(os.path.join(PLOTS_DIR, f"{prefix.lower()}_pca_zsem.png"))

def plot_umap(z: np.ndarray, y: np.ndarray, prefix: str, max_points: int = 40000):
    if not UMAP_AVAILABLE:
        return
    n = min(len(z), max_points)
    reducer = umap.UMAP(n_neighbors=30, min_dist=0.1, metric="cosine")
    emb = reducer.fit_transform(z[:n])
    yy = y[:n]
    plt.figure()
    plt.scatter(emb[yy == 0, 0], emb[yy == 0, 1], s=2, label="neg")
    plt.scatter(emb[yy == 1, 0], emb[yy == 1, 1], s=2, label="pos")
    plt.legend()
    plt.xlabel("UMAP1")
    plt.ylabel("UMAP2")
    plt.title(prefix + " UMAP (z_sem)")
    save_fig(os.path.join(PLOTS_DIR, f"{prefix.lower()}_umap_zsem.png"))

print("\nSaving main plots")
plot_roc_pr(val_main["y"], val_main["p"], "VAL_MAIN")
plot_roc_pr(test_main["y"], test_main["p"], "TEST_MAIN")
plot_confusion(val_main["y"], val_main["p"], thr_main, "VAL_MAIN")
plot_confusion(test_main["y"], test_main["p"], thr_main, "TEST_MAIN")
plot_calibration(val_main["y"], val_main["p"], "VAL_MAIN")
plot_calibration(test_main["y"], test_main["p"], "TEST_MAIN")
plot_pca(val_main["z_sem"], val_main["y"], "VAL_MAIN")
plot_pca(test_main["z_sem"], test_main["y"], "TEST_MAIN")
plot_umap(val_main["z_sem"], val_main["y"], "VAL_MAIN")
plot_umap(test_main["z_sem"], test_main["y"], "TEST_MAIN")

# Cosine heatmap small subset
print("Saving cosine similarity heatmap on small subset")
sub_n = min(2000, len(val_main["z_sem"]))
sim = cosine_similarity(val_main["z_sem"][:sub_n])
plt.figure(figsize=(6, 5))
plt.imshow(sim)
plt.title("VAL cosine similarity heatmap (z_sem subset)")
plt.xlabel("Index")
plt.ylabel("Index")
save_fig(os.path.join(PLOTS_DIR, "val_cosine_heatmap_zsem.png"))

# ============================================================
# MAJORITY AND RANDOM BASELINES (NO TRAINING)
# ============================================================

def majority_baseline_metrics(df: pd.DataFrame) -> Dict[str, float]:
    y = df["label"].to_numpy().astype(int)
    maj = int(np.round(np.mean(y)))
    pred = np.full_like(y, maj)
    p = pred.astype(float)
    return {
        "F1": float(f1_score(y, pred)),
        "Accuracy": float(accuracy_score(y, pred)),
        "Precision": float(precision_score(y, pred, zero_division=0)),
        "Recall": float(recall_score(y, pred, zero_division=0)),
        "AUC_ROC": float(roc_auc_score(y, p)),
        "AUC_PR": float(average_precision_score(y, p)),
        "MCC": float(matthews_corrcoef(y, pred)),
        "Brier": float(np.mean((p - y) ** 2)),
        "Threshold": 0.5,
        "PosRate": float(np.mean(y)),
        "PredPosRate": float(np.mean(pred)),
        "ECE": expected_calibration_error_numpy(y, p, 15, thr_for_acc=0.5),
    }

def random_baseline_metrics(df: pd.DataFrame, seed: int = 0) -> Dict[str, float]:
    rng = np.random.RandomState(seed)
    y = df["label"].to_numpy().astype(int)
    p = rng.rand(len(y)).astype(float)
    pred = (p >= 0.5).astype(int)
    return {
        "F1": float(f1_score(y, pred)),
        "Accuracy": float(accuracy_score(y, pred)),
        "Precision": float(precision_score(y, pred, zero_division=0)),
        "Recall": float(recall_score(y, pred, zero_division=0)),
        "AUC_ROC": float(roc_auc_score(y, p)),
        "AUC_PR": float(average_precision_score(y, p)),
        "MCC": float(matthews_corrcoef(y, pred)),
        "Brier": float(np.mean((p - y) ** 2)),
        "Threshold": 0.5,
        "PosRate": float(np.mean(y)),
        "PredPosRate": float(np.mean(pred)),
        "ECE": expected_calibration_error_numpy(y, p, 15, thr_for_acc=0.5),
    }

maj_val = majority_baseline_metrics(val_df)
maj_test = majority_baseline_metrics(test_df)
rnd_val = random_baseline_metrics(val_df, seed=1)
rnd_test = random_baseline_metrics(test_df, seed=1)

baseline_simple = pd.DataFrame([
    {"Model": "Majority", "Split": "val", **maj_val},
    {"Model": "Majority", "Split": "test", **maj_test},
    {"Model": "Random", "Split": "val", **rnd_val},
    {"Model": "Random", "Split": "test", **rnd_test},
])
print_df(baseline_simple, "SIMPLE BASELINES")
baseline_simple.to_csv(os.path.join(TABLES_DIR, "baselines_simple.csv"), index=False)

# ============================================================
# TRAINING HELPERS FOR TRAINED BASELINES AND ABLATIONS
# ============================================================

def build_optimizer_groups(model: nn.Module, lr_enc: float, lr_head: float, weight_decay: float):
    enc_params = []
    head_params = []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if n.startswith("enc_a.") or n.startswith("enc_b.") or n.startswith("roberta.") or n.startswith("deberta."):
            enc_params.append(p)
        else:
            head_params.append(p)
    return torch.optim.AdamW(
        [{"params": enc_params, "lr": lr_enc}, {"params": head_params, "lr": lr_head}],
        weight_decay=weight_decay,
    )

@torch.no_grad()
def eval_logits_and_probs_binary(
    model: nn.Module,
    loader: DataLoader,
    device: str,
    kind: str,
) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    ys = []
    logits = []
    use_amp = (device.startswith("cuda") and cfg.use_amp_bf16)
    for i, batch in enumerate(loader):
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_amp):
            out = model_forward_logits(model, batch, kind)
        ys.append(batch["labels"].detach().cpu().numpy())
        logits.append(out.detach().float().cpu().numpy())
        if (i % 50) == 0:
            print(f"Eval {kind} batch {i}/{len(loader)}", flush=True)
    y = np.concatenate(ys)
    log = np.concatenate(logits)
    p = probs_from_logits_2(log)
    return y, p

def model_forward_logits(model: nn.Module, batch: Dict[str, torch.Tensor], kind: str) -> torch.Tensor:
    if kind == "fusion_no_disentangle":
        out = model(batch["input_ids_a"], batch["attention_mask_a"], batch["input_ids_b"], batch["attention_mask_b"])
        return out["logits"]
    if kind == "main_hier":
        out = model(batch["input_ids_a"], batch["attention_mask_a"], batch["input_ids_b"], batch["attention_mask_b"])
        return out["log_meta"]
    if kind == "single_a":
        out = model(input_ids=batch["input_ids_a"], attention_mask=batch["attention_mask_a"])
        return out.logits
    if kind == "single_b":
        out = model(input_ids=batch["input_ids_b"], attention_mask=batch["attention_mask_b"])
        return out.logits
    raise ValueError("Unknown kind")

def train_classifier(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: str,
    kind: str,
    out_ckpt_path: str,
    epochs: int,
    lr_enc: float,
    lr_head: float,
    weight_decay: float,
    grad_clip: float,
    grad_accum: int,
    warmup_frac: float,
) -> Dict[str, float]:
    model.to(device)

    opt = build_optimizer_groups(model, lr_enc, lr_head, weight_decay)

    total_steps = int(math.ceil(len(train_loader) / max(1, grad_accum)) * epochs)
    warmup_steps = int(warmup_frac * total_steps)
    sched = get_cosine_schedule_with_warmup(opt, warmup_steps, total_steps)

    best_f1 = -1.0
    best_thr = 0.5
    best_state = None

    use_amp = (device.startswith("cuda") and cfg.use_amp_bf16)

    print(f"\nTraining {kind}")
    print("Total steps:", total_steps, "Warmup:", warmup_steps)

    global_step = 0
    for ep in range(epochs):
        model.train()
        opt.zero_grad(set_to_none=True)

        pbar = tqdm(train_loader, desc=f"{kind} epoch {ep+1}/{epochs}", leave=True)
        running_loss = 0.0
        denom = 0

        for step, batch in enumerate(pbar):
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            y = batch["labels"]

            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_amp):
                logits2 = model_forward_logits(model, batch, kind)
                loss = F.cross_entropy(logits2, y)
                loss = loss / float(grad_accum)

            loss.backward()

            if ((step + 1) % grad_accum) == 0:
                if grad_clip and grad_clip > 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                opt.step()
                sched.step()
                opt.zero_grad(set_to_none=True)
                global_step += 1

            running_loss += float(loss.detach().cpu().item()) * float(grad_accum)
            denom += 1

            if (step % 50) == 0:
                pbar.set_postfix(loss=running_loss / max(1, denom), lr=float(sched.get_last_lr()[0]))

        yv, pv = eval_logits_and_probs_binary(model, val_loader, device, kind)
        thr = threshold_tune_f1(yv, pv, cfg.threshold_grid_n)
        met = compute_metrics_binary(yv, pv, thr)
        print(f"\n{kind} epoch {ep+1} val F1 {met['F1']:.6f} thr {thr:.4f}")

        if met["F1"] > best_f1:
            best_f1 = met["F1"]
            best_thr = thr
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}

    if best_state is not None:
        torch.save({"state_dict": best_state, "best_thr": best_thr}, out_ckpt_path)
        print("Saved best checkpoint:", out_ckpt_path)

    return {"best_f1_val": float(best_f1), "best_thr": float(best_thr)}

# ============================================================
# TRAINED BASELINES
# 1) single encoder DeBERTa
# 2) single encoder RoBERTa
# 3) dual encoder fusion without disentanglement
# ============================================================

baseline_rows = []

def load_or_train_single(kind: str, model_name: str, ckpt_name: str):
    ckpt_path = os.path.join(CKPT_DIR, ckpt_name)
    if os.path.exists(ckpt_path):
        print("Loading baseline checkpoint:", ckpt_path)
        obj = torch.load(ckpt_path, map_location="cpu")
        return ckpt_path, obj["best_thr"]

    if not cfg.train_baselines:
        raise RuntimeError("Baseline checkpoint missing and train_baselines is False")

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    bs = cfg.baseline_batch_start

    train_loader = make_loader_dualtok(train_df, tok_a, tok_b, bs, True, cfg.baseline_num_workers)
    val_loader_local = make_loader_dualtok(val_df, tok_a, tok_b, bs, False, cfg.baseline_num_workers)

    stats = train_classifier(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader_local,
        device=cfg.device,
        kind=kind,
        out_ckpt_path=ckpt_path,
        epochs=cfg.epochs_small,
        lr_enc=cfg.lr_enc,
        lr_head=cfg.lr_head,
        weight_decay=cfg.weight_decay,
        grad_clip=cfg.grad_clip,
        grad_accum=cfg.grad_accum,
        warmup_frac=cfg.warmup_frac,
    )
    return ckpt_path, stats["best_thr"]

def load_or_train_fusion_no_disentangle():
    ckpt_path = os.path.join(CKPT_DIR, "fusion_no_disentangle.pt")
    if os.path.exists(ckpt_path):
        print("Loading fusion baseline checkpoint:", ckpt_path)
        obj = torch.load(ckpt_path, map_location="cpu")
        return ckpt_path, obj["best_thr"]

    if not cfg.train_baselines:
        raise RuntimeError("Fusion baseline checkpoint missing and train_baselines is False")

    model = DualEncoderFusionNoDisentangle(cfg.enc_a, cfg.enc_b)
    bs = max(32, cfg.baseline_batch_start // 2)

    train_loader = make_loader_dualtok(train_df, tok_a, tok_b, bs, True, cfg.baseline_num_workers)
    val_loader_local = make_loader_dualtok(val_df, tok_a, tok_b, bs, False, cfg.baseline_num_workers)

    stats = train_classifier(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader_local,
        device=cfg.device,
        kind="fusion_no_disentangle",
        out_ckpt_path=ckpt_path,
        epochs=cfg.epochs_small,
        lr_enc=cfg.lr_enc,
        lr_head=cfg.lr_head,
        weight_decay=cfg.weight_decay,
        grad_clip=cfg.grad_clip,
        grad_accum=cfg.grad_accum,
        warmup_frac=cfg.warmup_frac,
    )
    return ckpt_path, stats["best_thr"]

def evaluate_saved_checkpoint(kind: str, ckpt_path: str, thr: float) -> pd.DataFrame:
    if kind == "single_a":
        model = AutoModelForSequenceClassification.from_pretrained(cfg.enc_a, num_labels=2)
    elif kind == "single_b":
        model = AutoModelForSequenceClassification.from_pretrained(cfg.enc_b, num_labels=2)
    elif kind == "fusion_no_disentangle":
        model = DualEncoderFusionNoDisentangle(cfg.enc_a, cfg.enc_b)
    else:
        raise ValueError("Unknown kind")

    obj = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(obj["state_dict"], strict=True)
    model.to(cfg.device)
    model.eval()

    yv, pv = eval_logits_and_probs_binary(model, val_loader, cfg.device, kind)
    yt, pt = eval_logits_and_probs_binary(model, test_loader, cfg.device, kind)

    thr_use = thr
    if thr_use is None:
        thr_use = threshold_tune_f1(yv, pv, cfg.threshold_grid_n)

    mv = compute_metrics_binary(yv, pv, float(thr_use))
    mt = compute_metrics_binary(yt, pt, float(thr_use))
    mv["ECE"] = expected_calibration_error_numpy(yv, pv, 15, thr_for_acc=float(thr_use))
    mt["ECE"] = expected_calibration_error_numpy(yt, pt, 15, thr_for_acc=float(thr_use))

    df = pd.DataFrame([
        {"Model": kind, "Split": "val", **mv},
        {"Model": kind, "Split": "test", **mt},
    ])
    return df

print("\nBaselines section starting")

ckpt_deb, thr_deb = load_or_train_single("single_a", cfg.enc_a, "baseline_single_a.pt")
df_deb = evaluate_saved_checkpoint("single_a", ckpt_deb, thr_deb)
print_df(df_deb, "BASELINE SINGLE A")

ckpt_rob, thr_rob = load_or_train_single("single_b", cfg.enc_b, "baseline_single_b.pt")
df_rob = evaluate_saved_checkpoint("single_b", ckpt_rob, thr_rob)
print_df(df_rob, "BASELINE SINGLE B")

ckpt_fus, thr_fus = load_or_train_fusion_no_disentangle()
df_fus = evaluate_saved_checkpoint("fusion_no_disentangle", ckpt_fus, thr_fus)
print_df(df_fus, "BASELINE FUSION NO DISENTANGLE")

baselines_trained = pd.concat([df_deb, df_rob, df_fus], ignore_index=True)
baselines_trained.to_csv(os.path.join(TABLES_DIR, "baselines_trained.csv"), index=False)

# ============================================================
# LOSS WEIGHT SENSITIVITY ABLATION
# This retrains only the heads and meta on top of frozen encoders for speed
# ============================================================

class MainHeadOnlyWrapper(nn.Module):
    def __init__(self, base: HierDualEncoderDualTok, main_w: float, aux_w: float, ortho_w: float, simplex_w: float):
        super().__init__()
        self.base = base
        self.main_w = float(main_w)
        self.aux_w = float(aux_w)
        self.ortho_w = float(ortho_w)
        self.simplex_w = float(simplex_w)

    def forward(self, ids_a, mask_a, ids_b, mask_b, labels=None):
        out = self.base(ids_a, mask_a, ids_b, mask_b, return_features=False)
        if labels is None:
            return out

        loss_main = F.cross_entropy(out["log_meta"], labels)
        loss_lex = F.cross_entropy(out["log_lex"], labels)
        loss_syn = F.cross_entropy(out["log_syn"], labels)
        loss_sem = F.cross_entropy(out["log_sem"], labels)
        loss_aux = (loss_lex + loss_syn + loss_sem) / 3.0

        ort = orthogonality_penalty(self.base.Slex, self.base.Ssyn, self.base.Ssem)
        simp = simplex_penalty(self.base.Slex, self.base.Ssyn, self.base.Ssem, self.base.d_in)

        loss = self.main_w * loss_main + self.aux_w * loss_aux + self.ortho_w * ort + self.simplex_w * simp
        return out, loss

def freeze_encoders_only(model: HierDualEncoderDualTok):
    for n, p in model.named_parameters():
        if n.startswith("enc_a.") or n.startswith("enc_b."):
            p.requires_grad = False
        else:
            p.requires_grad = True

def train_ablation_weights(
    tag: str,
    main_w: float,
    aux_w: float,
    ortho_w: float,
    simplex_w: float,
    epochs: int = 1,
):
    ckpt_path = os.path.join(CKPT_DIR, f"ablation_weights_{tag}.pt")
    if os.path.exists(ckpt_path):
        print("Loading existing weights ablation:", ckpt_path)
        obj = torch.load(ckpt_path, map_location="cpu")
        return ckpt_path, obj["best_thr"]

    if not cfg.train_ablations:
        raise RuntimeError("Ablation checkpoint missing and train_ablations is False")

    base = HierDualEncoderDualTok(cfg.enc_a, cfg.enc_b, cfg.d_lex, cfg.d_syn, cfg.d_sem)
    base.load_state_dict(load_state_strip_compile_prefix(torch.load(MAIN_CKPT_PATH, map_location="cpu")), strict=True)

    freeze_encoders_only(base)
    base.to(cfg.device)

    wrap = MainHeadOnlyWrapper(base, main_w, aux_w, ortho_w, simplex_w).to(cfg.device)

    bs = max(32, cfg.baseline_batch_start // 2)
    train_loader = make_loader_dualtok(train_df, tok_a, tok_b, bs, True, cfg.baseline_num_workers)
    val_loader_local = make_loader_dualtok(val_df, tok_a, tok_b, bs, False, cfg.baseline_num_workers)

    opt = torch.optim.AdamW(
        [p for p in wrap.parameters() if p.requires_grad],
        lr=cfg.lr_head,
        weight_decay=cfg.weight_decay,
    )

    total_steps = int(math.ceil(len(train_loader) / max(1, cfg.grad_accum)) * epochs)
    warmup_steps = int(cfg.warmup_frac * total_steps)
    sched = get_cosine_schedule_with_warmup(opt, warmup_steps, total_steps)

    best_f1 = -1.0
    best_thr = 0.5
    best_state = None

    use_amp = (cfg.device.startswith("cuda") and cfg.use_amp_bf16)

    for ep in range(epochs):
        wrap.train()
        opt.zero_grad(set_to_none=True)
        pbar = tqdm(train_loader, desc=f"ablation {tag} epoch {ep+1}/{epochs}", leave=True)

        for step, batch in enumerate(pbar):
            batch = {k: v.to(cfg.device, non_blocking=True) for k, v in batch.items()}
            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_amp):
                _, loss = wrap(
                    batch["input_ids_a"], batch["attention_mask_a"],
                    batch["input_ids_b"], batch["attention_mask_b"],
                    labels=batch["labels"],
                )
                loss = loss / float(cfg.grad_accum)

            loss.backward()

            if ((step + 1) % cfg.grad_accum) == 0:
                if cfg.grad_clip and cfg.grad_clip > 0:
                    torch.nn.utils.clip_grad_norm_(wrap.parameters(), cfg.grad_clip)
                opt.step()
                sched.step()
                opt.zero_grad(set_to_none=True)

            if (step % 50) == 0:
                pbar.set_postfix(loss=float(loss.detach().cpu().item()) * float(cfg.grad_accum), lr=float(sched.get_last_lr()[0]))

        wrap.eval()
        yv, pv = eval_logits_and_probs_binary(wrap.base, val_loader_local, cfg.device, "main_hier")
        thr = threshold_tune_f1(yv, pv, cfg.threshold_grid_n)
        met = compute_metrics_binary(yv, pv, thr)
        print(f"ablation {tag} val F1 {met['F1']:.6f} thr {thr:.4f}")

        if met["F1"] > best_f1:
            best_f1 = met["F1"]
            best_thr = float(thr)
            best_state = {k: v.detach().cpu() for k, v in wrap.base.state_dict().items()}

    torch.save({"state_dict": best_state, "best_thr": best_thr, "tag": tag, "main_w": main_w, "aux_w": aux_w, "ortho_w": ortho_w, "simplex_w": simplex_w}, ckpt_path)
    print("Saved weights ablation:", ckpt_path)
    return ckpt_path, best_thr

print("\nLoss weight sensitivity ablations starting")

weight_grid = [
    ("w060_040", 0.60, 0.40, 0.02, 0.01),
    ("w080_020", 0.80, 0.20, 0.02, 0.01),
    ("w050_050", 0.50, 0.50, 0.02, 0.01),
    ("w070_030", 0.70, 0.30, 0.02, 0.01),
    ("no_ortho", 0.60, 0.40, 0.00, 0.01),
    ("no_simplex", 0.60, 0.40, 0.02, 0.00),
    ("no_aux", 1.00, 0.00, 0.02, 0.01),
]

ablation_rows = []
for tag, mw, aw, ow, sw in weight_grid:
    ckpt_path, thr = train_ablation_weights(tag, mw, aw, ow, sw, epochs=1)
    df_ab = evaluate_saved_checkpoint("main_hier", ckpt_path, thr)
    df_ab["Model"] = "Ablation_" + tag
    ablation_rows.append(df_ab)

ablations_df = pd.concat(ablation_rows, ignore_index=True)
print_df(ablations_df, "WEIGHT AND REGULARIZER ABLATIONS")
ablations_df.to_csv(os.path.join(TABLES_DIR, "ablations_weights_regularizers.csv"), index=False)

# ============================================================
# CALIBRATION: TEMPERATURE SCALING ON VAL, APPLY TO TEST
# ============================================================

def fit_temperature(val_logits: np.ndarray, val_y: np.ndarray) -> float:
    logits = torch.tensor(val_logits, dtype=torch.float32)
    y = torch.tensor(val_y, dtype=torch.long)

    T = torch.tensor([1.0], dtype=torch.float32, requires_grad=True)
    opt = torch.optim.LBFGS([T], lr=0.1, max_iter=200)

    def closure():
        opt.zero_grad()
        scaled = logits / torch.clamp(T, min=1e-4)
        loss = F.cross_entropy(scaled, y)
        loss.backward()
        return loss

    opt.step(closure)
    return float(T.detach().cpu().item())

print("\nTemperature scaling calibration starting")
T_opt = fit_temperature(val_main["logits"], val_main["y"])
print("Temperature:", T_opt)

def apply_temperature(logits_2: np.ndarray, T: float) -> np.ndarray:
    t = torch.tensor(logits_2, dtype=torch.float32)
    p = torch.softmax(t / float(T), dim=1).numpy()[:, 1]
    return p

val_p_cal = apply_temperature(val_main["logits"], T_opt)
test_p_cal = apply_temperature(test_main["logits"], T_opt)

thr_cal = threshold_tune_f1(val_main["y"], val_p_cal, cfg.threshold_grid_n)

met_val_cal = compute_metrics_binary(val_main["y"], val_p_cal, thr_cal)
met_test_cal = compute_metrics_binary(test_main["y"], test_p_cal, thr_cal)
met_val_cal["ECE"] = expected_calibration_error_numpy(val_main["y"], val_p_cal, 15, thr_for_acc=thr_cal)
met_test_cal["ECE"] = expected_calibration_error_numpy(test_main["y"], test_p_cal, 15, thr_for_acc=thr_cal)

cal_df = pd.DataFrame([
    {"Model": "Main_Hier_TempScaled", "Split": "val", "Temperature": T_opt, **met_val_cal},
    {"Model": "Main_Hier_TempScaled", "Split": "test", "Temperature": T_opt, **met_test_cal},
])
print_df(cal_df, "CALIBRATED METRICS")
cal_df.to_csv(os.path.join(TABLES_DIR, "calibrated_metrics.csv"), index=False)

# Save calibration plot calibrated
print("Saving calibrated calibration plots")
plot_calibration(val_main["y"], val_p_cal, "VAL_MAIN_CAL")
plot_calibration(test_main["y"], test_p_cal, "TEST_MAIN_CAL")

# ============================================================
# BOOTSTRAP CONFIDENCE INTERVALS ON TEST FOR MAIN MODEL
# ============================================================

def bootstrap_ci_metrics(y: np.ndarray, p: np.ndarray, thr: float, n: int, seed: int) -> pd.DataFrame:
    rng = np.random.RandomState(seed)
    N = len(y)
    rows = []
    for b in tqdm(range(n), desc="bootstrap", leave=True):
        idx = rng.randint(0, N, size=N)
        yy = y[idx]
        pp = p[idx]
        m = compute_metrics_binary(yy, pp, thr)
        rows.append(m)

    df = pd.DataFrame(rows)
    out_rows = []
    for col in ["F1", "Accuracy", "Precision", "Recall", "AUC_ROC", "AUC_PR", "MCC", "Brier"]:
        out_rows.append({
            "Metric": col,
            "Mean": float(df[col].mean()),
            "CI_2p5": float(np.quantile(df[col], 0.025)),
            "CI_97p5": float(np.quantile(df[col], 0.975)),
        })
    return pd.DataFrame(out_rows)

print("\nBootstrapping main model test confidence intervals")
ci_df = bootstrap_ci_metrics(test_main["y"], test_main["p"], thr_main, cfg.bootstrap_n, cfg.bootstrap_seed)
print_df(ci_df, "BOOTSTRAP CI TABLE (MAIN TEST)")
ci_df.to_csv(os.path.join(TABLES_DIR, "bootstrap_ci_main_test.csv"), index=False)

# ============================================================
# ROBUSTNESS
# 1) KS separation test
# 2) noise stability on probabilities
# ============================================================

print("\nRobustness analysis starting")

def ks_statistic(y: np.ndarray, p: np.ndarray) -> Dict[str, float]:
    from scipy.stats import ks_2samp
    p0 = p[y == 0]
    p1 = p[y == 1]
    stat, pv = ks_2samp(p0, p1)
    return {"KS_statistic": float(stat), "KS_p_value": float(pv)}

ks_res = ks_statistic(test_main["y"], test_main["p"])
print("KS:", ks_res)

def noise_stability_accuracy(p: np.ndarray, y: np.ndarray, thr: float, noise_std: float, trials: int, seed: int = 0) -> Dict[str, float]:
    rng = np.random.RandomState(seed)
    accs = []
    for _ in range(trials):
        pn = np.clip(p + rng.normal(0.0, noise_std, size=len(p)), 0.0, 1.0)
        pred = (pn >= thr).astype(int)
        accs.append(float(accuracy_score(y, pred)))
    return {"noise_mean_acc": float(np.mean(accs)), "noise_std_acc": float(np.std(accs))}

noise_res = noise_stability_accuracy(test_main["p"], test_main["y"], thr_main, cfg.noise_std, cfg.noise_trials, seed=7)
print("Noise:", noise_res)

robust_df = pd.DataFrame([{
    "Model": "Main_Hier",
    "Split": "test",
    "mean_neg_prob": float(np.mean(test_main["p"][test_main["y"] == 0])),
    "mean_pos_prob": float(np.mean(test_main["p"][test_main["y"] == 1])),
    "median_neg_prob": float(np.median(test_main["p"][test_main["y"] == 0])),
    "median_pos_prob": float(np.median(test_main["p"][test_main["y"] == 1])),
    **ks_res,
    **noise_res,
}])
print_df(robust_df, "ROBUSTNESS SUMMARY")
robust_df.to_csv(os.path.join(TABLES_DIR, "robustness_summary.csv"), index=False)

# Save low margin correct examples
print("Saving low margin correct cases")
test_pred = (test_main["p"] >= thr_main).astype(int)
correct = (test_pred == test_main["y"])
margin = np.abs(test_main["p"] - thr_main)
idx = np.where(correct)[0]
order = idx[np.argsort(margin[idx])]
k = min(500, len(order))
hard_idx = order[:k]
hard_cases = test_df.iloc[hard_idx].copy()
hard_cases["p"] = test_main["p"][hard_idx]
hard_cases["pred"] = test_pred[hard_idx]
hard_cases.to_csv(os.path.join(TABLES_DIR, "hard_correct_low_margin.csv"), index=False)

# ============================================================
# FINAL COMPARISON TABLE
# Includes main, calibrated, baselines, ablations, random, majority
# ============================================================

final_rows = []
final_rows.append(main_summary)
final_rows.append(cal_df.drop(columns=["Temperature"]))
final_rows.append(baseline_simple)
final_rows.append(baselines_trained)
final_rows.append(ablations_df)

final_all = pd.concat(final_rows, ignore_index=True)

# Keep consistent columns ordering
col_order = [
    "Model", "Split",
    "F1", "Accuracy", "Precision", "Recall",
    "AUC_ROC", "AUC_PR", "MCC", "Brier", "ECE",
    "Threshold", "PosRate", "PredPosRate",
]
for c in col_order:
    if c not in final_all.columns:
        final_all[c] = np.nan
final_all = final_all[col_order]

print_df(final_all, "FINAL OVERALL TABLE (ALL MODELS)")
final_all.to_csv(os.path.join(TABLES_DIR, "final_overall_table.csv"), index=False)

# ============================================================
# ARTIFACT INDEX
# ============================================================

artifact_index = {
    "out_dir": OUT_DIR,
    "plots_dir": PLOTS_DIR,
    "tables_dir": TABLES_DIR,
    "checkpoints_dir": CKPT_DIR,
    "main_ckpt_used": MAIN_CKPT_PATH,
    "main_threshold": float(thr_main),
    "calibration_temperature": float(T_opt),
    "files": {
        "tables": sorted([os.path.join(TABLES_DIR, f) for f in os.listdir(TABLES_DIR)]),
        "plots": sorted([os.path.join(PLOTS_DIR, f) for f in os.listdir(PLOTS_DIR)]),
        "checkpoints": sorted([os.path.join(CKPT_DIR, f) for f in os.listdir(CKPT_DIR)]),
    },
    "cfg": asdict(cfg),
}

save_json(os.path.join(OUT_DIR, "artifact_index.json"), artifact_index)

print("\nDONE")
print("Output folder:", OUT_DIR)
print("Index:", os.path.join(OUT_DIR, "artifact_index.json"))

Device: cuda
Output folder: /mnt/heirarchy/models_stage2_hier/acl_final_package
UMAP available: False
Loaded data sizes: 960000 120000 120000
Label rates: {'train_pos': 0.6666666666666666, 'val_pos': 0.6666666666666666, 'test_pos': 0.6666666666666666}


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


...

Eval fusion_no_disentangle batch 0/938
Eval fusion_no_disentangle batch 50/938
Eval fusion_no_disentangle batch 100/938
Eval fusion_no_disentangle batch 150/938
Eval fusion_no_disentangle batch 200/938
Eval fusion_no_disentangle batch 250/938
Eval fusion_no_disentangle batch 300/938
Eval fusion_no_disentangle batch 350/938
Eval fusion_no_disentangle batch 400/938
Eval fusion_no_disentangle batch 450/938
Eval fusion_no_disentangle batch 500/938
Eval fusion_no_disentangle batch 550/938
Eval fusion_no_disentangle batch 600/938
Eval fusion_no_disentangle batch 650/938
Eval fusion_no_disentangle batch 700/938
Eval fusion_no_disentangle batch 750/938
Eval fusion_no_disentangle batch 800/938
Eval fusion_no_disentangle batch 850/938
Eval fusion_no_disentangle batch 900/938
Eval fusion_no_disentangle batch 0/938
Eval fusion_no_disentangle batch 50/938
Eval fusion_no_disentangle batch 100/938
Eval fusion_no_disentangle batch 150/938
Eval fusion_no_disentangle batch 200/938
Eval fusion_no_disenta

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ablation w060_040 epoch 1/1:   0%|          | 0/20000 [00:00<?, ?it/s]

Eval main_hier batch 0/2500
Eval main_hier batch 50/2500
Eval main_hier batch 100/2500
Eval main_hier batch 150/2500
Eval main_hier batch 200/2500
Eval main_hier batch 250/2500
Eval main_hier batch 300/2500
Eval main_hier batch 350/2500
Eval main_hier batch 400/2500
Eval main_hier batch 450/2500
Eval main_hier batch 500/2500
Eval main_hier batch 550/2500
Eval main_hier batch 600/2500
Eval main_hier batch 650/2500
Eval main_hier batch 700/2500
Eval main_hier batch 750/2500
Eval main_hier batch 800/2500
Eval main_hier batch 850/2500
Eval main_hier batch 900/2500
Eval main_hier batch 950/2500
Eval main_hier batch 1000/2500
Eval main_hier batch 1050/2500
Eval main_hier batch 1100/2500
Eval main_hier batch 1150/2500
Eval main_hier batch 1200/2500
Eval main_hier batch 1250/2500
Eval main_hier batch 1300/2500
Eval main_hier batch 1350/2500
Eval main_hier batch 1400/2500
Eval main_hier batch 1450/2500
Eval main_hier batch 1500/2500
Eval main_hier batch 1550/2500
Eval main_hier batch 1600/2500


ValueError: Unknown kind

In [7]:
# ============================================================
# COMPLETE ACL-READY EVALUATION AND ABLATION SUITE — FIXED
# End-to-end, single script
# Loads saved main model from storage
# Runs full metrics, plots, tables, calibration, robustness
# Trains missing baselines and missing ablations if checkpoints absent
# Prints progress at every step
# Saves every artifact to one output folder with an index json
#
# FIXES OVER ORIGINAL:
#   1. evaluate_saved_checkpoint now handles "main_hier" kind
#   2. model_forward_logits handles "main_hier" properly for all wrappers
#   3. Added structural probing (lexical overlap, syntactic depth, STS)
#   4. Added CKA (Centered Kernel Alignment) between subspaces
#   5. Added inference latency benchmarking across all models
#   6. Added per-model bootstrap CIs and paired bootstrap significance
#   7. Better error handling throughout
#   8. Cleaner ablation checkpoint evaluation flow
# ============================================================

import os
import sys
import json
import math
import time
import random
import warnings
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    matthews_corrcoef,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    confusion_matrix,
)
from sklearn.calibration import calibration_curve
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

try:
    import umap
    UMAP_AVAILABLE = True
except Exception:
    UMAP_AVAILABLE = False

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup,
)

warnings.filterwarnings("ignore", category=FutureWarning)

# ============================================================
# ENV SAFETY
# ============================================================

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ============================================================
# CONFIG
# ============================================================

@dataclass
class CFG:
    data_root: str = "/mnt/heirarchy/data_stage1"
    model_root: str = "/mnt/heirarchy/models_stage2_hier"
    main_ckpt_name: str = "best_model.pt"

    out_dir_name: str = "acl_final_package"
    seed: int = 42

    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    enc_a: str = "microsoft/deberta-v3-base"
    enc_b: str = "roberta-base"

    max_len: int = 128
    batch_size_eval: int = 128
    num_workers_eval: int = 4

    d_lex: int = 256
    d_syn: int = 256
    d_sem: int = 256

    threshold_grid_n: int = 199

    bootstrap_n: int = 1000
    bootstrap_seed: int = 123

    noise_trials: int = 20
    noise_std: float = 0.10

    # Baseline training controls
    train_baselines: bool = True
    train_ablations: bool = True

    # Baseline and ablation training hyperparams
    epochs_small: int = 2
    lr_enc: float = 2e-6
    lr_head: float = 1e-5
    weight_decay: float = 0.01
    grad_clip: float = 1.0
    grad_accum: int = 2
    warmup_frac: float = 0.10
    use_amp_bf16: bool = True

    baseline_batch_start: int = 96
    baseline_num_workers: int = 4

    # Latency benchmark
    latency_warmup_batches: int = 3
    latency_measure_batches: int = 20

cfg = CFG()

OUT_DIR = os.path.join(cfg.model_root, cfg.out_dir_name)
os.makedirs(OUT_DIR, exist_ok=True)

PLOTS_DIR = os.path.join(OUT_DIR, "plots")
TABLES_DIR = os.path.join(OUT_DIR, "tables")
CKPT_DIR = os.path.join(OUT_DIR, "checkpoints")
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

print("=" * 60)
print("ACL FINAL EVALUATION SUITE — FIXED")
print("=" * 60)
print("Device:", cfg.device)
print("Output folder:", OUT_DIR)
print("UMAP available:", UMAP_AVAILABLE)

# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)

# ============================================================
# DATA
# ============================================================

def normalize_text(x: str) -> str:
    return " ".join(str(x).split()).strip()


class DualTokPairDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tok_a, tok_b, max_len: int):
        self.df = df.reset_index(drop=True)
        self.ta = tok_a
        self.tb = tok_b
        self.max_len = max_len

        required = {"sentence1", "sentence2", "label"}
        missing = required - set(self.df.columns)
        if missing:
            raise ValueError(f"Missing columns: {missing}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        r = self.df.iloc[idx]
        s1 = normalize_text(r["sentence1"])
        s2 = normalize_text(r["sentence2"])
        ea = self.ta(
            s1, s2,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        eb = self.tb(
            s1, s2,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        item = {
            "input_ids_a": ea["input_ids"].squeeze(0),
            "attention_mask_a": ea["attention_mask"].squeeze(0),
            "input_ids_b": eb["input_ids"].squeeze(0),
            "attention_mask_b": eb["attention_mask"].squeeze(0),
            "labels": torch.tensor(int(r["label"]), dtype=torch.long),
        }
        return item


def collate_stack(batch_list: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
    out = {}
    for k in batch_list[0].keys():
        out[k] = torch.stack([b[k] for b in batch_list], dim=0)
    return out


def make_loader_dualtok(
    df: pd.DataFrame,
    tok_a,
    tok_b,
    batch_size: int,
    shuffle: bool,
    num_workers: int,
) -> DataLoader:
    ds = DualTokPairDataset(df, tok_a, tok_b, cfg.max_len)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=(num_workers > 0),
        collate_fn=collate_stack,
    )


def load_splits(data_root: str) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train_path = os.path.join(data_root, "train.csv")
    val_path = os.path.join(data_root, "val.csv")
    test_path = os.path.join(data_root, "test.csv")
    train_df = pd.read_csv(train_path)
    val_df = pd.read_csv(val_path)
    test_df = pd.read_csv(test_path)
    return train_df, val_df, test_df


print("\n[1/15] Loading data splits...")
train_df, val_df, test_df = load_splits(cfg.data_root)
print("  Loaded:", len(train_df), "train |", len(val_df), "val |", len(test_df), "test")
print("  Label rates:", {
    "train_pos": round(float(train_df["label"].mean()), 4),
    "val_pos": round(float(val_df["label"].mean()), 4),
    "test_pos": round(float(test_df["label"].mean()), 4),
})

print("\n[2/15] Loading tokenizers...")
tok_a = AutoTokenizer.from_pretrained(cfg.enc_a, use_fast=True)
tok_b = AutoTokenizer.from_pretrained(cfg.enc_b, use_fast=True)

val_loader = make_loader_dualtok(val_df, tok_a, tok_b, cfg.batch_size_eval, False, cfg.num_workers_eval)
test_loader = make_loader_dualtok(test_df, tok_a, tok_b, cfg.batch_size_eval, False, cfg.num_workers_eval)

# ============================================================
# MODEL DEFINITIONS
# ============================================================

class AdditiveAttnPool(nn.Module):
    def __init__(self, d_in: int, d_mid: int = 384):
        super().__init__()
        self.fc1 = nn.Linear(d_in, d_mid)
        self.fc2 = nn.Linear(d_mid, 1)

    def forward(self, H: torch.Tensor, mask_keep: torch.Tensor) -> torch.Tensor:
        s = self.fc2(torch.tanh(self.fc1(H))).squeeze(-1)
        s = s.masked_fill(mask_keep == 0, -1e9)
        a = torch.softmax(s, dim=-1).unsqueeze(-1)
        return torch.sum(a * H, dim=1)


class CrossAttentionBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int = 8, dropout: float = 0.0):
        super().__init__()
        self.mha = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.ln = nn.LayerNorm(d_model)

    def forward(
        self,
        q: torch.Tensor,
        k: torch.Tensor,
        v: torch.Tensor,
        key_padding_mask: torch.Tensor,
    ) -> torch.Tensor:
        attn_out, _ = self.mha(q, k, v, key_padding_mask=key_padding_mask, need_weights=False)
        return self.ln(q + attn_out)


class GatedBilinearFuse(nn.Module):
    def __init__(self, d: int):
        super().__init__()
        self.align_b = nn.Linear(d, d)
        self.bilin = nn.Bilinear(d, d, d)
        self.gate = nn.Sequential(nn.Linear(2 * d, d), nn.Sigmoid())
        self.proj = nn.Linear(d, d)

    def forward(self, a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
        b2 = self.align_b(b)
        bil = self.bilin(a, b2)
        g = self.gate(torch.cat([a, b2], dim=-1))
        fused = g * bil + (1.0 - g) * a
        return self.proj(fused)


def orthogonality_penalty(Slex: nn.Linear, Ssyn: nn.Linear, Ssem: nn.Linear) -> torch.Tensor:
    Wl = Slex.weight
    Ws = Ssyn.weight
    Wm = Ssem.weight

    def normed_dot(A, B):
        return torch.sum(A * B) / (torch.norm(A) * torch.norm(B) + 1e-8)

    return (normed_dot(Wl, Ws).abs() + normed_dot(Wl, Wm).abs() + normed_dot(Ws, Wm).abs()) / 3.0


def simplex_penalty(Slex: nn.Linear, Ssyn: nn.Linear, Ssem: nn.Linear, d_in: int) -> torch.Tensor:
    Wl = Slex.weight
    Ws = Ssyn.weight
    Wm = Ssem.weight
    A = Wl.t() @ Wl + Ws.t() @ Ws + Wm.t() @ Wm
    I = torch.eye(d_in, device=A.device, dtype=A.dtype)
    return torch.mean((A - I) ** 2)


class HierDualEncoderDualTok(nn.Module):
    def __init__(self, enc_a_name: str, enc_b_name: str, d_lex: int, d_syn: int, d_sem: int):
        super().__init__()
        self.enc_a = AutoModel.from_pretrained(enc_a_name)
        self.enc_b = AutoModel.from_pretrained(enc_b_name)

        da = int(self.enc_a.config.hidden_size)
        db = int(self.enc_b.config.hidden_size)
        if da != db:
            raise ValueError(f"Hidden sizes must match, got {da} and {db}")

        self.d_in = da

        self.Slex = nn.Linear(self.d_in, d_lex, bias=False)
        self.Ssyn = nn.Linear(self.d_in, d_syn, bias=False)
        self.Ssem = nn.Linear(self.d_in, d_sem, bias=False)

        self.pool_lex = AdditiveAttnPool(d_lex)
        self.pool_syn = AdditiveAttnPool(d_syn)
        self.pool_sem = AdditiveAttnPool(d_sem)

        self.cross12 = CrossAttentionBlock(d_sem, n_heads=8)
        self.cross21 = CrossAttentionBlock(d_sem, n_heads=8)

        self.fuse_lex = GatedBilinearFuse(d_lex)
        self.fuse_syn = GatedBilinearFuse(d_syn)
        self.fuse_sem = GatedBilinearFuse(d_sem)

        self.head_lex = nn.Linear(d_lex, 2)
        self.head_syn = nn.Linear(d_syn, 2)
        self.head_sem = nn.Linear(d_sem, 2)

        meta_in = (2 + 2 + 2) + (d_lex + d_syn + d_sem)
        self.meta = nn.Sequential(
            nn.Linear(meta_in, 512),
            nn.GELU(),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Linear(256, 2),
        )

    def encode(self, encoder: AutoModel, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        out = encoder(input_ids=input_ids, attention_mask=attention_mask)
        return out.last_hidden_state

    def forward(
        self,
        input_ids_a: torch.Tensor,
        attention_mask_a: torch.Tensor,
        input_ids_b: torch.Tensor,
        attention_mask_b: torch.Tensor,
        return_features: bool = False,
    ) -> Dict[str, torch.Tensor]:
        Ha = self.encode(self.enc_a, input_ids_a, attention_mask_a)
        Hb = self.encode(self.enc_b, input_ids_b, attention_mask_b)

        mask_a = attention_mask_a
        mask_b = attention_mask_b
        kpm_a = (mask_a == 0)
        kpm_b = (mask_b == 0)

        Ha_lex = self.Slex(Ha)
        Hb_lex = self.Slex(Hb)
        Ha_syn = self.Ssyn(Ha)
        Hb_syn = self.Ssyn(Hb)
        Ha_sem = self.Ssem(Ha)
        Hb_sem = self.Ssem(Hb)

        fa_lex = self.pool_lex(Ha_lex, mask_a)
        fb_lex = self.pool_lex(Hb_lex, mask_b)
        fa_syn = self.pool_syn(Ha_syn, mask_a)
        fb_syn = self.pool_syn(Hb_syn, mask_b)

        Ha_sem_x = self.cross12(Ha_sem, Hb_sem, Hb_sem, key_padding_mask=kpm_b)
        Hb_sem_x = self.cross21(Hb_sem, Ha_sem, Ha_sem, key_padding_mask=kpm_a)

        fa_sem = self.pool_sem(Ha_sem_x, mask_a)
        fb_sem = self.pool_sem(Hb_sem_x, mask_b)

        z_lex = self.fuse_lex(fa_lex, fb_lex)
        z_syn = self.fuse_syn(fa_syn, fb_syn)
        z_sem = self.fuse_sem(fa_sem, fb_sem)

        log_lex = self.head_lex(z_lex)
        log_syn = self.head_syn(z_syn)
        log_sem = self.head_sem(z_sem)

        meta_in = torch.cat([log_lex, log_syn, log_sem, z_lex, z_syn, z_sem], dim=-1)
        log_meta = self.meta(meta_in)

        out = {"log_meta": log_meta, "log_lex": log_lex, "log_syn": log_syn, "log_sem": log_sem}
        if return_features:
            out.update({"z_lex": z_lex, "z_syn": z_syn, "z_sem": z_sem})
        return out


# Fusion baseline without disentanglement
class DualEncoderFusionNoDisentangle(nn.Module):
    def __init__(self, enc_a_name: str, enc_b_name: str):
        super().__init__()
        self.enc_a = AutoModel.from_pretrained(enc_a_name)
        self.enc_b = AutoModel.from_pretrained(enc_b_name)
        da = int(self.enc_a.config.hidden_size)
        db = int(self.enc_b.config.hidden_size)
        self.da = da
        self.db = db

        self.proj_a = nn.Linear(da, 256)
        self.proj_b = nn.Linear(db, 256)

        self.cls = nn.Sequential(
            nn.Linear(512, 512),
            nn.GELU(),
            nn.Linear(512, 2),
        )

    def forward(self, ids_a, mask_a, ids_b, mask_b):
        Ha = self.enc_a(input_ids=ids_a, attention_mask=mask_a).last_hidden_state
        Hb = self.enc_b(input_ids=ids_b, attention_mask=mask_b).last_hidden_state

        fa = Ha[:, 0, :]
        fb = Hb[:, 0, :]

        za = self.proj_a(fa)
        zb = self.proj_b(fb)

        logits = self.cls(torch.cat([za, zb], dim=-1))
        return {"logits": logits, "za": za, "zb": zb}


# ============================================================
# CHECKPOINT LOADING
# ============================================================

def load_state_strip_compile_prefix(state_dict: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    out = {}
    for k, v in state_dict.items():
        if k.startswith("_orig_mod."):
            out[k.replace("_orig_mod.", "", 1)] = v
        else:
            out[k] = v
    return out


MAIN_CKPT_PATH = os.path.join(cfg.model_root, cfg.main_ckpt_name)
assert os.path.exists(MAIN_CKPT_PATH), f"Missing main checkpoint: {MAIN_CKPT_PATH}"

print("\n[3/15] Loading main hierarchical model checkpoint:", MAIN_CKPT_PATH)
main_model = HierDualEncoderDualTok(cfg.enc_a, cfg.enc_b, cfg.d_lex, cfg.d_syn, cfg.d_sem).to(cfg.device)
state = torch.load(MAIN_CKPT_PATH, map_location=cfg.device)
state = load_state_strip_compile_prefix(state)
main_model.load_state_dict(state, strict=True)
main_model.eval()
print("  Main model loaded successfully.")

# ============================================================
# METRICS AND UTILS
# ============================================================

def probs_from_logits_2(logits_2: np.ndarray) -> np.ndarray:
    t = torch.tensor(logits_2)
    p = torch.softmax(t, dim=1).numpy()[:, 1]
    return p


def threshold_tune_f1(y: np.ndarray, p: np.ndarray, grid_n: int) -> float:
    ts = np.linspace(0.01, 0.99, grid_n)
    best_t = 0.5
    best_f = -1.0
    for t in ts:
        f = f1_score(y, (p >= t).astype(int))
        if f > best_f:
            best_f = f
            best_t = float(t)
    return best_t


def compute_metrics_binary(y: np.ndarray, p: np.ndarray, thr: float) -> Dict[str, float]:
    pred = (p >= thr).astype(int)
    f1 = float(f1_score(y, pred))
    acc = float(accuracy_score(y, pred))
    prec = float(precision_score(y, pred, zero_division=0))
    rec = float(recall_score(y, pred, zero_division=0))
    auc_roc = float(roc_auc_score(y, p))
    auc_pr = float(average_precision_score(y, p))
    mcc = float(matthews_corrcoef(y, pred))
    brier = float(np.mean((p - y) ** 2))
    return {
        "F1": f1,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "AUC_ROC": auc_roc,
        "AUC_PR": auc_pr,
        "MCC": mcc,
        "Brier": brier,
        "Threshold": float(thr),
        "PosRate": float(np.mean(y)),
        "PredPosRate": float(np.mean(pred)),
    }


def expected_calibration_error_numpy(y: np.ndarray, p: np.ndarray, n_bins: int = 15, thr_for_acc: float = 0.5) -> float:
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    N = len(y)
    for i in range(n_bins):
        lo = bins[i]
        hi = bins[i + 1]
        mask = (p >= lo) & (p < hi) if i < n_bins - 1 else (p >= lo) & (p <= hi)
        if np.sum(mask) == 0:
            continue
        acc = np.mean(((p[mask] >= thr_for_acc).astype(int) == y[mask]).astype(float))
        conf = float(np.mean(p[mask]))
        ece += (np.sum(mask) / N) * abs(acc - conf)
    return float(ece)


def save_json(path: str, obj):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


def save_fig(path: str):
    plt.tight_layout()
    plt.savefig(path, dpi=300)
    plt.close()


def print_df(df: pd.DataFrame, title: str):
    print("\n" + title)
    with pd.option_context("display.max_rows", 200, "display.max_columns", 200, "display.width", 180):
        print(df)


# ============================================================
# INFERENCE FOR MAIN MODEL
# ============================================================

@torch.no_grad()
def run_main_inference(model: HierDualEncoderDualTok, loader: DataLoader, name: str) -> Dict[str, np.ndarray]:
    model.eval()
    ys = []
    logits = []
    z_lex_list = []
    z_syn_list = []
    z_sem_list = []

    # Also collect per-head logits for auxiliary analysis
    log_lex_list = []
    log_syn_list = []
    log_sem_list = []

    start = time.time()
    for i, batch in enumerate(loader):
        batch = {k: v.to(cfg.device, non_blocking=True) for k, v in batch.items()}
        use_amp = (cfg.device.startswith("cuda") and cfg.use_amp_bf16)
        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_amp):
            out = model(
                batch["input_ids_a"], batch["attention_mask_a"],
                batch["input_ids_b"], batch["attention_mask_b"],
                return_features=True,
            )
        ys.append(batch["labels"].detach().cpu().numpy())
        logits.append(out["log_meta"].detach().float().cpu().numpy())
        z_lex_list.append(out["z_lex"].detach().float().cpu().numpy())
        z_syn_list.append(out["z_syn"].detach().float().cpu().numpy())
        z_sem_list.append(out["z_sem"].detach().float().cpu().numpy())
        log_lex_list.append(out["log_lex"].detach().float().cpu().numpy())
        log_syn_list.append(out["log_syn"].detach().float().cpu().numpy())
        log_sem_list.append(out["log_sem"].detach().float().cpu().numpy())

        if i == 0:
            print(f"  {name} first batch log_meta shape: {tuple(out['log_meta'].shape)}")
        if (i % 50) == 0:
            elapsed = time.time() - start
            print(f"  {name} batch {i}/{len(loader)} elapsed {elapsed:.1f}s", flush=True)

    y = np.concatenate(ys)
    log = np.concatenate(logits)
    p = probs_from_logits_2(log)
    return {
        "y": y,
        "logits": log,
        "p": p,
        "z_lex": np.concatenate(z_lex_list),
        "z_syn": np.concatenate(z_syn_list),
        "z_sem": np.concatenate(z_sem_list),
        "log_lex": np.concatenate(log_lex_list),
        "log_syn": np.concatenate(log_syn_list),
        "log_sem": np.concatenate(log_sem_list),
    }


print("\n[4/15] Running main inference on VAL")
val_main = run_main_inference(main_model, val_loader, "VAL_MAIN")

print("\n[4/15] Running main inference on TEST")
test_main = run_main_inference(main_model, test_loader, "TEST_MAIN")

thr_main = threshold_tune_f1(val_main["y"], val_main["p"], cfg.threshold_grid_n)
metrics_val_main = compute_metrics_binary(val_main["y"], val_main["p"], thr_main)
metrics_test_main = compute_metrics_binary(test_main["y"], test_main["p"], thr_main)

metrics_val_main["ECE"] = expected_calibration_error_numpy(val_main["y"], val_main["p"], 15, thr_for_acc=thr_main)
metrics_test_main["ECE"] = expected_calibration_error_numpy(test_main["y"], test_main["p"], 15, thr_for_acc=thr_main)

# Auxiliary head metrics
for head_name, logit_key in [("lex", "log_lex"), ("syn", "log_syn"), ("sem", "log_sem")]:
    p_aux_val = probs_from_logits_2(val_main[logit_key])
    p_aux_test = probs_from_logits_2(test_main[logit_key])
    thr_aux = threshold_tune_f1(val_main["y"], p_aux_val, cfg.threshold_grid_n)
    f1_val_aux = float(f1_score(val_main["y"], (p_aux_val >= thr_aux).astype(int)))
    f1_test_aux = float(f1_score(test_main["y"], (p_aux_test >= thr_aux).astype(int)))
    metrics_val_main[f"F1_aux_{head_name}"] = f1_val_aux
    metrics_test_main[f"F1_aux_{head_name}"] = f1_test_aux
    print(f"  Auxiliary head {head_name}: val F1={f1_val_aux:.6f}, test F1={f1_test_aux:.6f}")

main_summary = pd.DataFrame([
    {"Model": "Main_Hier", "Split": "val", **metrics_val_main},
    {"Model": "Main_Hier", "Split": "test", **metrics_test_main},
])

print_df(main_summary, "MAIN MODEL METRICS")
main_summary.to_csv(os.path.join(TABLES_DIR, "main_metrics.csv"), index=False)

# ============================================================
# PLOTS FOR MAIN MODEL
# ============================================================

def plot_roc_pr(y: np.ndarray, p: np.ndarray, prefix: str):
    fpr, tpr, _ = roc_curve(y, p)
    auc_val = roc_auc_score(y, p)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"AUC={auc_val:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{prefix} ROC Curve")
    plt.legend()
    save_fig(os.path.join(PLOTS_DIR, f"{prefix.lower()}_roc.png"))

    prec_arr, rec_arr, _ = precision_recall_curve(y, p)
    ap_val = average_precision_score(y, p)
    plt.figure(figsize=(6, 5))
    plt.plot(rec_arr, prec_arr, label=f"AP={ap_val:.4f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"{prefix} Precision-Recall Curve")
    plt.legend()
    save_fig(os.path.join(PLOTS_DIR, f"{prefix.lower()}_pr.png"))


def plot_confusion(y: np.ndarray, p: np.ndarray, thr: float, prefix: str):
    pred = (p >= thr).astype(int)
    cm = confusion_matrix(y, pred)
    plt.figure(figsize=(5, 4))
    plt.imshow(cm, cmap="Blues")
    plt.xticks([0, 1], ["Non-Para", "Para"])
    plt.yticks([0, 1], ["Non-Para", "Para"])
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"{prefix} Confusion Matrix")
    for i in range(2):
        for j in range(2):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=14)
    plt.colorbar()
    save_fig(os.path.join(PLOTS_DIR, f"{prefix.lower()}_confusion.png"))


def plot_calibration(y: np.ndarray, p: np.ndarray, prefix: str):
    frac_pos, mean_pred = calibration_curve(y, p, n_bins=15)
    plt.figure(figsize=(6, 5))
    plt.plot(mean_pred, frac_pos, marker="o", label="Model")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect")
    plt.xlabel("Mean Predicted Probability")
    plt.ylabel("Fraction Positive")
    plt.title(f"{prefix} Calibration")
    plt.legend()
    save_fig(os.path.join(PLOTS_DIR, f"{prefix.lower()}_calibration.png"))


def plot_score_distributions(y: np.ndarray, p: np.ndarray, thr: float, prefix: str):
    plt.figure(figsize=(8, 5))
    plt.hist(p[y == 0], bins=100, alpha=0.6, label="Non-Paraphrase", color="blue")
    plt.hist(p[y == 1], bins=100, alpha=0.6, label="Paraphrase", color="orange")
    plt.axvline(x=thr, color="red", linestyle="--", label=f"Threshold={thr:.3f}")
    plt.xlabel("Predicted Probability")
    plt.ylabel("Count")
    plt.title(f"{prefix} Score Distributions by Class")
    plt.legend()
    save_fig(os.path.join(PLOTS_DIR, f"{prefix.lower()}_score_dist.png"))


def plot_pca(z: np.ndarray, y: np.ndarray, prefix: str, label: str = "z_sem"):
    pca = PCA(n_components=2)
    emb = pca.fit_transform(z)
    plt.figure(figsize=(6, 5))
    plt.scatter(emb[y == 0, 0], emb[y == 0, 1], s=2, label="Non-Para", alpha=0.3)
    plt.scatter(emb[y == 1, 0], emb[y == 1, 1], s=2, label="Para", alpha=0.3)
    plt.legend()
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.title(f"{prefix} PCA ({label})")
    save_fig(os.path.join(PLOTS_DIR, f"{prefix.lower()}_pca_{label}.png"))


def plot_umap(z: np.ndarray, y: np.ndarray, prefix: str, label: str = "z_sem", max_points: int = 40000):
    if not UMAP_AVAILABLE:
        print(f"  UMAP not available, skipping {prefix} {label}")
        return
    n = min(len(z), max_points)
    reducer = umap.UMAP(n_neighbors=30, min_dist=0.1, metric="cosine")
    emb = reducer.fit_transform(z[:n])
    yy = y[:n]
    plt.figure(figsize=(6, 5))
    plt.scatter(emb[yy == 0, 0], emb[yy == 0, 1], s=2, label="Non-Para", alpha=0.3)
    plt.scatter(emb[yy == 1, 0], emb[yy == 1, 1], s=2, label="Para", alpha=0.3)
    plt.legend()
    plt.xlabel("UMAP1")
    plt.ylabel("UMAP2")
    plt.title(f"{prefix} UMAP ({label})")
    save_fig(os.path.join(PLOTS_DIR, f"{prefix.lower()}_umap_{label}.png"))


print("\n[5/15] Saving main model plots...")
for split_name, data_dict, thr_val in [("VAL_MAIN", val_main, thr_main), ("TEST_MAIN", test_main, thr_main)]:
    plot_roc_pr(data_dict["y"], data_dict["p"], split_name)
    plot_confusion(data_dict["y"], data_dict["p"], thr_val, split_name)
    plot_calibration(data_dict["y"], data_dict["p"], split_name)
    plot_score_distributions(data_dict["y"], data_dict["p"], thr_val, split_name)
    # PCA for all three subspaces
    plot_pca(data_dict["z_lex"], data_dict["y"], split_name, "z_lex")
    plot_pca(data_dict["z_syn"], data_dict["y"], split_name, "z_syn")
    plot_pca(data_dict["z_sem"], data_dict["y"], split_name, "z_sem")
    # UMAP for all three subspaces
    plot_umap(data_dict["z_lex"], data_dict["y"], split_name, "z_lex")
    plot_umap(data_dict["z_syn"], data_dict["y"], split_name, "z_syn")
    plot_umap(data_dict["z_sem"], data_dict["y"], split_name, "z_sem")

# Cosine heatmap small subset
print("  Saving cosine similarity heatmap on small subset")
sub_n = min(2000, len(val_main["z_sem"]))
sim = cosine_similarity(val_main["z_sem"][:sub_n])
plt.figure(figsize=(6, 5))
plt.imshow(sim, cmap="viridis")
plt.title("VAL Cosine Similarity Heatmap (z_sem subset)")
plt.xlabel("Sample Index")
plt.ylabel("Sample Index")
plt.colorbar()
save_fig(os.path.join(PLOTS_DIR, "val_cosine_heatmap_zsem.png"))


# ============================================================
# CKA: CENTERED KERNEL ALIGNMENT BETWEEN SUBSPACES
# Measures representational similarity between z_lex, z_syn, z_sem
# Reference: Kornblith et al., 2019, ICML
# ============================================================

def linear_CKA(X: np.ndarray, Y: np.ndarray) -> float:
    """Compute Linear Centered Kernel Alignment between two representations."""
    n = X.shape[0]
    # Center
    X = X - X.mean(axis=0, keepdims=True)
    Y = Y - Y.mean(axis=0, keepdims=True)
    # HSIC
    XtX = X.T @ X
    YtY = Y.T @ Y
    XtY = X.T @ Y
    hsic_xy = np.trace(XtX @ YtY) / ((n - 1) ** 2)  # simplified
    hsic_xx = np.trace(XtX @ XtX) / ((n - 1) ** 2)
    hsic_yy = np.trace(YtY @ YtY) / ((n - 1) ** 2)
    cka = hsic_xy / (np.sqrt(hsic_xx * hsic_yy) + 1e-10)
    return float(cka)


print("\n[6/15] Computing CKA between subspaces (disentanglement verification)...")
# Use a subset for speed
cka_n = min(10000, len(val_main["z_lex"]))
cka_results = {}
for name_a, name_b in [("z_lex", "z_syn"), ("z_lex", "z_sem"), ("z_syn", "z_sem")]:
    cka_val = linear_CKA(val_main[name_a][:cka_n], val_main[name_b][:cka_n])
    cka_results[f"CKA({name_a},{name_b})"] = cka_val
    print(f"  CKA({name_a}, {name_b}) = {cka_val:.6f}")

# Also compute orthogonality of projection weights
with torch.no_grad():
    ort_val = orthogonality_penalty(main_model.Slex, main_model.Ssyn, main_model.Ssem).item()
    simp_val = simplex_penalty(main_model.Slex, main_model.Ssyn, main_model.Ssem, main_model.d_in).item()
    cka_results["orthogonality_penalty"] = ort_val
    cka_results["simplex_penalty"] = simp_val
    print(f"  Orthogonality penalty: {ort_val:.6f}")
    print(f"  Simplex penalty: {simp_val:.6f}")

# Plot CKA matrix
cka_matrix = np.array([
    [1.0, cka_results["CKA(z_lex,z_syn)"], cka_results["CKA(z_lex,z_sem)"]],
    [cka_results["CKA(z_lex,z_syn)"], 1.0, cka_results["CKA(z_syn,z_sem)"]],
    [cka_results["CKA(z_lex,z_sem)"], cka_results["CKA(z_syn,z_sem)"], 1.0],
])
plt.figure(figsize=(5, 4))
plt.imshow(cka_matrix, cmap="YlOrRd", vmin=0, vmax=1)
labels = ["Lexical", "Syntactic", "Semantic"]
plt.xticks(range(3), labels)
plt.yticks(range(3), labels)
for i in range(3):
    for j in range(3):
        plt.text(j, i, f"{cka_matrix[i, j]:.3f}", ha="center", va="center", fontsize=12)
plt.title("Linear CKA Between Subspaces")
plt.colorbar()
save_fig(os.path.join(PLOTS_DIR, "cka_subspace_matrix.png"))

save_json(os.path.join(TABLES_DIR, "cka_disentanglement.json"), cka_results)


# ============================================================
# STRUCTURAL PROBING: Lexical overlap correlates with z_lex
# Reference: Hewitt & Manning (2019), NAACL
# ============================================================

print("\n[7/15] Structural probing: correlating subspaces with linguistic properties...")

probe_n = min(20000, len(val_df))
probe_df = val_df.iloc[:probe_n].copy()

# 1. Lexical overlap: Jaccard similarity of tokens
def jaccard_tokens(s1: str, s2: str) -> float:
    t1 = set(str(s1).lower().split())
    t2 = set(str(s2).lower().split())
    if len(t1 | t2) == 0:
        return 0.0
    return len(t1 & t2) / len(t1 | t2)

lexical_overlaps = np.array([
    jaccard_tokens(probe_df.iloc[i]["sentence1"], probe_df.iloc[i]["sentence2"])
    for i in range(probe_n)
])

# 2. Length ratio as a proxy for syntactic similarity
def length_ratio(s1: str, s2: str) -> float:
    l1 = len(str(s1).split())
    l2 = len(str(s2).split())
    if max(l1, l2) == 0:
        return 1.0
    return min(l1, l2) / max(l1, l2)

length_ratios = np.array([
    length_ratio(probe_df.iloc[i]["sentence1"], probe_df.iloc[i]["sentence2"])
    for i in range(probe_n)
])

# Cosine similarity of subspace vectors
from numpy.linalg import norm as np_norm

def batch_cosine_sim(X: np.ndarray) -> np.ndarray:
    """For paired features where each row is a fused pair representation,
    we use the norm as a proxy for within-pair agreement."""
    return np_norm(X, axis=1)

# Correlation of each subspace's magnitude with lexical overlap and length ratio
from scipy.stats import spearmanr

probe_results = {}
for sub_name in ["z_lex", "z_syn", "z_sem"]:
    z = val_main[sub_name][:probe_n]
    z_norms = np_norm(z, axis=1)

    corr_lex, p_lex = spearmanr(z_norms, lexical_overlaps)
    corr_len, p_len = spearmanr(z_norms, length_ratios)

    probe_results[f"{sub_name}_vs_lexical_overlap_spearman"] = float(corr_lex)
    probe_results[f"{sub_name}_vs_lexical_overlap_pval"] = float(p_lex)
    probe_results[f"{sub_name}_vs_length_ratio_spearman"] = float(corr_len)
    probe_results[f"{sub_name}_vs_length_ratio_pval"] = float(p_len)

    print(f"  {sub_name} vs lexical_overlap: rho={corr_lex:.4f} (p={p_lex:.2e})")
    print(f"  {sub_name} vs length_ratio:    rho={corr_len:.4f} (p={p_len:.2e})")

# Linear probe: train small logistic regression from each subspace to predict label
from sklearn.linear_model import LogisticRegression

probe_labels = val_main["y"][:probe_n]
for sub_name in ["z_lex", "z_syn", "z_sem"]:
    z = val_main[sub_name][:probe_n]
    # 80/20 split within probe set
    split_idx = int(0.8 * probe_n)
    lr = LogisticRegression(max_iter=500, C=1.0, solver="lbfgs")
    lr.fit(z[:split_idx], probe_labels[:split_idx])
    probe_acc = lr.score(z[split_idx:], probe_labels[split_idx:])
    probe_f1 = f1_score(probe_labels[split_idx:], lr.predict(z[split_idx:]))
    probe_results[f"{sub_name}_probe_acc"] = float(probe_acc)
    probe_results[f"{sub_name}_probe_f1"] = float(probe_f1)
    print(f"  {sub_name} linear probe: acc={probe_acc:.4f}, F1={probe_f1:.4f}")

save_json(os.path.join(TABLES_DIR, "structural_probing.json"), probe_results)


# ============================================================
# MAJORITY AND RANDOM BASELINES (NO TRAINING)
# ============================================================

print("\n[8/15] Computing simple baselines (majority, random)...")

def majority_baseline_metrics(df: pd.DataFrame) -> Dict[str, float]:
    y = df["label"].to_numpy().astype(int)
    maj = int(np.round(np.mean(y)))
    pred = np.full_like(y, maj)
    p = pred.astype(float)
    return {
        "F1": float(f1_score(y, pred)),
        "Accuracy": float(accuracy_score(y, pred)),
        "Precision": float(precision_score(y, pred, zero_division=0)),
        "Recall": float(recall_score(y, pred, zero_division=0)),
        "AUC_ROC": float(roc_auc_score(y, p)),
        "AUC_PR": float(average_precision_score(y, p)),
        "MCC": float(matthews_corrcoef(y, pred)),
        "Brier": float(np.mean((p - y) ** 2)),
        "Threshold": 0.5,
        "PosRate": float(np.mean(y)),
        "PredPosRate": float(np.mean(pred)),
        "ECE": expected_calibration_error_numpy(y, p, 15, thr_for_acc=0.5),
    }

def random_baseline_metrics(df: pd.DataFrame, seed: int = 0) -> Dict[str, float]:
    rng = np.random.RandomState(seed)
    y = df["label"].to_numpy().astype(int)
    p = rng.rand(len(y)).astype(float)
    pred = (p >= 0.5).astype(int)
    return {
        "F1": float(f1_score(y, pred)),
        "Accuracy": float(accuracy_score(y, pred)),
        "Precision": float(precision_score(y, pred, zero_division=0)),
        "Recall": float(recall_score(y, pred, zero_division=0)),
        "AUC_ROC": float(roc_auc_score(y, p)),
        "AUC_PR": float(average_precision_score(y, p)),
        "MCC": float(matthews_corrcoef(y, pred)),
        "Brier": float(np.mean((p - y) ** 2)),
        "Threshold": 0.5,
        "PosRate": float(np.mean(y)),
        "PredPosRate": float(np.mean(pred)),
        "ECE": expected_calibration_error_numpy(y, p, 15, thr_for_acc=0.5),
    }

maj_val = majority_baseline_metrics(val_df)
maj_test = majority_baseline_metrics(test_df)
rnd_val = random_baseline_metrics(val_df, seed=1)
rnd_test = random_baseline_metrics(test_df, seed=1)

baseline_simple = pd.DataFrame([
    {"Model": "Majority", "Split": "val", **maj_val},
    {"Model": "Majority", "Split": "test", **maj_test},
    {"Model": "Random", "Split": "val", **rnd_val},
    {"Model": "Random", "Split": "test", **rnd_test},
])
print_df(baseline_simple, "SIMPLE BASELINES")
baseline_simple.to_csv(os.path.join(TABLES_DIR, "baselines_simple.csv"), index=False)


# ============================================================
# TRAINING HELPERS FOR TRAINED BASELINES AND ABLATIONS
# ============================================================

def build_optimizer_groups(model: nn.Module, lr_enc: float, lr_head: float, weight_decay: float):
    enc_params = []
    head_params = []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if any(n.startswith(pfx) for pfx in ["enc_a.", "enc_b.", "roberta.", "deberta."]):
            enc_params.append(p)
        else:
            head_params.append(p)
    groups = []
    if enc_params:
        groups.append({"params": enc_params, "lr": lr_enc})
    if head_params:
        groups.append({"params": head_params, "lr": lr_head})
    return torch.optim.AdamW(groups, weight_decay=weight_decay)


def model_forward_logits(model: nn.Module, batch: Dict[str, torch.Tensor], kind: str) -> torch.Tensor:
    if kind == "fusion_no_disentangle":
        out = model(batch["input_ids_a"], batch["attention_mask_a"],
                     batch["input_ids_b"], batch["attention_mask_b"])
        return out["logits"]
    if kind == "main_hier":
        out = model(batch["input_ids_a"], batch["attention_mask_a"],
                     batch["input_ids_b"], batch["attention_mask_b"])
        return out["log_meta"]
    if kind == "single_a":
        out = model(input_ids=batch["input_ids_a"], attention_mask=batch["attention_mask_a"])
        return out.logits
    if kind == "single_b":
        out = model(input_ids=batch["input_ids_b"], attention_mask=batch["attention_mask_b"])
        return out.logits
    raise ValueError(f"Unknown kind: {kind}")


@torch.no_grad()
def eval_logits_and_probs_binary(
    model: nn.Module,
    loader: DataLoader,
    device: str,
    kind: str,
) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    ys = []
    logits = []
    use_amp = (device.startswith("cuda") and cfg.use_amp_bf16)
    for i, batch in enumerate(loader):
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_amp):
            out = model_forward_logits(model, batch, kind)
        ys.append(batch["labels"].detach().cpu().numpy())
        logits.append(out.detach().float().cpu().numpy())
        if (i % 100) == 0:
            print(f"    Eval {kind} batch {i}/{len(loader)}", flush=True)
    y = np.concatenate(ys)
    log = np.concatenate(logits)
    p = probs_from_logits_2(log)
    return y, p


def train_classifier(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: str,
    kind: str,
    out_ckpt_path: str,
    epochs: int,
    lr_enc: float,
    lr_head: float,
    weight_decay: float,
    grad_clip: float,
    grad_accum: int,
    warmup_frac: float,
) -> Dict[str, float]:
    model.to(device)

    opt = build_optimizer_groups(model, lr_enc, lr_head, weight_decay)

    total_steps = int(math.ceil(len(train_loader) / max(1, grad_accum)) * epochs)
    warmup_steps = int(warmup_frac * total_steps)
    sched = get_cosine_schedule_with_warmup(opt, warmup_steps, total_steps)

    best_f1 = -1.0
    best_thr = 0.5
    best_state = None

    use_amp = (device.startswith("cuda") and cfg.use_amp_bf16)

    print(f"\n  Training {kind}")
    print(f"  Total steps: {total_steps}, Warmup: {warmup_steps}")

    for ep in range(epochs):
        model.train()
        opt.zero_grad(set_to_none=True)

        pbar = tqdm(train_loader, desc=f"  {kind} epoch {ep+1}/{epochs}", leave=True)
        running_loss = 0.0
        denom = 0

        for step, batch in enumerate(pbar):
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            y = batch["labels"]

            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_amp):
                logits2 = model_forward_logits(model, batch, kind)
                loss = F.cross_entropy(logits2, y)
                loss = loss / float(grad_accum)

            loss.backward()

            if ((step + 1) % grad_accum) == 0:
                if grad_clip and grad_clip > 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                opt.step()
                sched.step()
                opt.zero_grad(set_to_none=True)

            running_loss += float(loss.detach().cpu().item()) * float(grad_accum)
            denom += 1

            if (step % 100) == 0:
                pbar.set_postfix(loss=running_loss / max(1, denom), lr=float(sched.get_last_lr()[0]))

        yv, pv = eval_logits_and_probs_binary(model, val_loader, device, kind)
        thr = threshold_tune_f1(yv, pv, cfg.threshold_grid_n)
        met = compute_metrics_binary(yv, pv, thr)
        print(f"\n  {kind} epoch {ep+1} val F1 {met['F1']:.6f} thr {thr:.4f}")

        if met["F1"] > best_f1:
            best_f1 = met["F1"]
            best_thr = thr
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}

    if best_state is not None:
        torch.save({"state_dict": best_state, "best_thr": best_thr}, out_ckpt_path)
        print(f"  Saved best checkpoint: {out_ckpt_path}")

    return {"best_f1_val": float(best_f1), "best_thr": float(best_thr)}


# ============================================================
# FIX: evaluate_saved_checkpoint now handles ALL model kinds
# ============================================================

def _build_model_for_kind(kind: str) -> nn.Module:
    """Instantiate the correct model architecture for a given kind."""
    if kind == "single_a":
        return AutoModelForSequenceClassification.from_pretrained(cfg.enc_a, num_labels=2)
    elif kind == "single_b":
        return AutoModelForSequenceClassification.from_pretrained(cfg.enc_b, num_labels=2)
    elif kind == "fusion_no_disentangle":
        return DualEncoderFusionNoDisentangle(cfg.enc_a, cfg.enc_b)
    elif kind == "main_hier":
        return HierDualEncoderDualTok(cfg.enc_a, cfg.enc_b, cfg.d_lex, cfg.d_syn, cfg.d_sem)
    else:
        raise ValueError(f"Unknown model kind: {kind}")


def evaluate_saved_checkpoint(kind: str, ckpt_path: str, thr: float) -> pd.DataFrame:
    """Load a saved checkpoint and evaluate on val and test sets.

    FIX: Now properly handles 'main_hier' kind by instantiating
    HierDualEncoderDualTok and loading the state dict.
    """
    print(f"  Evaluating checkpoint: {ckpt_path} (kind={kind})")
    model = _build_model_for_kind(kind)

    obj = torch.load(ckpt_path, map_location="cpu")
    sd = obj["state_dict"] if "state_dict" in obj else obj
    sd = load_state_strip_compile_prefix(sd)
    model.load_state_dict(sd, strict=True)
    model.to(cfg.device)
    model.eval()

    yv, pv = eval_logits_and_probs_binary(model, val_loader, cfg.device, kind)
    yt, pt = eval_logits_and_probs_binary(model, test_loader, cfg.device, kind)

    thr_use = thr
    if thr_use is None:
        thr_use = threshold_tune_f1(yv, pv, cfg.threshold_grid_n)

    mv = compute_metrics_binary(yv, pv, float(thr_use))
    mt = compute_metrics_binary(yt, pt, float(thr_use))
    mv["ECE"] = expected_calibration_error_numpy(yv, pv, 15, thr_for_acc=float(thr_use))
    mt["ECE"] = expected_calibration_error_numpy(yt, pt, 15, thr_for_acc=float(thr_use))

    df = pd.DataFrame([
        {"Model": kind, "Split": "val", **mv},
        {"Model": kind, "Split": "test", **mt},
    ])

    # Clean up GPU memory
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return df


# ============================================================
# TRAINED BASELINES
# 1) single encoder DeBERTa
# 2) single encoder RoBERTa
# 3) dual encoder fusion without disentanglement
# ============================================================

print("\n[9/15] Training/loading baselines...")

def load_or_train_single(kind: str, model_name: str, ckpt_name: str):
    ckpt_path = os.path.join(CKPT_DIR, ckpt_name)
    if os.path.exists(ckpt_path):
        print(f"  Loading baseline checkpoint: {ckpt_path}")
        obj = torch.load(ckpt_path, map_location="cpu")
        return ckpt_path, obj["best_thr"]

    if not cfg.train_baselines:
        raise RuntimeError(f"Baseline checkpoint missing and train_baselines is False: {ckpt_path}")

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    bs = cfg.baseline_batch_start

    train_loader_local = make_loader_dualtok(train_df, tok_a, tok_b, bs, True, cfg.baseline_num_workers)
    val_loader_local = make_loader_dualtok(val_df, tok_a, tok_b, bs, False, cfg.baseline_num_workers)

    stats = train_classifier(
        model=model,
        train_loader=train_loader_local,
        val_loader=val_loader_local,
        device=cfg.device,
        kind=kind,
        out_ckpt_path=ckpt_path,
        epochs=cfg.epochs_small,
        lr_enc=cfg.lr_enc,
        lr_head=cfg.lr_head,
        weight_decay=cfg.weight_decay,
        grad_clip=cfg.grad_clip,
        grad_accum=cfg.grad_accum,
        warmup_frac=cfg.warmup_frac,
    )
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return ckpt_path, stats["best_thr"]


def load_or_train_fusion_no_disentangle():
    ckpt_path = os.path.join(CKPT_DIR, "fusion_no_disentangle.pt")
    if os.path.exists(ckpt_path):
        print(f"  Loading fusion baseline checkpoint: {ckpt_path}")
        obj = torch.load(ckpt_path, map_location="cpu")
        return ckpt_path, obj["best_thr"]

    if not cfg.train_baselines:
        raise RuntimeError(f"Fusion baseline checkpoint missing and train_baselines is False")

    model = DualEncoderFusionNoDisentangle(cfg.enc_a, cfg.enc_b)
    bs = max(32, cfg.baseline_batch_start // 2)

    train_loader_local = make_loader_dualtok(train_df, tok_a, tok_b, bs, True, cfg.baseline_num_workers)
    val_loader_local = make_loader_dualtok(val_df, tok_a, tok_b, bs, False, cfg.baseline_num_workers)

    stats = train_classifier(
        model=model,
        train_loader=train_loader_local,
        val_loader=val_loader_local,
        device=cfg.device,
        kind="fusion_no_disentangle",
        out_ckpt_path=ckpt_path,
        epochs=cfg.epochs_small,
        lr_enc=cfg.lr_enc,
        lr_head=cfg.lr_head,
        weight_decay=cfg.weight_decay,
        grad_clip=cfg.grad_clip,
        grad_accum=cfg.grad_accum,
        warmup_frac=cfg.warmup_frac,
    )
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return ckpt_path, stats["best_thr"]


ckpt_deb, thr_deb = load_or_train_single("single_a", cfg.enc_a, "baseline_single_a.pt")
df_deb = evaluate_saved_checkpoint("single_a", ckpt_deb, thr_deb)
print_df(df_deb, "BASELINE: DeBERTa-v3 (single_a)")

ckpt_rob, thr_rob = load_or_train_single("single_b", cfg.enc_b, "baseline_single_b.pt")
df_rob = evaluate_saved_checkpoint("single_b", ckpt_rob, thr_rob)
print_df(df_rob, "BASELINE: RoBERTa (single_b)")

ckpt_fus, thr_fus = load_or_train_fusion_no_disentangle()
df_fus = evaluate_saved_checkpoint("fusion_no_disentangle", ckpt_fus, thr_fus)
print_df(df_fus, "BASELINE: Dual Fusion (no disentangle)")

baselines_trained = pd.concat([df_deb, df_rob, df_fus], ignore_index=True)
baselines_trained.to_csv(os.path.join(TABLES_DIR, "baselines_trained.csv"), index=False)


# ============================================================
# LOSS WEIGHT SENSITIVITY ABLATION
# Retrains only the heads and meta on top of frozen encoders
# ============================================================

class MainHeadOnlyWrapper(nn.Module):
    def __init__(self, base: HierDualEncoderDualTok, main_w: float, aux_w: float, ortho_w: float, simplex_w: float):
        super().__init__()
        self.base = base
        self.main_w = float(main_w)
        self.aux_w = float(aux_w)
        self.ortho_w = float(ortho_w)
        self.simplex_w = float(simplex_w)

    def forward(self, ids_a, mask_a, ids_b, mask_b, labels=None):
        out = self.base(ids_a, mask_a, ids_b, mask_b, return_features=False)
        if labels is None:
            return out

        loss_main = F.cross_entropy(out["log_meta"], labels)
        loss_lex = F.cross_entropy(out["log_lex"], labels)
        loss_syn = F.cross_entropy(out["log_syn"], labels)
        loss_sem = F.cross_entropy(out["log_sem"], labels)
        loss_aux = (loss_lex + loss_syn + loss_sem) / 3.0

        ort = orthogonality_penalty(self.base.Slex, self.base.Ssyn, self.base.Ssem)
        simp = simplex_penalty(self.base.Slex, self.base.Ssyn, self.base.Ssem, self.base.d_in)

        loss = self.main_w * loss_main + self.aux_w * loss_aux + self.ortho_w * ort + self.simplex_w * simp
        return out, loss


def freeze_encoders_only(model: HierDualEncoderDualTok):
    for n, p in model.named_parameters():
        if n.startswith("enc_a.") or n.startswith("enc_b."):
            p.requires_grad = False
        else:
            p.requires_grad = True


def train_ablation_weights(
    tag: str,
    main_w: float,
    aux_w: float,
    ortho_w: float,
    simplex_w: float,
    epochs: int = 1,
):
    ckpt_path = os.path.join(CKPT_DIR, f"ablation_weights_{tag}.pt")
    if os.path.exists(ckpt_path):
        print(f"  Loading existing weights ablation: {ckpt_path}")
        obj = torch.load(ckpt_path, map_location="cpu")
        return ckpt_path, obj["best_thr"]

    if not cfg.train_ablations:
        raise RuntimeError(f"Ablation checkpoint missing and train_ablations is False: {ckpt_path}")

    base = HierDualEncoderDualTok(cfg.enc_a, cfg.enc_b, cfg.d_lex, cfg.d_syn, cfg.d_sem)
    ckpt_state = torch.load(MAIN_CKPT_PATH, map_location="cpu")
    base.load_state_dict(load_state_strip_compile_prefix(ckpt_state), strict=True)

    freeze_encoders_only(base)
    base.to(cfg.device)

    wrap = MainHeadOnlyWrapper(base, main_w, aux_w, ortho_w, simplex_w).to(cfg.device)

    bs = max(32, cfg.baseline_batch_start // 2)
    train_loader_local = make_loader_dualtok(train_df, tok_a, tok_b, bs, True, cfg.baseline_num_workers)
    val_loader_local = make_loader_dualtok(val_df, tok_a, tok_b, bs, False, cfg.baseline_num_workers)

    opt = torch.optim.AdamW(
        [p for p in wrap.parameters() if p.requires_grad],
        lr=cfg.lr_head,
        weight_decay=cfg.weight_decay,
    )

    total_steps = int(math.ceil(len(train_loader_local) / max(1, cfg.grad_accum)) * epochs)
    warmup_steps = int(cfg.warmup_frac * total_steps)
    sched = get_cosine_schedule_with_warmup(opt, warmup_steps, total_steps)

    best_f1 = -1.0
    best_thr = 0.5
    best_state = None

    use_amp = (cfg.device.startswith("cuda") and cfg.use_amp_bf16)

    print(f"\n  Ablation {tag} (main={main_w}, aux={aux_w}, ortho={ortho_w}, simplex={simplex_w})")

    for ep in range(epochs):
        wrap.train()
        opt.zero_grad(set_to_none=True)
        pbar = tqdm(train_loader_local, desc=f"  ablation {tag} epoch {ep+1}/{epochs}", leave=True)

        for step, batch in enumerate(pbar):
            batch = {k: v.to(cfg.device, non_blocking=True) for k, v in batch.items()}
            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_amp):
                _, loss = wrap(
                    batch["input_ids_a"], batch["attention_mask_a"],
                    batch["input_ids_b"], batch["attention_mask_b"],
                    labels=batch["labels"],
                )
                loss = loss / float(cfg.grad_accum)

            loss.backward()

            if ((step + 1) % cfg.grad_accum) == 0:
                if cfg.grad_clip and cfg.grad_clip > 0:
                    torch.nn.utils.clip_grad_norm_(wrap.parameters(), cfg.grad_clip)
                opt.step()
                sched.step()
                opt.zero_grad(set_to_none=True)

            if (step % 100) == 0:
                pbar.set_postfix(loss=float(loss.detach().cpu().item()) * float(cfg.grad_accum),
                                 lr=float(sched.get_last_lr()[0]))

        # Evaluate
        wrap.eval()
        yv, pv = eval_logits_and_probs_binary(wrap.base, val_loader_local, cfg.device, "main_hier")
        thr = threshold_tune_f1(yv, pv, cfg.threshold_grid_n)
        met = compute_metrics_binary(yv, pv, thr)
        print(f"  ablation {tag} val F1 {met['F1']:.6f} thr {thr:.4f}")

        if met["F1"] > best_f1:
            best_f1 = met["F1"]
            best_thr = float(thr)
            best_state = {k: v.detach().cpu() for k, v in wrap.base.state_dict().items()}

    torch.save({
        "state_dict": best_state,
        "best_thr": best_thr,
        "tag": tag,
        "main_w": main_w, "aux_w": aux_w,
        "ortho_w": ortho_w, "simplex_w": simplex_w,
    }, ckpt_path)
    print(f"  Saved weights ablation: {ckpt_path}")

    # Cleanup
    del wrap, base
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return ckpt_path, best_thr


print("\n[10/15] Loss weight sensitivity ablations...")

weight_grid = [
    ("w060_040", 0.60, 0.40, 0.02, 0.01),
    ("w080_020", 0.80, 0.20, 0.02, 0.01),
    ("w050_050", 0.50, 0.50, 0.02, 0.01),
    ("w070_030", 0.70, 0.30, 0.02, 0.01),
    ("no_ortho", 0.60, 0.40, 0.00, 0.01),
    ("no_simplex", 0.60, 0.40, 0.02, 0.00),
    ("no_aux", 1.00, 0.00, 0.02, 0.01),
]

ablation_rows = []
for tag, mw, aw, ow, sw in weight_grid:
    print(f"\n  --- Ablation: {tag} ---")
    ckpt_path, thr = train_ablation_weights(tag, mw, aw, ow, sw, epochs=1)
    # FIX: Now uses "main_hier" which is properly handled
    df_ab = evaluate_saved_checkpoint("main_hier", ckpt_path, thr)
    df_ab["Model"] = "Ablation_" + tag
    ablation_rows.append(df_ab)
    print_df(df_ab, f"  ABLATION: {tag}")

ablations_df = pd.concat(ablation_rows, ignore_index=True)
print_df(ablations_df, "ALL WEIGHT AND REGULARIZER ABLATIONS")
ablations_df.to_csv(os.path.join(TABLES_DIR, "ablations_weights_regularizers.csv"), index=False)


# ============================================================
# CALIBRATION: TEMPERATURE SCALING ON VAL, APPLY TO TEST
# ============================================================

def fit_temperature(val_logits: np.ndarray, val_y: np.ndarray) -> float:
    logits = torch.tensor(val_logits, dtype=torch.float32)
    y = torch.tensor(val_y, dtype=torch.long)

    T = torch.tensor([1.0], dtype=torch.float32, requires_grad=True)
    opt = torch.optim.LBFGS([T], lr=0.1, max_iter=200)

    def closure():
        opt.zero_grad()
        scaled = logits / torch.clamp(T, min=1e-4)
        loss = F.cross_entropy(scaled, y)
        loss.backward()
        return loss

    opt.step(closure)
    return float(T.detach().cpu().item())


print("\n[11/15] Temperature scaling calibration...")
T_opt = fit_temperature(val_main["logits"], val_main["y"])
print(f"  Optimal temperature: {T_opt:.4f}")


def apply_temperature(logits_2: np.ndarray, T: float) -> np.ndarray:
    t = torch.tensor(logits_2, dtype=torch.float32)
    p = torch.softmax(t / float(T), dim=1).numpy()[:, 1]
    return p


val_p_cal = apply_temperature(val_main["logits"], T_opt)
test_p_cal = apply_temperature(test_main["logits"], T_opt)

thr_cal = threshold_tune_f1(val_main["y"], val_p_cal, cfg.threshold_grid_n)

met_val_cal = compute_metrics_binary(val_main["y"], val_p_cal, thr_cal)
met_test_cal = compute_metrics_binary(test_main["y"], test_p_cal, thr_cal)
met_val_cal["ECE"] = expected_calibration_error_numpy(val_main["y"], val_p_cal, 15, thr_for_acc=thr_cal)
met_test_cal["ECE"] = expected_calibration_error_numpy(test_main["y"], test_p_cal, 15, thr_for_acc=thr_cal)

cal_df = pd.DataFrame([
    {"Model": "Main_Hier_TempScaled", "Split": "val", "Temperature": T_opt, **met_val_cal},
    {"Model": "Main_Hier_TempScaled", "Split": "test", "Temperature": T_opt, **met_test_cal},
])
print_df(cal_df, "CALIBRATED METRICS")
cal_df.to_csv(os.path.join(TABLES_DIR, "calibrated_metrics.csv"), index=False)

# Save calibration plots (before and after)
print("  Saving calibrated calibration plots")
plot_calibration(val_main["y"], val_p_cal, "VAL_MAIN_CAL")
plot_calibration(test_main["y"], test_p_cal, "TEST_MAIN_CAL")


# ============================================================
# BOOTSTRAP CONFIDENCE INTERVALS — ALL MODELS
# ============================================================

def bootstrap_ci_metrics(y: np.ndarray, p: np.ndarray, thr: float, n: int, seed: int) -> pd.DataFrame:
    rng = np.random.RandomState(seed)
    N = len(y)
    rows = []
    for b in tqdm(range(n), desc="  bootstrap", leave=True):
        idx = rng.randint(0, N, size=N)
        yy = y[idx]
        pp = p[idx]
        m = compute_metrics_binary(yy, pp, thr)
        rows.append(m)

    df = pd.DataFrame(rows)
    out_rows = []
    for col in ["F1", "Accuracy", "Precision", "Recall", "AUC_ROC", "AUC_PR", "MCC", "Brier"]:
        out_rows.append({
            "Metric": col,
            "Mean": float(df[col].mean()),
            "Std": float(df[col].std()),
            "CI_2p5": float(np.quantile(df[col], 0.025)),
            "CI_97p5": float(np.quantile(df[col], 0.975)),
        })
    return pd.DataFrame(out_rows)


print("\n[12/15] Bootstrapping confidence intervals (main model test)...")
ci_df = bootstrap_ci_metrics(test_main["y"], test_main["p"], thr_main, cfg.bootstrap_n, cfg.bootstrap_seed)
print_df(ci_df, "BOOTSTRAP CI TABLE (MAIN TEST)")
ci_df.to_csv(os.path.join(TABLES_DIR, "bootstrap_ci_main_test.csv"), index=False)


# ============================================================
# PAIRED BOOTSTRAP SIGNIFICANCE TEST
# Reference: Dror et al. (2018), ACL — "The Hitchhiker's Guide
# to Testing Statistical Significance in NLP"
# ============================================================

def paired_bootstrap_test(
    y: np.ndarray,
    p_model: np.ndarray,
    p_baseline: np.ndarray,
    thr_model: float,
    thr_baseline: float,
    metric_fn,
    n_bootstrap: int = 1000,
    seed: int = 42,
) -> Dict[str, float]:
    """Two-sided paired bootstrap test. Returns p-value for H0: model == baseline."""
    rng = np.random.RandomState(seed)
    N = len(y)
    delta_obs = metric_fn(y, p_model, thr_model) - metric_fn(y, p_baseline, thr_baseline)
    count = 0
    for _ in range(n_bootstrap):
        idx = rng.randint(0, N, size=N)
        d = metric_fn(y[idx], p_model[idx], thr_model) - metric_fn(y[idx], p_baseline[idx], thr_baseline)
        if d <= 0:  # one-sided: model not better
            count += 1
    p_value = count / n_bootstrap
    return {"delta": float(delta_obs), "p_value": float(p_value)}


def f1_metric_fn(y, p, thr):
    return f1_score(y, (p >= thr).astype(int))


# We'll collect baseline test probs for significance testing later
# Store them as we evaluate each baseline
baseline_test_probs = {}


# ============================================================
# ROBUSTNESS
# ============================================================

print("\n[13/15] Robustness analysis...")

def ks_statistic(y: np.ndarray, p: np.ndarray) -> Dict[str, float]:
    from scipy.stats import ks_2samp
    p0 = p[y == 0]
    p1 = p[y == 1]
    stat, pv = ks_2samp(p0, p1)
    return {"KS_statistic": float(stat), "KS_p_value": float(pv)}


ks_res = ks_statistic(test_main["y"], test_main["p"])
print(f"  KS statistic: {ks_res['KS_statistic']:.6f}, p-value: {ks_res['KS_p_value']:.2e}")


def noise_stability_accuracy(p: np.ndarray, y: np.ndarray, thr: float, noise_std: float,
                              trials: int, seed: int = 0) -> Dict[str, float]:
    rng = np.random.RandomState(seed)
    accs = []
    f1s = []
    for _ in range(trials):
        pn = np.clip(p + rng.normal(0.0, noise_std, size=len(p)), 0.0, 1.0)
        pred = (pn >= thr).astype(int)
        accs.append(float(accuracy_score(y, pred)))
        f1s.append(float(f1_score(y, pred)))
    return {
        "noise_mean_acc": float(np.mean(accs)),
        "noise_std_acc": float(np.std(accs)),
        "noise_mean_f1": float(np.mean(f1s)),
        "noise_std_f1": float(np.std(f1s)),
    }


noise_res = noise_stability_accuracy(test_main["p"], test_main["y"], thr_main,
                                      cfg.noise_std, cfg.noise_trials, seed=7)
print(f"  Noise stability: mean_acc={noise_res['noise_mean_acc']:.4f} "
      f"std_acc={noise_res['noise_std_acc']:.4f} "
      f"mean_f1={noise_res['noise_mean_f1']:.4f}")

robust_df = pd.DataFrame([{
    "Model": "Main_Hier",
    "Split": "test",
    "mean_neg_prob": float(np.mean(test_main["p"][test_main["y"] == 0])),
    "mean_pos_prob": float(np.mean(test_main["p"][test_main["y"] == 1])),
    "median_neg_prob": float(np.median(test_main["p"][test_main["y"] == 0])),
    "median_pos_prob": float(np.median(test_main["p"][test_main["y"] == 1])),
    **ks_res,
    **noise_res,
}])
print_df(robust_df, "ROBUSTNESS SUMMARY")
robust_df.to_csv(os.path.join(TABLES_DIR, "robustness_summary.csv"), index=False)

# Save low margin correct examples
print("  Saving low-margin correct cases for error analysis")
test_pred = (test_main["p"] >= thr_main).astype(int)
correct = (test_pred == test_main["y"])
margin = np.abs(test_main["p"] - thr_main)
idx = np.where(correct)[0]
order = idx[np.argsort(margin[idx])]
k = min(500, len(order))
hard_idx = order[:k]
hard_cases = test_df.iloc[hard_idx].copy()
hard_cases["p"] = test_main["p"][hard_idx]
hard_cases["pred"] = test_pred[hard_idx]
hard_cases.to_csv(os.path.join(TABLES_DIR, "hard_correct_low_margin.csv"), index=False)

# Also save misclassified examples
wrong_idx = np.where(~correct)[0]
if len(wrong_idx) > 0:
    wrong_k = min(500, len(wrong_idx))
    wrong_order = wrong_idx[np.argsort(-margin[wrong_idx])]  # worst mistakes first
    wrong_cases = test_df.iloc[wrong_order[:wrong_k]].copy()
    wrong_cases["p"] = test_main["p"][wrong_order[:wrong_k]]
    wrong_cases["pred"] = test_pred[wrong_order[:wrong_k]]
    wrong_cases.to_csv(os.path.join(TABLES_DIR, "misclassified_examples.csv"), index=False)
    print(f"  Saved {wrong_k} misclassified examples")


# ============================================================
# INFERENCE LATENCY BENCHMARKING
# ============================================================

@torch.no_grad()
def benchmark_latency(model: nn.Module, loader: DataLoader, kind: str,
                       warmup_batches: int = 3, measure_batches: int = 20) -> Dict[str, float]:
    """Measure average inference latency per sample."""
    model.eval()
    use_amp = (cfg.device.startswith("cuda") and cfg.use_amp_bf16)

    # Warmup
    for i, batch in enumerate(loader):
        if i >= warmup_batches:
            break
        batch = {k: v.to(cfg.device, non_blocking=True) for k, v in batch.items()}
        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_amp):
            _ = model_forward_logits(model, batch, kind)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    # Measure
    total_time = 0.0
    total_samples = 0
    for i, batch in enumerate(loader):
        if i >= measure_batches:
            break
        batch = {k: v.to(cfg.device, non_blocking=True) for k, v in batch.items()}
        bs = batch["labels"].shape[0]

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_amp):
            _ = model_forward_logits(model, batch, kind)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t1 = time.perf_counter()

        total_time += (t1 - t0)
        total_samples += bs

    ms_per_sample = (total_time / max(1, total_samples)) * 1000.0
    samples_per_sec = total_samples / max(1e-9, total_time)
    return {
        "ms_per_sample": float(ms_per_sample),
        "samples_per_sec": float(samples_per_sec),
        "total_batches": measure_batches,
        "total_samples": total_samples,
    }


print("\n[14/15] Inference latency benchmarking...")
latency_rows = []

# Main model latency
print("  Benchmarking Main_Hier...")
lat_main = benchmark_latency(main_model, test_loader, "main_hier",
                              cfg.latency_warmup_batches, cfg.latency_measure_batches)
latency_rows.append({"Model": "Main_Hier", **lat_main})
print(f"    {lat_main['ms_per_sample']:.2f} ms/sample, {lat_main['samples_per_sec']:.0f} samples/sec")

# Count parameters
def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

param_counts = {"Main_Hier": count_params(main_model)}

# Baseline latencies
for kind, ckpt_path, label in [
    ("single_a", ckpt_deb, "DeBERTa_single"),
    ("single_b", ckpt_rob, "RoBERTa_single"),
    ("fusion_no_disentangle", ckpt_fus, "Fusion_NoDis"),
]:
    print(f"  Benchmarking {label}...")
    model = _build_model_for_kind(kind)
    obj = torch.load(ckpt_path, map_location="cpu")
    sd = obj["state_dict"] if "state_dict" in obj else obj
    sd = load_state_strip_compile_prefix(sd)
    model.load_state_dict(sd, strict=True)
    model.to(cfg.device)
    model.eval()
    param_counts[label] = count_params(model)

    lat = benchmark_latency(model, test_loader, kind,
                             cfg.latency_warmup_batches, cfg.latency_measure_batches)
    latency_rows.append({"Model": label, **lat})
    print(f"    {lat['ms_per_sample']:.2f} ms/sample, {lat['samples_per_sec']:.0f} samples/sec")

    # Collect test probs for significance testing
    _, test_probs = eval_logits_and_probs_binary(model, test_loader, cfg.device, kind)
    baseline_test_probs[label] = test_probs

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

latency_df = pd.DataFrame(latency_rows)
# Add parameter counts
latency_df["Params_M"] = latency_df["Model"].map(
    {k: v / 1e6 for k, v in param_counts.items()}
)
print_df(latency_df, "INFERENCE LATENCY COMPARISON")
latency_df.to_csv(os.path.join(TABLES_DIR, "latency_comparison.csv"), index=False)


# ============================================================
# PAIRED BOOTSTRAP SIGNIFICANCE TESTS VS ALL BASELINES
# ============================================================

print("\n  Running paired bootstrap significance tests...")
sig_rows = []

# Need to get baseline thresholds for fair comparison
baseline_thrs = {
    "DeBERTa_single": thr_deb,
    "RoBERTa_single": thr_rob,
    "Fusion_NoDis": thr_fus,
}

for bl_name, bl_probs in baseline_test_probs.items():
    bl_thr = baseline_thrs.get(bl_name, 0.5)
    result = paired_bootstrap_test(
        test_main["y"], test_main["p"], bl_probs,
        thr_main, bl_thr,
        f1_metric_fn,
        n_bootstrap=cfg.bootstrap_n,
        seed=cfg.bootstrap_seed,
    )
    sig_rows.append({
        "Comparison": f"Main_Hier vs {bl_name}",
        "Delta_F1": result["delta"],
        "p_value": result["p_value"],
        "Significant_005": result["p_value"] < 0.05,
    })
    print(f"  Main_Hier vs {bl_name}: ΔF1={result['delta']:.6f}, p={result['p_value']:.4f}")

sig_df = pd.DataFrame(sig_rows)
print_df(sig_df, "SIGNIFICANCE TESTS (Paired Bootstrap)")
sig_df.to_csv(os.path.join(TABLES_DIR, "significance_tests.csv"), index=False)


# ============================================================
# FINAL COMPARISON TABLE
# ============================================================

print("\n[15/15] Assembling final comparison table...")

final_rows = []
final_rows.append(main_summary)
final_rows.append(cal_df.drop(columns=["Temperature"], errors="ignore"))
final_rows.append(baseline_simple)
final_rows.append(baselines_trained)
final_rows.append(ablations_df)

final_all = pd.concat(final_rows, ignore_index=True)

# Keep consistent columns ordering
col_order = [
    "Model", "Split",
    "F1", "Accuracy", "Precision", "Recall",
    "AUC_ROC", "AUC_PR", "MCC", "Brier", "ECE",
    "Threshold", "PosRate", "PredPosRate",
]
for c in col_order:
    if c not in final_all.columns:
        final_all[c] = np.nan

# Keep only standard columns (drop any extra like F1_aux_*)
extra_cols = [c for c in final_all.columns if c not in col_order]
final_all = final_all[col_order]

print_df(final_all, "FINAL OVERALL TABLE (ALL MODELS)")
final_all.to_csv(os.path.join(TABLES_DIR, "final_overall_table.csv"), index=False)


# ============================================================
# ARTIFACT INDEX
# ============================================================

artifact_index = {
    "out_dir": OUT_DIR,
    "plots_dir": PLOTS_DIR,
    "tables_dir": TABLES_DIR,
    "checkpoints_dir": CKPT_DIR,
    "main_ckpt_used": MAIN_CKPT_PATH,
    "main_threshold": float(thr_main),
    "calibration_temperature": float(T_opt),
    "cka_disentanglement": cka_results,
    "structural_probing": probe_results,
    "parameter_counts": {k: int(v) for k, v in param_counts.items()},
    "files": {
        "tables": sorted([os.path.join(TABLES_DIR, f) for f in os.listdir(TABLES_DIR)]),
        "plots": sorted([os.path.join(PLOTS_DIR, f) for f in os.listdir(PLOTS_DIR)]),
        "checkpoints": sorted([os.path.join(CKPT_DIR, f) for f in os.listdir(CKPT_DIR)]),
    },
    "cfg": asdict(cfg),
}

save_json(os.path.join(OUT_DIR, "artifact_index.json"), artifact_index)


# ============================================================
# SUMMARY PRINTOUT
# ============================================================

print("\n" + "=" * 60)
print("DONE — ALL EVALUATIONS COMPLETE")
print("=" * 60)
print(f"Output folder: {OUT_DIR}")
print(f"Index: {os.path.join(OUT_DIR, 'artifact_index.json')}")
print(f"\nGenerated tables ({len(artifact_index['files']['tables'])}):")
for t in artifact_index["files"]["tables"]:
    print(f"  {os.path.basename(t)}")
print(f"\nGenerated plots ({len(artifact_index['files']['plots'])}):")
for p in artifact_index["files"]["plots"]:
    print(f"  {os.path.basename(p)}")
print(f"\nKey results:")
print(f"  Main model test F1:    {metrics_test_main['F1']:.6f}")
print(f"  Main model test MCC:   {metrics_test_main['MCC']:.6f}")
print(f"  Main model test AUC:   {metrics_test_main['AUC_ROC']:.6f}")
print(f"  Threshold (tuned):     {thr_main:.4f}")
print(f"  Temperature (cal):     {T_opt:.4f}")
print(f"  CKA(lex,syn):          {cka_results['CKA(z_lex,z_syn)']:.4f}")
print(f"  CKA(lex,sem):          {cka_results['CKA(z_lex,z_sem)']:.4f}")
print(f"  CKA(syn,sem):          {cka_results['CKA(z_syn,z_sem)']:.4f}")
print(f"  Ortho penalty:         {cka_results['orthogonality_penalty']:.6f}")
print(f"  Simplex penalty:       {cka_results['simplex_penalty']:.6f}")

ACL FINAL EVALUATION SUITE — FIXED
Device: cuda
Output folder: /mnt/heirarchy/models_stage2_hier/acl_final_package
UMAP available: False

[1/15] Loading data splits...
  Loaded: 960000 train | 120000 val | 120000 test
  Label rates: {'train_pos': 0.6667, 'val_pos': 0.6667, 'test_pos': 0.6667}

[2/15] Loading tokenizers...


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


...

    0.20 ms/sample, 4899 samples/sec
    Eval single_a batch 0/938
    Eval single_a batch 100/938
    Eval single_a batch 200/938
    Eval single_a batch 300/938
    Eval single_a batch 400/938
    Eval single_a batch 500/938
    Eval single_a batch 600/938
    Eval single_a batch 700/938
    Eval single_a batch 800/938
    Eval single_a batch 900/938


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Benchmarking RoBERTa_single...
    0.09 ms/sample, 10826 samples/sec
    Eval single_b batch 0/938
    Eval single_b batch 100/938
    Eval single_b batch 200/938
    Eval single_b batch 300/938
    Eval single_b batch 400/938
    Eval single_b batch 500/938
    Eval single_b batch 600/938
    Eval single_b batch 700/938
    Eval single_b batch 800/938
    Eval single_b batch 900/938
  Benchmarking Fusion_NoDis...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


    0.29 ms/sample, 3451 samples/sec
    Eval fusion_no_disentangle batch 0/938
    Eval fusion_no_disentangle batch 100/938
    Eval fusion_no_disentangle batch 200/938
    Eval fusion_no_disentangle batch 300/938
    Eval fusion_no_disentangle batch 400/938
    Eval fusion_no_disentangle batch 500/938
    Eval fusion_no_disentangle batch 600/938
    Eval fusion_no_disentangle batch 700/938
    Eval fusion_no_disentangle batch 800/938
    Eval fusion_no_disentangle batch 900/938

INFERENCE LATENCY COMPARISON
            Model  ms_per_sample  samples_per_sec  total_batches  total_samples    Params_M
0       Main_Hier       0.521799      1916.445587             20           2560  361.542923
1  DeBERTa_single       0.204117      4899.149749             20           2560  184.423682
2  RoBERTa_single       0.092368     10826.301978             20           2560  124.647170
3    Fusion_NoDis       0.289737      3451.404893             20           2560  309.134594

  Running paired bootstr

In [8]:
import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

# ============================================================
# 1. SETUP /mnt/heirarchy/Final7 AND MIGRATE ASSETS
# ============================================================
FINAL_DIR = "/mnt/heirarchy/Final7"
FINAL_PLOTS_DIR = os.path.join(FINAL_DIR, "plots")
FINAL_TABLES_DIR = os.path.join(FINAL_DIR, "tables")
os.makedirs(FINAL_PLOTS_DIR, exist_ok=True)
os.makedirs(FINAL_TABLES_DIR, exist_ok=True)

print(f"\n[1/2] Setting up {FINAL_DIR} and generating extra plots...")

# Migrate existing data to Final7 if it exists from the previous run
if os.path.exists(OUT_DIR) and OUT_DIR != FINAL_DIR:
    print(f"  Copying existing artifacts from {OUT_DIR} to {FINAL_DIR}")
    shutil.copytree(OUT_DIR, FINAL_DIR, dirs_exist_ok=True)

# ============================================================
# 2. GENERATE EXTRA PLOTS (VIOLIN & THRESHOLD SENSITIVITY)
# ============================================================

# Plot 1: Violin Plot of Score Distributions (Better visual density than histograms)
def plot_violin_scores(y: np.ndarray, p: np.ndarray, thr: float, save_path: str):
    plt.figure(figsize=(7, 5))
    df_plot = pd.DataFrame({"Probability": p, "Class": ["Paraphrase" if label == 1 else "Non-Para" for label in y]})
    sns.violinplot(x="Class", y="Probability", data=df_plot, palette={"Paraphrase": "orange", "Non-Para": "blue"}, inner="quartile")
    plt.axhline(y=thr, color="red", linestyle="--", label=f"Threshold={thr:.3f}")
    plt.title("Violin Plot of Predicted Probabilities (Test Set)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()

# Plot 2: F1 vs. Threshold Sensitivity Curve
def plot_f1_sensitivity(y: np.ndarray, p: np.ndarray, save_path: str):
    from sklearn.metrics import f1_score
    thresholds = np.linspace(0.01, 0.99, 100)
    f1_scores = [f1_score(y, (p >= t).astype(int)) for t in thresholds]
    
    plt.figure(figsize=(7, 5))
    plt.plot(thresholds, f1_scores, label="Main_Hier F1", color="purple", linewidth=2)
    best_idx = np.argmax(f1_scores)
    plt.axvline(x=thresholds[best_idx], color='red', linestyle='--', label=f'Optimal Thr={thresholds[best_idx]:.2f}')
    plt.xlabel("Classification Threshold")
    plt.ylabel("F1 Score")
    plt.title("Model Sensitivity to Threshold Tuning")
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()

# Generate the new plots using the in-memory test_main data from the previous script
plot_violin_scores(test_main["y"], test_main["p"], thr_main, os.path.join(FINAL_PLOTS_DIR, "test_main_violin_dist.png"))
plot_f1_sensitivity(test_main["y"], test_main["p"], os.path.join(FINAL_PLOTS_DIR, "test_main_f1_sensitivity.png"))
print("  Extra plots generated and saved to Final7/plots.")


[1/2] Setting up /mnt/heirarchy/Final7 and generating extra plots...
  Copying existing artifacts from /mnt/heirarchy/models_stage2_hier/acl_final_package to /mnt/heirarchy/Final7
  Extra plots generated and saved to Final7/plots.


In [10]:
# ============================================================
# CROSS-LLM PARAPHRASE TYPE BENCHMARKING
# ============================================================
# Analyses:
#   1. Paraphrase type profiling (lexical, syntactic, semantic)
#      per LLM using auxiliary head scores
#   2. Surface-level metrics: Jaccard, BLEU, edit distance,
#      length ratio per LLM
#   3. Confidence margin analysis (how easy/hard to detect)
#   4. Per-source-text difficulty analysis
#   5. Correlation: surface metrics vs model head scores
#   6. Comparative radar + heatmap + violin visualizations
#
# References:
#   - Wahle et al. (2023), "Paraphrase Types for Generation
#     and Detection", EMNLP 2023
#   - Hewitt & Manning (2019), NAACL
# ============================================================

import os
import json
import time
import warnings
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
from matplotlib.colors import LinearSegmentedColormap

from scipy.stats import spearmanr
from collections import Counter

warnings.filterwarnings("ignore", category=FutureWarning)

# ============================================================
# NOTE: This script assumes the main eval suite has already run
# and all objects below are in scope:
#   cfg, main_model, tok_a, tok_b, thr_main,
#   MAIN_CKPT_PATH, CKPT_DIR,
#   _build_model_for_kind, load_state_strip_compile_prefix,
#   model_forward_logits, probs_from_logits_2,
#   make_loader_dualtok, HierDualEncoderDualTok,
#   DualEncoderFusionNoDisentangle,
#   collate_stack, DualTokPairDataset, normalize_text
#
# If running standalone, paste the model/data definitions from
# acl_eval_suite_fixed.py above this point.
# ============================================================

# Output dirs
BENCH_DIR = os.path.join(cfg.model_root, "llm_benchmark")
BENCH_PLOTS = os.path.join(BENCH_DIR, "plots")
BENCH_TABLES = os.path.join(BENCH_DIR, "tables")
os.makedirs(BENCH_PLOTS, exist_ok=True)
os.makedirs(BENCH_TABLES, exist_ok=True)

print("=" * 60)
print("CROSS-LLM PARAPHRASE TYPE BENCHMARK")
print("=" * 60)


# ============================================================
# 1. LOAD THE LLM PARAPHRASE CSV
# ============================================================

CSV_URL = "https://raw.githubusercontent.com/GIND123/Hierarchical-Transformer/main/OriginalVsLLM.csv"
print("\n[1/7] Loading LLM paraphrase data...")
try:
    llm_df = pd.read_csv(CSV_URL)
    print(f"  Loaded {len(llm_df)} rows from GitHub.")
except Exception as e:
    local_path = os.path.join(cfg.data_root, "OriginalVsLLM.csv")
    print(f"  URL failed ({e}), trying local: {local_path}")
    llm_df = pd.read_csv(local_path)
    print(f"  Loaded {len(llm_df)} rows locally.")

llm_df = llm_df.rename(columns={
    "original_sentence": "sentence1",
    "paraphrased_sentence": "sentence2",
})
llm_df["label"] = 1  # all are true paraphrases

print(f"  LLMs present: {sorted(llm_df['model_name'].unique().tolist())}")
print(f"  Unique source texts: {llm_df['sentence1'].nunique()}")


# ============================================================
# 2. COMPUTE SURFACE-LEVEL PARAPHRASE METRICS
# ============================================================

print("\n[2/7] Computing surface-level paraphrase metrics...")

def jaccard_tokens(s1: str, s2: str) -> float:
    t1 = set(str(s1).lower().split())
    t2 = set(str(s2).lower().split())
    union = t1 | t2
    if len(union) == 0:
        return 0.0
    return len(t1 & t2) / len(union)


def token_edit_distance(s1: str, s2: str) -> int:
    """Word-level Levenshtein distance."""
    a = str(s1).lower().split()
    b = str(s2).lower().split()
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1, dp[i - 1][j - 1] + cost)
    return dp[m][n]


def normalized_edit_distance(s1: str, s2: str) -> float:
    ed = token_edit_distance(s1, s2)
    max_len = max(len(str(s1).split()), len(str(s2).split()), 1)
    return ed / max_len


def length_ratio(s1: str, s2: str) -> float:
    l1 = len(str(s1).split())
    l2 = len(str(s2).split())
    if max(l1, l2) == 0:
        return 1.0
    return min(l1, l2) / max(l1, l2)


def length_change(s1: str, s2: str) -> float:
    """Positive = paraphrase is longer, negative = shorter."""
    l1 = len(str(s1).split())
    l2 = len(str(s2).split())
    return (l2 - l1) / max(l1, 1)


def ngram_overlap(s1: str, s2: str, n: int = 2) -> float:
    """N-gram overlap (Jaccard of n-grams)."""
    def get_ngrams(s, ng):
        toks = str(s).lower().split()
        return set(tuple(toks[i:i + ng]) for i in range(len(toks) - ng + 1))
    ng1 = get_ngrams(s1, n)
    ng2 = get_ngrams(s2, n)
    union = ng1 | ng2
    if len(union) == 0:
        return 0.0
    return len(ng1 & ng2) / len(union)


def novel_word_ratio(s1: str, s2: str) -> float:
    """Fraction of words in paraphrase NOT in original."""
    t1 = set(str(s1).lower().split())
    t2 = str(s2).lower().split()
    if len(t2) == 0:
        return 0.0
    novel = sum(1 for w in t2 if w not in t1)
    return novel / len(t2)


# Compute all surface metrics
surface_records = []
for i, row in llm_df.iterrows():
    s1, s2 = str(row["sentence1"]), str(row["sentence2"])
    surface_records.append({
        "idx": i,
        "model_name": row["model_name"],
        "jaccard": jaccard_tokens(s1, s2),
        "bigram_overlap": ngram_overlap(s1, s2, 2),
        "trigram_overlap": ngram_overlap(s1, s2, 3),
        "norm_edit_dist": normalized_edit_distance(s1, s2),
        "length_ratio": length_ratio(s1, s2),
        "length_change": length_change(s1, s2),
        "novel_word_ratio": novel_word_ratio(s1, s2),
        "orig_len": len(s1.split()),
        "para_len": len(s2.split()),
    })

surface_df = pd.DataFrame(surface_records)

# Aggregate per LLM
surface_agg = surface_df.groupby("model_name").agg({
    "jaccard": ["mean", "std"],
    "bigram_overlap": ["mean", "std"],
    "trigram_overlap": ["mean", "std"],
    "norm_edit_dist": ["mean", "std"],
    "length_ratio": ["mean", "std"],
    "length_change": ["mean", "std"],
    "novel_word_ratio": ["mean", "std"],
}).round(4)
surface_agg.columns = ["_".join(c) for c in surface_agg.columns]

print("\n  Surface Metrics per LLM:")
with pd.option_context("display.width", 200, "display.max_columns", 20):
    print(surface_agg)

surface_agg.to_csv(os.path.join(BENCH_TABLES, "surface_metrics_per_llm.csv"))
surface_df.to_csv(os.path.join(BENCH_TABLES, "surface_metrics_all.csv"), index=False)


# ============================================================
# 3. RUN MAIN MODEL INFERENCE — EXTRACT ALL HEAD SCORES
# ============================================================

print("\n[3/7] Running main model inference on LLM paraphrases...")

llm_loader = make_loader_dualtok(llm_df, tok_a, tok_b, min(cfg.batch_size_eval, len(llm_df)),
                                  False, 0)  # num_workers=0 for small dataset

@torch.no_grad()
def run_hier_inference_detailed(model, loader):
    """Run hierarchical model and return per-head probabilities and features."""
    model.eval()
    use_amp = cfg.device.startswith("cuda") and cfg.use_amp_bf16
    results = {"log_meta": [], "log_lex": [], "log_syn": [], "log_sem": [],
               "z_lex": [], "z_syn": [], "z_sem": []}

    for batch in loader:
        batch = {k: v.to(cfg.device, non_blocking=True) for k, v in batch.items()}
        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_amp):
            out = model(
                batch["input_ids_a"], batch["attention_mask_a"],
                batch["input_ids_b"], batch["attention_mask_b"],
                return_features=True,
            )
        for key in results:
            results[key].append(out[key].detach().float().cpu().numpy())

    return {k: np.concatenate(v) for k, v in results.items()}


hier_out = run_hier_inference_detailed(main_model, llm_loader)

# Convert logits to probabilities (class 1 = paraphrase)
p_meta = probs_from_logits_2(hier_out["log_meta"])
p_lex = probs_from_logits_2(hier_out["log_lex"])
p_syn = probs_from_logits_2(hier_out["log_syn"])
p_sem = probs_from_logits_2(hier_out["log_sem"])

llm_df["p_meta"] = p_meta
llm_df["p_lex"] = p_lex
llm_df["p_syn"] = p_syn
llm_df["p_sem"] = p_sem
llm_df["confidence_margin"] = p_meta - thr_main
llm_df["detected"] = (p_meta >= thr_main).astype(int)

# Merge surface metrics
for col in ["jaccard", "bigram_overlap", "norm_edit_dist", "length_change", "novel_word_ratio"]:
    llm_df[col] = surface_df[col].values


# ============================================================
# 4. RUN BASELINE MODELS FOR COMPARISON
# ============================================================

print("\n[4/7] Running baseline models on LLM paraphrases...")

baseline_configs = {
    "DeBERTa_Single": {"kind": "single_a", "ckpt": os.path.join(CKPT_DIR, "baseline_single_a.pt")},
    "RoBERTa_Single": {"kind": "single_b", "ckpt": os.path.join(CKPT_DIR, "baseline_single_b.pt")},
    "Fusion_NoDis": {"kind": "fusion_no_disentangle", "ckpt": os.path.join(CKPT_DIR, "fusion_no_disentangle.pt")},
}

for model_label, info in baseline_configs.items():
    if not os.path.exists(info["ckpt"]):
        print(f"  Skipping {model_label} — checkpoint not found")
        llm_df[f"p_{model_label}"] = np.nan
        continue

    print(f"  Running {model_label}...")
    model = _build_model_for_kind(info["kind"])
    obj = torch.load(info["ckpt"], map_location="cpu")
    sd = obj.get("state_dict", obj)
    sd = load_state_strip_compile_prefix(sd)
    model.load_state_dict(sd, strict=True)
    model.to(cfg.device)
    model.eval()

    bl_logits = []
    use_amp = cfg.device.startswith("cuda") and cfg.use_amp_bf16
    with torch.no_grad():
        for batch in llm_loader:
            batch = {k: v.to(cfg.device, non_blocking=True) for k, v in batch.items()}
            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_amp):
                out = model_forward_logits(model, batch, info["kind"])
            bl_logits.append(out.detach().float().cpu().numpy())

    bl_probs = probs_from_logits_2(np.concatenate(bl_logits))
    llm_df[f"p_{model_label}"] = bl_probs
    print(f"    Mean confidence: {bl_probs.mean():.4f}")

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Save full annotated dataframe
llm_df.to_csv(os.path.join(BENCH_TABLES, "llm_paraphrases_annotated.csv"), index=False)


# ============================================================
# 5. AGGREGATE ANALYSIS TABLES
# ============================================================

print("\n[5/7] Building analysis tables...")

# --- Table A: Per-LLM head score profile ---
head_profile = llm_df.groupby("model_name").agg({
    "p_meta": ["mean", "std", "min"],
    "p_lex": ["mean", "std"],
    "p_syn": ["mean", "std"],
    "p_sem": ["mean", "std"],
    "confidence_margin": ["mean", "min"],
    "detected": "mean",
}).round(4)
head_profile.columns = ["_".join(c) for c in head_profile.columns]
head_profile = head_profile.sort_values("p_meta_mean", ascending=True)

print("\n  Per-LLM Head Score Profile:")
with pd.option_context("display.width", 200, "display.max_columns", 30):
    print(head_profile)
head_profile.to_csv(os.path.join(BENCH_TABLES, "head_profile_per_llm.csv"))

# --- Table B: Per-LLM paraphrase type characterization ---
# Classify dominant paraphrase type per sample based on which head
# has the LOWEST confidence (= most challenging at that level)
def classify_para_type(row):
    scores = {"lexical": row["p_lex"], "syntactic": row["p_syn"], "semantic": row["p_sem"]}
    weakest = min(scores, key=scores.get)
    strongest = max(scores, key=scores.get)
    return pd.Series({"weakest_level": weakest, "strongest_level": strongest,
                       "head_gap": scores[strongest] - scores[weakest]})

type_info = llm_df.apply(classify_para_type, axis=1)
llm_df = pd.concat([llm_df, type_info], axis=1)

type_dist = llm_df.groupby(["model_name", "weakest_level"]).size().unstack(fill_value=0)
print("\n  Weakest Detection Level Distribution per LLM:")
print(type_dist)
type_dist.to_csv(os.path.join(BENCH_TABLES, "weakest_level_distribution.csv"))

# --- Table C: Combined surface + model analysis per LLM ---
combined_profile = llm_df.groupby("model_name").agg({
    "jaccard": "mean",
    "novel_word_ratio": "mean",
    "norm_edit_dist": "mean",
    "length_change": "mean",
    "p_meta": "mean",
    "p_lex": "mean",
    "p_syn": "mean",
    "p_sem": "mean",
    "head_gap": "mean",
}).round(4)
combined_profile.columns = [c if c.startswith("p_") or c == "head_gap"
                             else f"surf_{c}" for c in combined_profile.columns]
print("\n  Combined Profile:")
with pd.option_context("display.width", 200):
    print(combined_profile)
combined_profile.to_csv(os.path.join(BENCH_TABLES, "combined_profile_per_llm.csv"))

# --- Table D: Per source text difficulty ---
source_difficulty = llm_df.groupby("sentence1").agg({
    "p_meta": ["mean", "std", "min"],
    "confidence_margin": "mean",
    "jaccard": "mean",
    "novel_word_ratio": "mean",
}).round(4)
source_difficulty.columns = ["_".join(c) for c in source_difficulty.columns]
source_difficulty = source_difficulty.sort_values("p_meta_min")
source_difficulty.to_csv(os.path.join(BENCH_TABLES, "per_source_difficulty.csv"))

# --- Table E: Correlation between surface metrics and model scores ---
corr_metrics = ["jaccard", "bigram_overlap", "norm_edit_dist", "novel_word_ratio",
                "length_change"]
model_scores = ["p_meta", "p_lex", "p_syn", "p_sem"]

corr_rows = []
for surf in corr_metrics:
    for mscore in model_scores:
        rho, pval = spearmanr(llm_df[surf].values, llm_df[mscore].values)
        corr_rows.append({"surface_metric": surf, "model_score": mscore,
                          "spearman_rho": round(rho, 4), "p_value": round(pval, 6)})

corr_df = pd.DataFrame(corr_rows)
print("\n  Correlation: Surface Metrics vs Model Scores:")
with pd.option_context("display.width", 200):
    print(corr_df)
corr_df.to_csv(os.path.join(BENCH_TABLES, "correlation_surface_vs_model.csv"), index=False)


# ============================================================
# 6. VISUALIZATIONS
# ============================================================

print("\n[6/7] Generating visualizations...")

# --- Color palette ---
LLM_COLORS = {
    "Claude Haiku 3.5": "#E07B39",
    "Claude Sonnet 4.5": "#D94F30",
    "Claude Opus": "#B03A2E",
    "Gpt 5": "#2E86C1",
    "Grok 4": "#8E44AD",
    "Llama 4": "#27AE60",
}
HEAD_COLORS = {"p_lex": "#3498DB", "p_syn": "#E74C3C", "p_sem": "#2ECC71"}


def save_plot(path):
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close()


# --- Plot 1: Grouped bar — head scores per LLM ---
fig, ax = plt.subplots(figsize=(12, 6))
llms = sorted(llm_df["model_name"].unique())
x = np.arange(len(llms))
width = 0.2

for i, (head, color) in enumerate(HEAD_COLORS.items()):
    means = [llm_df[llm_df["model_name"] == llm][head].mean() for llm in llms]
    stds = [llm_df[llm_df["model_name"] == llm][head].std() for llm in llms]
    label = head.replace("p_", "").capitalize()
    ax.bar(x + i * width, means, width, yerr=stds, label=label,
           color=color, alpha=0.85, capsize=3, edgecolor="white", linewidth=0.5)

# Add meta score as diamonds
meta_means = [llm_df[llm_df["model_name"] == llm]["p_meta"].mean() for llm in llms]
ax.scatter(x + width, meta_means, marker="D", s=80, c="black", zorder=5,
           label="Meta (final)")

ax.set_xlabel("LLM Generator", fontsize=12)
ax.set_ylabel("Paraphrase Probability (Head Score)", fontsize=12)
ax.set_title("Auxiliary Head Scores per LLM: Lexical vs Syntactic vs Semantic", fontsize=14, fontweight="bold")
ax.set_xticks(x + width)
ax.set_xticklabels(llms, rotation=20, ha="right")
ax.legend(loc="lower left", framealpha=0.9)
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.3)
save_plot(os.path.join(BENCH_PLOTS, "head_scores_per_llm.png"))
print("  Saved head_scores_per_llm.png")


# --- Plot 2: Heatmap — head scores (LLM × Head) ---
fig, ax = plt.subplots(figsize=(8, 5))
heat_data = llm_df.groupby("model_name")[["p_lex", "p_syn", "p_sem", "p_meta"]].mean()
heat_data = heat_data.loc[llms]
heat_data.columns = ["Lexical", "Syntactic", "Semantic", "Meta"]

cmap = LinearSegmentedColormap.from_list("custom", ["#FDEBD0", "#E74C3C", "#7B241C"])
im = ax.imshow(heat_data.values, cmap=cmap, aspect="auto", vmin=0.5, vmax=1.0)

ax.set_xticks(range(4))
ax.set_xticklabels(heat_data.columns, fontsize=11)
ax.set_yticks(range(len(llms)))
ax.set_yticklabels(llms, fontsize=10)

for i in range(len(llms)):
    for j in range(4):
        val = heat_data.values[i, j]
        color = "white" if val > 0.85 else "black"
        ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=10, color=color,
                fontweight="bold")

ax.set_title("Mean Paraphrase Detection Confidence by Head and LLM", fontsize=13, fontweight="bold")
plt.colorbar(im, ax=ax, label="Confidence", shrink=0.8)
save_plot(os.path.join(BENCH_PLOTS, "heatmap_head_x_llm.png"))
print("  Saved heatmap_head_x_llm.png")


# --- Plot 3: Radar chart — per LLM paraphrase profile ---
fig, axes = plt.subplots(2, 3, figsize=(16, 11), subplot_kw=dict(polar=True))
axes = axes.flatten()

categories = ["Lexical\nHead", "Syntactic\nHead", "Semantic\nHead",
              "Novel\nWords", "Edit\nDistance"]
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

for idx, llm in enumerate(llms):
    ax = axes[idx]
    sub = llm_df[llm_df["model_name"] == llm]
    values = [
        sub["p_lex"].mean(),
        sub["p_syn"].mean(),
        sub["p_sem"].mean(),
        sub["novel_word_ratio"].mean(),
        sub["norm_edit_dist"].mean(),
    ]
    values += values[:1]

    color = LLM_COLORS.get(llm, "#555555")
    ax.fill(angles, values, alpha=0.25, color=color)
    ax.plot(angles, values, "o-", color=color, linewidth=2, markersize=6)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_title(llm, fontsize=11, fontweight="bold", pad=15, color=color)
    ax.grid(alpha=0.3)

plt.suptitle("Paraphrase Profile Radar: Detection Heads + Surface Metrics",
             fontsize=15, fontweight="bold", y=1.02)
save_plot(os.path.join(BENCH_PLOTS, "radar_per_llm.png"))
print("  Saved radar_per_llm.png")


# --- Plot 4: Violin plot — confidence distribution per LLM ---
fig, axes = plt.subplots(1, 4, figsize=(18, 5), sharey=True)
heads = [("p_meta", "Meta Classifier"), ("p_lex", "Lexical Head"),
         ("p_syn", "Syntactic Head"), ("p_sem", "Semantic Head")]

for ax, (col, title) in zip(axes, heads):
    data_per_llm = [llm_df[llm_df["model_name"] == llm][col].values for llm in llms]
    parts = ax.violinplot(data_per_llm, showmeans=True, showmedians=True)

    for i, pc in enumerate(parts["bodies"]):
        color = LLM_COLORS.get(llms[i], "#555555")
        pc.set_facecolor(color)
        pc.set_alpha(0.6)

    ax.set_xticks(range(1, len(llms) + 1))
    ax.set_xticklabels([l.replace(" ", "\n") for l in llms], fontsize=8)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_ylabel("Confidence" if ax == axes[0] else "")
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("Confidence Distribution per LLM across Detection Heads",
             fontsize=14, fontweight="bold")
save_plot(os.path.join(BENCH_PLOTS, "violin_confidence_per_llm.png"))
print("  Saved violin_confidence_per_llm.png")


# --- Plot 5: Surface metrics comparison ---
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

surf_cols = [
    ("jaccard", "Token Jaccard Similarity"),
    ("novel_word_ratio", "Novel Word Ratio"),
    ("norm_edit_dist", "Normalized Edit Distance"),
    ("length_change", "Length Change (fraction)"),
    ("bigram_overlap", "Bigram Overlap"),
]

for idx, (col, title) in enumerate(surf_cols):
    ax = axes[idx]
    means = [llm_df[llm_df["model_name"] == llm][col].mean() for llm in llms]
    stds = [llm_df[llm_df["model_name"] == llm][col].std() for llm in llms]
    colors = [LLM_COLORS.get(llm, "#555") for llm in llms]

    bars = ax.bar(range(len(llms)), means, yerr=stds, color=colors, alpha=0.8,
                  capsize=4, edgecolor="white", linewidth=0.5)
    ax.set_xticks(range(len(llms)))
    ax.set_xticklabels([l.replace(" ", "\n") for l in llms], fontsize=8)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.grid(axis="y", alpha=0.3)

# Hide empty subplot
axes[-1].axis("off")

plt.suptitle("Surface-Level Paraphrase Metrics per LLM", fontsize=14, fontweight="bold")
save_plot(os.path.join(BENCH_PLOTS, "surface_metrics_per_llm.png"))
print("  Saved surface_metrics_per_llm.png")


# --- Plot 6: Correlation heatmap (surface metrics vs model heads) ---
fig, ax = plt.subplots(figsize=(8, 5))

corr_pivot = corr_df.pivot(index="surface_metric", columns="model_score", values="spearman_rho")
corr_pivot = corr_pivot[["p_lex", "p_syn", "p_sem", "p_meta"]]
corr_pivot.columns = ["Lexical", "Syntactic", "Semantic", "Meta"]

cmap_div = LinearSegmentedColormap.from_list("rg", ["#2E86C1", "white", "#E74C3C"])
im = ax.imshow(corr_pivot.values, cmap=cmap_div, vmin=-1, vmax=1, aspect="auto")

ax.set_xticks(range(4))
ax.set_xticklabels(corr_pivot.columns, fontsize=11)
ax.set_yticks(range(len(corr_pivot)))
ax.set_yticklabels(corr_pivot.index, fontsize=10)

for i in range(len(corr_pivot)):
    for j in range(4):
        val = corr_pivot.values[i, j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=10, fontweight="bold")

ax.set_title("Spearman Correlation: Surface Metrics vs Model Head Scores",
             fontsize=13, fontweight="bold")
plt.colorbar(im, ax=ax, label="Spearman ρ", shrink=0.8)
save_plot(os.path.join(BENCH_PLOTS, "correlation_heatmap.png"))
print("  Saved correlation_heatmap.png")


# --- Plot 7: Per-model comparison — who detects what best? ---
fig, ax = plt.subplots(figsize=(12, 6))

model_cols = ["p_meta"]
for bl in baseline_configs:
    col = f"p_{bl}"
    if col in llm_df.columns and llm_df[col].notna().all():
        model_cols.append(col)

model_labels = {
    "p_meta": "Hier (Ours)",
    "p_DeBERTa_Single": "DeBERTa",
    "p_RoBERTa_Single": "RoBERTa",
    "p_Fusion_NoDis": "Fusion (no dis.)",
}

bar_colors = ["#2C3E50", "#3498DB", "#E74C3C", "#F39C12"]
width = 0.8 / len(model_cols)

for i, col in enumerate(model_cols):
    means = [llm_df[llm_df["model_name"] == llm][col].mean() for llm in llms]
    label = model_labels.get(col, col)
    ax.bar(np.arange(len(llms)) + i * width, means, width, label=label,
           color=bar_colors[i % len(bar_colors)], alpha=0.85, edgecolor="white", linewidth=0.5)

ax.set_xlabel("LLM Generator", fontsize=12)
ax.set_ylabel("Mean Detection Confidence", fontsize=12)
ax.set_title("Detection Confidence: Hierarchical Model vs Baselines across LLMs",
             fontsize=14, fontweight="bold")
ax.set_xticks(np.arange(len(llms)) + width * len(model_cols) / 2)
ax.set_xticklabels(llms, rotation=20, ha="right")
ax.legend(loc="lower left")
ax.set_ylim(0.5, 1.02)
ax.grid(axis="y", alpha=0.3)
save_plot(os.path.join(BENCH_PLOTS, "model_comparison_per_llm.png"))
print("  Saved model_comparison_per_llm.png")


# --- Plot 8: Scatter — lexical overlap vs semantic head confidence ---
fig, ax = plt.subplots(figsize=(8, 6))
for llm in llms:
    sub = llm_df[llm_df["model_name"] == llm]
    color = LLM_COLORS.get(llm, "#555")
    ax.scatter(sub["jaccard"], sub["p_sem"], s=60, alpha=0.7, color=color,
               edgecolors="white", linewidth=0.5, label=llm)

ax.set_xlabel("Token Jaccard Similarity (lexical overlap)", fontsize=12)
ax.set_ylabel("Semantic Head Confidence", fontsize=12)
ax.set_title("Lexical Overlap vs Semantic Detection Confidence", fontsize=13, fontweight="bold")
ax.legend(fontsize=9, framealpha=0.9)
ax.grid(alpha=0.3)
save_plot(os.path.join(BENCH_PLOTS, "scatter_jaccard_vs_semantic.png"))
print("  Saved scatter_jaccard_vs_semantic.png")


# ============================================================
# 7. SUMMARY
# ============================================================

print("\n[7/7] Generating summary...")

summary = {
    "total_samples": len(llm_df),
    "llms": llms,
    "source_texts": int(llm_df["sentence1"].nunique()),
    "overall_detection_rate": float(llm_df["detected"].mean()),
    "overall_meta_confidence": float(llm_df["p_meta"].mean()),
    "per_llm_meta_confidence": {
        llm: round(float(llm_df[llm_df["model_name"] == llm]["p_meta"].mean()), 4)
        for llm in llms
    },
    "per_llm_weakest_head": {},
    "paraphrase_style_characterization": {},
    "files": {
        "tables": sorted(os.listdir(BENCH_TABLES)),
        "plots": sorted(os.listdir(BENCH_PLOTS)),
    },
}

# Characterize each LLM's paraphrase style
for llm in llms:
    sub = llm_df[llm_df["model_name"] == llm]
    heads = {"lexical": sub["p_lex"].mean(), "syntactic": sub["p_syn"].mean(),
             "semantic": sub["p_sem"].mean()}
    weakest = min(heads, key=heads.get)
    summary["per_llm_weakest_head"][llm] = weakest

    # Style characterization
    novel_ratio = sub["novel_word_ratio"].mean()
    edit_dist = sub["norm_edit_dist"].mean()
    jaccard_val = sub["jaccard"].mean()

    if novel_ratio > 0.4:
        style = "heavy_lexical_rewrite"
    elif edit_dist > 0.5:
        style = "structural_restructuring"
    elif jaccard_val > 0.6:
        style = "light_surface_edit"
    else:
        style = "moderate_paraphrase"

    summary["paraphrase_style_characterization"][llm] = {
        "style": style,
        "novel_word_ratio": round(float(novel_ratio), 4),
        "edit_distance": round(float(edit_dist), 4),
        "jaccard": round(float(jaccard_val), 4),
    }

with open(os.path.join(BENCH_DIR, "benchmark_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 60)
print("BENCHMARK COMPLETE")
print("=" * 60)
print(f"Output: {BENCH_DIR}")
print(f"Tables: {len(summary['files']['tables'])} files")
print(f"Plots:  {len(summary['files']['plots'])} files")
print(f"\nKey Findings:")
print(f"  Overall detection rate: {summary['overall_detection_rate']:.1%}")
print(f"  Overall meta confidence: {summary['overall_meta_confidence']:.4f}")
print(f"\n  Per-LLM Paraphrase Style:")
for llm in llms:
    s = summary["paraphrase_style_characterization"][llm]
    wh = summary["per_llm_weakest_head"][llm]
    print(f"    {llm:22s} | style={s['style']:28s} | weakest_head={wh:10s} "
          f"| novel_words={s['novel_word_ratio']:.3f} | edit_dist={s['edit_distance']:.3f}")

CROSS-LLM PARAPHRASE TYPE BENCHMARK

[1/7] Loading LLM paraphrase data...
  Loaded 60 rows from GitHub.
  LLMs present: ['Claude Haiku 3.5', 'Claude Opus', 'Claude Sonnet 4.5', 'Gpt 5', 'Grok 4', 'Llama 4']
  Unique source texts: 11

[2/7] Computing surface-level paraphrase metrics...

  Surface Metrics per LLM:
                   jaccard_mean  jaccard_std  bigram_overlap_mean  bigram_overlap_std  trigram_overlap_mean  trigram_overlap_std  norm_edit_dist_mean  norm_edit_dist_std  length_ratio_mean  \
model_name                                                                                                                                                                                     
Claude Haiku 3.5         0.2475       0.1221               0.1000              0.0936                0.0464               0.0578               0.7534              0.1566             0.9269   
Claude Opus              0.3901       0.0741               0.1700              0.0780                0.0790   

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


    Mean confidence: 1.0000
  Running RoBERTa_Single...


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


    Mean confidence: 0.8272
  Running Fusion_NoDis...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


    Mean confidence: 0.9996

[5/7] Building analysis tables...

  Per-LLM Head Score Profile:
                   p_meta_mean  p_meta_std  p_meta_min  p_lex_mean  p_lex_std  p_syn_mean  p_syn_std  p_sem_mean  p_sem_std  confidence_margin_mean  confidence_margin_min  detected_mean
model_name                                                                                                                                                                               
Gpt 5                   0.9999      0.0001      0.9996         1.0     0.0001         1.0     0.0001      0.9999     0.0003                  0.4455                 0.4452            1.0
Claude Sonnet 4.5       0.9999      0.0000      0.9999         1.0     0.0000         1.0     0.0000      0.9999     0.0000                  0.4455                 0.4455            1.0
Grok 4                  0.9999      0.0001      0.9998         1.0     0.0000         1.0     0.0001      0.9999     0.0001                  0.4455               

In [12]:
%uv pip install umap-learn

Using Python 3.12.6 environment at: /usr/local
Resolved 10 packages in 91ms
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
pynndescent ------------------------------ 14.91 KiB/71.79 KiB
⠙ Preparing packages... (0/2)
pynndescent ------------------------------ 30.91 KiB/71.79 KiB
⠙ Preparing packages... (0/2)
pynndescent ------------------------------ 30.91 KiB/71.79 KiB
⠙ Preparing packages... (0/2)
pynndescent ------------------------------ 46.91 KiB/71.79 KiB
⠙ Preparing packages... (0/2)
pynndescent ------------------------------ 62.91 KiB/71.79 KiB
⠙ Preparing packages... (0/2)
pynndescent ------------------------------ 62.91 KiB/71.79 KiB
⠙ Preparing packages... (0/2)
pynndescent ------------------------------ 71.79 KiB/71.79 KiB
⠙ Preparing packages... (0/2)
pynndescent ------------------------------ 71.79 KiB/71.79 KiB
⠙ Preparing packages... (0/2)
pynndescent ------------------------------ 71.79 KiB/71.79 KiB
⠙ Preparing packages... (0/

In [13]:
# ============================================================
# UMAP SUBSPACE VISUALIZATION SUITE
# ============================================================
# Produces publication-quality UMAP projections of z_lex,
# z_syn, z_sem for:
#   A. Main test set (para vs non-para)
#   B. LLM paraphrases (colored by generator LLM)
#   C. Combined overlay (test background + LLM foreground)
#   D. Confidence-colored projections
#   E. Side-by-side 3-panel figure for the paper
#   F. Interactive HTML explorer
#
# Assumes the main eval suite + benchmark have run,
# so these are in scope:
#   main_model, cfg, tok_a, tok_b, thr_main,
#   test_main (dict with z_lex, z_syn, z_sem, y, p),
#   llm_df (annotated), hier_out (dict with z_lex, z_syn, z_sem),
#   val_main, BENCH_DIR, BENCH_PLOTS, etc.
#
# If UMAP is not installed: pip install umap-learn
# ============================================================

import os
import json
import time
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import matplotlib.patheffects as pe

try:
    import umap
    UMAP_OK = True
except ImportError:
    print("ERROR: umap-learn not installed. Run: pip install umap-learn")
    UMAP_OK = False

# ============================================================
# CONFIG
# ============================================================

UMAP_DIR = os.path.join(cfg.model_root, "umap_visualizations")
os.makedirs(UMAP_DIR, exist_ok=True)

# Subsample test set for speed (UMAP is O(n log n) but still slow)
MAX_TEST_POINTS = 30000
UMAP_PARAMS = dict(n_neighbors=30, min_dist=0.3, metric="cosine", random_state=42)

LLM_COLORS = {
    "Claude Haiku 3.5": "#E07B39",
    "Claude Sonnet 4.5": "#D94F30",
    "Claude Opus":       "#B03A2E",
    "Gpt 5":             "#2E86C1",
    "Grok 4":            "#8E44AD",
    "Llama 4":           "#27AE60",
}

LABEL_COLORS = {0: "#3498DB", 1: "#E74C3C"}  # non-para=blue, para=red
HEAD_NAMES = {"z_lex": "Lexical", "z_syn": "Syntactic", "z_sem": "Semantic"}


def save_fig(path, dpi=300):
    plt.savefig(path, dpi=dpi, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"  Saved: {os.path.basename(path)}")


# ============================================================
# 1. PREPARE DATA
# ============================================================

print("=" * 60)
print("UMAP SUBSPACE VISUALIZATION SUITE")
print("=" * 60)

if not UMAP_OK:
    print("UMAP not available — skipping all UMAP plots.")
else:
    # --- Test set subsample ---
    n_test = len(test_main["y"])
    if n_test > MAX_TEST_POINTS:
        rng = np.random.RandomState(42)
        idx_sub = rng.choice(n_test, MAX_TEST_POINTS, replace=False)
        idx_sub.sort()
    else:
        idx_sub = np.arange(n_test)

    test_y = test_main["y"][idx_sub]
    test_p = test_main["p"][idx_sub]
    test_z = {
        "z_lex": test_main["z_lex"][idx_sub],
        "z_syn": test_main["z_syn"][idx_sub],
        "z_sem": test_main["z_sem"][idx_sub],
    }
    print(f"\n[1/6] Test set subsample: {len(idx_sub)} points")

    # --- LLM embeddings (already small, ~60 points) ---
    llm_z = {
        "z_lex": hier_out["z_lex"],
        "z_syn": hier_out["z_syn"],
        "z_sem": hier_out["z_sem"],
    }
    llm_labels = llm_df["model_name"].values
    llm_llms = sorted(llm_df["model_name"].unique().tolist())
    print(f"  LLM samples: {len(llm_labels)} points, {len(llm_llms)} LLMs")


    # ============================================================
    # 2. FIT UMAP PER SUBSPACE
    # ============================================================

    print(f"\n[2/6] Fitting UMAP projections (this may take a few minutes)...")

    umap_test = {}
    umap_llm = {}
    umap_combined = {}
    umap_models = {}

    for key, name in HEAD_NAMES.items():
        t0 = time.time()
        print(f"  Fitting {name} subspace...", end=" ", flush=True)

        # Combined: test + LLM so they share the same projection space
        Z_test = test_z[key]
        Z_llm = llm_z[key]
        Z_all = np.vstack([Z_test, Z_llm])

        reducer = umap.UMAP(**UMAP_PARAMS)
        emb_all = reducer.fit_transform(Z_all)

        n_t = len(Z_test)
        umap_test[key] = emb_all[:n_t]
        umap_llm[key] = emb_all[n_t:]
        umap_combined[key] = emb_all
        umap_models[key] = reducer

        elapsed = time.time() - t0
        print(f"done ({elapsed:.1f}s)")


    # ============================================================
    # 3. PLOT A: TEST SET — PARA VS NON-PARA (3 PANELS)
    # ============================================================

    print(f"\n[3/6] Plotting test set UMAP (para vs non-para)...")

    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    for ax, (key, name) in zip(axes, HEAD_NAMES.items()):
        emb = umap_test[key]
        # Non-para first (background)
        mask0 = test_y == 0
        mask1 = test_y == 1
        ax.scatter(emb[mask0, 0], emb[mask0, 1], s=1.5, c=LABEL_COLORS[0],
                   alpha=0.15, rasterized=True, label="Non-Paraphrase")
        ax.scatter(emb[mask1, 0], emb[mask1, 1], s=1.5, c=LABEL_COLORS[1],
                   alpha=0.15, rasterized=True, label="Paraphrase")

        ax.set_title(f"{name} Subspace", fontsize=16, fontweight="bold", pad=10)
        ax.set_xlabel("UMAP-1", fontsize=11)
        ax.set_ylabel("UMAP-2" if ax == axes[0] else "", fontsize=11)
        ax.tick_params(labelsize=9)
        ax.set_aspect("equal", adjustable="datalim")

        # Clean axes
        for spine in ax.spines.values():
            spine.set_linewidth(0.5)
            spine.set_color("#CCCCCC")

    # Shared legend
    legend_elements = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor=LABEL_COLORS[0],
               markersize=8, label="Non-Paraphrase"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor=LABEL_COLORS[1],
               markersize=8, label="Paraphrase"),
    ]
    fig.legend(handles=legend_elements, loc="lower center", ncol=2, fontsize=12,
               frameon=True, fancybox=True, shadow=False, bbox_to_anchor=(0.5, -0.02))

    fig.suptitle("UMAP Projections of Disentangled Subspaces (Test Set)",
                 fontsize=18, fontweight="bold", y=1.03)
    save_fig(os.path.join(UMAP_DIR, "umap_test_para_vs_nonpara.png"))


    # ============================================================
    # 4. PLOT B: LLM PARAPHRASES — COLORED BY LLM (3 PANELS)
    # ============================================================

    print(f"\n[4/6] Plotting LLM paraphrases UMAP (colored by generator)...")

    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    for ax, (key, name) in zip(axes, HEAD_NAMES.items()):
        emb_test = umap_test[key]
        emb_llm = umap_llm[key]

        # Background: test set as light gray
        ax.scatter(emb_test[:, 0], emb_test[:, 1], s=0.5, c="#E0E0E0",
                   alpha=0.08, rasterized=True, zorder=1)

        # Foreground: LLM samples with distinct colors
        for llm in llm_llms:
            mask = llm_labels == llm
            color = LLM_COLORS.get(llm, "#555555")
            ax.scatter(emb_llm[mask, 0], emb_llm[mask, 1],
                       s=80, c=color, alpha=0.9, edgecolors="white",
                       linewidth=0.8, zorder=5, label=llm)

        ax.set_title(f"{name} Subspace", fontsize=16, fontweight="bold", pad=10)
        ax.set_xlabel("UMAP-1", fontsize=11)
        ax.set_ylabel("UMAP-2" if ax == axes[0] else "", fontsize=11)
        ax.tick_params(labelsize=9)
        ax.set_aspect("equal", adjustable="datalim")
        for spine in ax.spines.values():
            spine.set_linewidth(0.5)
            spine.set_color("#CCCCCC")

    # Shared legend
    legend_elements = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor=LLM_COLORS.get(llm, "#555"),
               markersize=9, markeredgecolor="white", markeredgewidth=0.5, label=llm)
        for llm in llm_llms
    ]
    legend_elements.insert(0, Line2D([0], [0], marker="o", color="w",
                                      markerfacecolor="#E0E0E0", markersize=6,
                                      label="Test set (background)"))
    fig.legend(handles=legend_elements, loc="lower center", ncol=4, fontsize=10,
               frameon=True, fancybox=True, shadow=False, bbox_to_anchor=(0.5, -0.06))

    fig.suptitle("LLM Paraphrases Projected into Disentangled Subspaces",
                 fontsize=18, fontweight="bold", y=1.03)
    save_fig(os.path.join(UMAP_DIR, "umap_llm_by_generator.png"))


    # ============================================================
    # 5. PLOT C: CONFIDENCE-COLORED (META SCORE)
    # ============================================================

    print(f"\n[5/6] Plotting confidence-colored UMAP...")

    fig, axes = plt.subplots(1, 3, figsize=(21, 6))

    cmap_conf = LinearSegmentedColormap.from_list("conf",
        ["#2C3E50", "#E74C3C", "#F39C12", "#2ECC71"])

    for ax, (key, name) in zip(axes, HEAD_NAMES.items()):
        emb = umap_test[key]
        sc = ax.scatter(emb[:, 0], emb[:, 1], s=1.5, c=test_p[idx_sub] if len(test_p) > len(idx_sub) else test_p,
                        cmap=cmap_conf, alpha=0.25, vmin=0, vmax=1, rasterized=True)

        ax.set_title(f"{name} Subspace", fontsize=16, fontweight="bold", pad=10)
        ax.set_xlabel("UMAP-1", fontsize=11)
        ax.set_ylabel("UMAP-2" if ax == axes[0] else "", fontsize=11)
        ax.tick_params(labelsize=9)
        ax.set_aspect("equal", adjustable="datalim")
        for spine in ax.spines.values():
            spine.set_linewidth(0.5)
            spine.set_color("#CCCCCC")

    cbar = fig.colorbar(sc, ax=axes, shrink=0.7, aspect=30, pad=0.02)
    cbar.set_label("Meta Classifier Confidence (P(paraphrase))", fontsize=11)

    fig.suptitle("UMAP Colored by Detection Confidence",
                 fontsize=18, fontweight="bold", y=1.03)
    save_fig(os.path.join(UMAP_DIR, "umap_confidence_colored.png"))


    # ============================================================
    # 6. PLOT D: PAPER FIGURE — 3×2 GRID
    #    Top row: test set (para vs non-para)
    #    Bottom row: LLM overlay
    # ============================================================

    print(f"\n[6/6] Generating combined paper figure (3x2 grid)...")

    fig = plt.figure(figsize=(20, 13))
    gs = gridspec.GridSpec(2, 3, hspace=0.28, wspace=0.22,
                           top=0.92, bottom=0.08, left=0.04, right=0.96)

    subspace_keys = list(HEAD_NAMES.keys())

    # --- Top row: Test set ---
    for col, key in enumerate(subspace_keys):
        ax = fig.add_subplot(gs[0, col])
        emb = umap_test[key]
        name = HEAD_NAMES[key]

        mask0 = test_y == 0
        mask1 = test_y == 1
        ax.scatter(emb[mask0, 0], emb[mask0, 1], s=1, c=LABEL_COLORS[0],
                   alpha=0.12, rasterized=True)
        ax.scatter(emb[mask1, 0], emb[mask1, 1], s=1, c=LABEL_COLORS[1],
                   alpha=0.12, rasterized=True)

        ax.set_title(f"{name}", fontsize=15, fontweight="bold")
        if col == 0:
            ax.set_ylabel("Test Set\n(Para vs Non-Para)", fontsize=13, fontweight="bold")
        ax.tick_params(labelsize=8)
        ax.set_aspect("equal", adjustable="datalim")
        for spine in ax.spines.values():
            spine.set_linewidth(0.4)
            spine.set_color("#CCCCCC")

    # --- Bottom row: LLM overlay ---
    for col, key in enumerate(subspace_keys):
        ax = fig.add_subplot(gs[1, col])
        emb_test = umap_test[key]
        emb_llm = umap_llm[key]
        name = HEAD_NAMES[key]

        # Gray background
        ax.scatter(emb_test[:, 0], emb_test[:, 1], s=0.3, c="#E0E0E0",
                   alpha=0.06, rasterized=True, zorder=1)

        # LLM points
        for llm in llm_llms:
            mask = llm_labels == llm
            color = LLM_COLORS.get(llm, "#555")
            ax.scatter(emb_llm[mask, 0], emb_llm[mask, 1],
                       s=60, c=color, alpha=0.85, edgecolors="white",
                       linewidth=0.6, zorder=5)

        if col == 0:
            ax.set_ylabel("LLM Paraphrases\n(by Generator)", fontsize=13, fontweight="bold")
        ax.tick_params(labelsize=8)
        ax.set_aspect("equal", adjustable="datalim")
        for spine in ax.spines.values():
            spine.set_linewidth(0.4)
            spine.set_color("#CCCCCC")

    # --- Legends ---
    # Top row legend
    top_legend = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor=LABEL_COLORS[0],
               markersize=7, label="Non-Paraphrase"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor=LABEL_COLORS[1],
               markersize=7, label="Paraphrase"),
    ]
    fig.legend(handles=top_legend, loc="upper center", ncol=2, fontsize=11,
               frameon=True, bbox_to_anchor=(0.5, 0.96))

    # Bottom row legend
    bot_legend = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor=LLM_COLORS.get(llm, "#555"),
               markersize=8, markeredgecolor="white", markeredgewidth=0.4, label=llm)
        for llm in llm_llms
    ]
    fig.legend(handles=bot_legend, loc="lower center", ncol=len(llm_llms), fontsize=9,
               frameon=True, bbox_to_anchor=(0.5, 0.01))

    fig.suptitle("Disentangled Subspace Geometry: Test Data and Cross-LLM Paraphrase Projections",
                 fontsize=17, fontweight="bold", y=0.99)
    save_fig(os.path.join(UMAP_DIR, "umap_paper_figure_3x2.png"), dpi=300)


    # ============================================================
    # 7. EXTRA: PER-LLM DETAIL PANELS (one figure per LLM)
    # ============================================================

    print(f"\n  Generating per-LLM detail panels...")

    for llm in llm_llms:
        fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
        mask = llm_labels == llm
        color = LLM_COLORS.get(llm, "#555")

        for ax, (key, name) in zip(axes, HEAD_NAMES.items()):
            emb_test_sub = umap_test[key]
            emb_llm_sub = umap_llm[key]

            # Background test with label coloring
            m0 = test_y == 0
            m1 = test_y == 1
            ax.scatter(emb_test_sub[m0, 0], emb_test_sub[m0, 1], s=0.5,
                       c=LABEL_COLORS[0], alpha=0.06, rasterized=True, zorder=1)
            ax.scatter(emb_test_sub[m1, 0], emb_test_sub[m1, 1], s=0.5,
                       c=LABEL_COLORS[1], alpha=0.06, rasterized=True, zorder=1)

            # This LLM's points
            ax.scatter(emb_llm_sub[mask, 0], emb_llm_sub[mask, 1],
                       s=100, c=color, alpha=0.9, edgecolors="black",
                       linewidth=1.0, zorder=10, marker="*")

            # Other LLMs faintly
            other_mask = ~mask
            ax.scatter(emb_llm_sub[other_mask, 0], emb_llm_sub[other_mask, 1],
                       s=25, c="#AAAAAA", alpha=0.4, edgecolors="white",
                       linewidth=0.3, zorder=4)

            ax.set_title(f"{name}", fontsize=14, fontweight="bold")
            ax.tick_params(labelsize=8)
            ax.set_aspect("equal", adjustable="datalim")
            for spine in ax.spines.values():
                spine.set_linewidth(0.4)
                spine.set_color("#CCCCCC")

        fig.suptitle(f"Subspace Projection — {llm} Paraphrases",
                     fontsize=16, fontweight="bold", y=1.02, color=color)
        safe_name = llm.replace(" ", "_").replace(".", "")
        save_fig(os.path.join(UMAP_DIR, f"umap_detail_{safe_name}.png"))


    # ============================================================
    # 8. SAVE UMAP COORDINATES FOR INTERACTIVE EXPLORER
    # ============================================================

    print(f"\n  Saving UMAP coordinates for interactive use...")

    # Test set coordinates
    test_umap_df = pd.DataFrame({
        "umap_lex_x": umap_test["z_lex"][:, 0],
        "umap_lex_y": umap_test["z_lex"][:, 1],
        "umap_syn_x": umap_test["z_syn"][:, 0],
        "umap_syn_y": umap_test["z_syn"][:, 1],
        "umap_sem_x": umap_test["z_sem"][:, 0],
        "umap_sem_y": umap_test["z_sem"][:, 1],
        "label": test_y,
        "p_meta": test_p,
    })
    test_umap_df.to_csv(os.path.join(UMAP_DIR, "umap_coords_test.csv"), index=False)

    # LLM coordinates
    llm_umap_df = pd.DataFrame({
        "umap_lex_x": umap_llm["z_lex"][:, 0],
        "umap_lex_y": umap_llm["z_lex"][:, 1],
        "umap_syn_x": umap_llm["z_syn"][:, 0],
        "umap_syn_y": umap_llm["z_syn"][:, 1],
        "umap_sem_x": umap_llm["z_sem"][:, 0],
        "umap_sem_y": umap_llm["z_sem"][:, 1],
        "model_name": llm_labels,
        "p_meta": llm_df["p_meta"].values,
        "p_lex": llm_df["p_lex"].values,
        "p_syn": llm_df["p_syn"].values,
        "p_sem": llm_df["p_sem"].values,
    })
    llm_umap_df.to_csv(os.path.join(UMAP_DIR, "umap_coords_llm.csv"), index=False)

    print(f"\n{'=' * 60}")
    print(f"UMAP VISUALIZATION COMPLETE")
    print(f"{'=' * 60}")
    print(f"Output directory: {UMAP_DIR}")
    print(f"Files generated:")
    for f in sorted(os.listdir(UMAP_DIR)):
        size = os.path.getsize(os.path.join(UMAP_DIR, f))
        print(f"  {f:45s}  ({size/1024:.0f} KB)")

UMAP SUBSPACE VISUALIZATION SUITE

[1/6] Test set subsample: 30000 points
  LLM samples: 60 points, 6 LLMs

[2/6] Fitting UMAP projections (this may take a few minutes)...
  Fitting Lexical subspace... 

/usr/local/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


done (41.4s)
  Fitting Syntactic subspace... 

/usr/local/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


done (22.0s)
  Fitting Semantic subspace... 

/usr/local/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


done (22.2s)

[3/6] Plotting test set UMAP (para vs non-para)...
  Saved: umap_test_para_vs_nonpara.png

[4/6] Plotting LLM paraphrases UMAP (colored by generator)...
  Saved: umap_llm_by_generator.png

[5/6] Plotting confidence-colored UMAP...
  Saved: umap_confidence_colored.png

[6/6] Generating combined paper figure (3x2 grid)...
  Saved: umap_paper_figure_3x2.png

  Generating per-LLM detail panels...
  Saved: umap_detail_Claude_Haiku_35.png
  Saved: umap_detail_Claude_Opus.png
  Saved: umap_detail_Claude_Sonnet_45.png
  Saved: umap_detail_Gpt_5.png
  Saved: umap_detail_Grok_4.png
  Saved: umap_detail_Llama_4.png

  Saving UMAP coordinates for interactive use...

UMAP VISUALIZATION COMPLETE
Output directory: /mnt/heirarchy/models_stage2_hier/umap_visualizations
Files generated:
  umap_confidence_colored.png                    (3290 KB)
  umap_coords_llm.csv                            (7 KB)
  umap_coords_test.csv                           (2188 KB)
  umap_detail_Claude_Haiku_35.pn

In [14]:
# ============================================================
# UMAP ON ORIGINAL TEST DATA (DiPPer)
# ============================================================
# Plots UMAP projections of z_lex, z_syn, z_sem from the
# main test split (non-LLM paraphrase dataset).
# Saves all figures to the umap_visualizations folder.
#
# Assumes in scope from main eval suite:
#   test_main  — dict with z_lex, z_syn, z_sem, y, p, logits,
#                log_lex, log_syn, log_sem
#   cfg, main_model, thr_main, probs_from_logits_2
# ============================================================

import os
import time
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap, BoundaryNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable

try:
    import umap
    UMAP_OK = True
except ImportError:
    print("ERROR: pip install umap-learn")
    UMAP_OK = False

# ============================================================
# OUTPUT
# ============================================================

UMAP_DIR = os.path.join(cfg.model_root, "umap_visualizations")
os.makedirs(UMAP_DIR, exist_ok=True)

# ============================================================
# SUBSAMPLE — full 120k is too slow; 40k is the sweet spot
# ============================================================

MAX_PTS = 40000
UMAP_PARAMS = dict(n_neighbors=30, min_dist=0.25, metric="cosine", random_state=42)

n_total = len(test_main["y"])
if n_total > MAX_PTS:
    rng = np.random.RandomState(42)
    idx = rng.choice(n_total, MAX_PTS, replace=False)
    idx.sort()
else:
    idx = np.arange(n_total)

y   = test_main["y"][idx]
p   = test_main["p"][idx]
z   = {k: test_main[k][idx] for k in ["z_lex", "z_syn", "z_sem"]}

# Per-head probs
p_lex = probs_from_logits_2(test_main["log_lex"][idx])
p_syn = probs_from_logits_2(test_main["log_syn"][idx])
p_sem = probs_from_logits_2(test_main["log_sem"][idx])

print("=" * 60)
print("UMAP — ORIGINAL TEST DATA (DiPPer)")
print("=" * 60)
print(f"Total test:  {n_total}")
print(f"Subsampled:  {len(idx)}")
print(f"Pos rate:    {y.mean():.4f}")

HEAD = {"z_lex": "Lexical", "z_syn": "Syntactic", "z_sem": "Semantic"}

# ============================================================
# HELPER
# ============================================================

def save(path, dpi=300):
    plt.savefig(path, dpi=dpi, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"  ✓ {os.path.basename(path)}")

# ============================================================
# FIT UMAP
# ============================================================

embeddings = {}
for key, name in HEAD.items():
    t0 = time.time()
    print(f"\n  Fitting UMAP for {name} ({z[key].shape})...", end=" ", flush=True)
    reducer = umap.UMAP(**UMAP_PARAMS)
    embeddings[key] = reducer.fit_transform(z[key])
    print(f"done ({time.time()-t0:.1f}s)")


# ============================================================
# FIGURE 1: 3-PANEL  PARA VS NON-PARA
# ============================================================

print("\n[1/8] Para vs Non-Para (3-panel)...")

COL = {0: "#3498DB", 1: "#E74C3C"}

fig, axes = plt.subplots(1, 3, figsize=(21, 6.5))

for ax, (key, name) in zip(axes, HEAD.items()):
    emb = embeddings[key]
    m0, m1 = y == 0, y == 1
    # shuffle so neither class is always on top
    order = rng.permutation(len(y))
    for i in order:
        pass  # scatter handles it
    ax.scatter(emb[m0, 0], emb[m0, 1], s=1.0, c=COL[0], alpha=0.12, rasterized=True)
    ax.scatter(emb[m1, 0], emb[m1, 1], s=1.0, c=COL[1], alpha=0.12, rasterized=True)
    ax.set_title(name, fontsize=17, fontweight="bold", pad=10)
    ax.set_xlabel("UMAP-1", fontsize=11)
    if ax == axes[0]:
        ax.set_ylabel("UMAP-2", fontsize=11)
    ax.tick_params(labelsize=8)
    ax.set_aspect("equal", adjustable="datalim")
    for sp in ax.spines.values():
        sp.set_linewidth(0.4)
        sp.set_color("#CCCCCC")

legend = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=COL[0], markersize=8, label="Non-Paraphrase"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor=COL[1], markersize=8, label="Paraphrase"),
]
fig.legend(handles=legend, loc="lower center", ncol=2, fontsize=12, frameon=True,
           bbox_to_anchor=(0.5, -0.03))
fig.suptitle("UMAP Test Set — Paraphrase vs Non-Paraphrase", fontsize=19, fontweight="bold", y=1.02)
save(os.path.join(UMAP_DIR, "umap_test_label.png"))


# ============================================================
# FIGURE 2: 3-PANEL  META CONFIDENCE COLORMAP
# ============================================================

print("[2/8] Meta confidence colored...")

cmap_conf = LinearSegmentedColormap.from_list("conf",
    ["#1B2631", "#943126", "#F39C12", "#2ECC71", "#58D68D"])

fig, axes = plt.subplots(1, 3, figsize=(22, 6.5))

for ax, (key, name) in zip(axes, HEAD.items()):
    emb = embeddings[key]
    order = np.argsort(p)  # low confidence on bottom
    sc = ax.scatter(emb[order, 0], emb[order, 1], s=1.0, c=p[order],
                    cmap=cmap_conf, alpha=0.2, vmin=0, vmax=1, rasterized=True)
    ax.set_title(name, fontsize=17, fontweight="bold", pad=10)
    ax.set_xlabel("UMAP-1", fontsize=11)
    if ax == axes[0]:
        ax.set_ylabel("UMAP-2", fontsize=11)
    ax.tick_params(labelsize=8)
    ax.set_aspect("equal", adjustable="datalim")
    for sp in ax.spines.values():
        sp.set_linewidth(0.4)
        sp.set_color("#CCCCCC")

cbar = fig.colorbar(sc, ax=axes, shrink=0.7, aspect=30, pad=0.015)
cbar.set_label("Meta Classifier P(paraphrase)", fontsize=11)
fig.suptitle("UMAP Test Set — Colored by Detection Confidence", fontsize=19, fontweight="bold", y=1.02)
save(os.path.join(UMAP_DIR, "umap_test_confidence.png"))


# ============================================================
# FIGURE 3: 3-PANEL  PER-HEAD CONFIDENCE (each panel colored
#            by its OWN head probability)
# ============================================================

print("[3/8] Per-head confidence (each panel uses its own head)...")

head_probs = {"z_lex": p_lex, "z_syn": p_syn, "z_sem": p_sem}
head_cmaps = {
    "z_lex": LinearSegmentedColormap.from_list("lex", ["#1B2631", "#2471A3", "#85C1E9"]),
    "z_syn": LinearSegmentedColormap.from_list("syn", ["#1B2631", "#A93226", "#F1948A"]),
    "z_sem": LinearSegmentedColormap.from_list("sem", ["#1B2631", "#1E8449", "#82E0AA"]),
}

fig, axes = plt.subplots(1, 3, figsize=(22, 6.5))

for ax, (key, name) in zip(axes, HEAD.items()):
    emb = embeddings[key]
    hp = head_probs[key]
    order = np.argsort(hp)
    sc = ax.scatter(emb[order, 0], emb[order, 1], s=1.0, c=hp[order],
                    cmap=head_cmaps[key], alpha=0.22, vmin=0, vmax=1, rasterized=True)
    ax.set_title(f"{name} Head Score", fontsize=17, fontweight="bold", pad=10)
    ax.set_xlabel("UMAP-1", fontsize=11)
    if ax == axes[0]:
        ax.set_ylabel("UMAP-2", fontsize=11)
    ax.tick_params(labelsize=8)
    ax.set_aspect("equal", adjustable="datalim")
    for sp in ax.spines.values():
        sp.set_linewidth(0.4)
        sp.set_color("#CCCCCC")
    plt.colorbar(sc, ax=ax, shrink=0.8, aspect=25)

fig.suptitle("UMAP Test Set — Each Panel Colored by Its Own Auxiliary Head",
             fontsize=19, fontweight="bold", y=1.02)
save(os.path.join(UMAP_DIR, "umap_test_per_head_confidence.png"))


# ============================================================
# FIGURE 4: 3-PANEL  CORRECT / INCORRECT CLASSIFICATION
# ============================================================

print("[4/8] Correct vs incorrect classification...")

pred = (p >= thr_main).astype(int)
correct = pred == y

fig, axes = plt.subplots(1, 3, figsize=(21, 6.5))

for ax, (key, name) in zip(axes, HEAD.items()):
    emb = embeddings[key]
    # Correct first (faint), incorrect on top (bright)
    ax.scatter(emb[correct, 0], emb[correct, 1], s=0.8, c="#2ECC71",
               alpha=0.07, rasterized=True, label="Correct")
    ax.scatter(emb[~correct, 0], emb[~correct, 1], s=6, c="#E74C3C",
               alpha=0.6, rasterized=True, label="Misclassified",
               edgecolors="white", linewidth=0.3)
    ax.set_title(name, fontsize=17, fontweight="bold", pad=10)
    ax.set_xlabel("UMAP-1", fontsize=11)
    if ax == axes[0]:
        ax.set_ylabel("UMAP-2", fontsize=11)
    ax.tick_params(labelsize=8)
    ax.set_aspect("equal", adjustable="datalim")
    for sp in ax.spines.values():
        sp.set_linewidth(0.4)
        sp.set_color("#CCCCCC")

n_wrong = int((~correct).sum())
legend = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#2ECC71", markersize=7,
           label=f"Correct ({len(y)-n_wrong:,})"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#E74C3C", markersize=9,
           markeredgecolor="white", markeredgewidth=0.5,
           label=f"Misclassified ({n_wrong:,})"),
]
fig.legend(handles=legend, loc="lower center", ncol=2, fontsize=12, frameon=True,
           bbox_to_anchor=(0.5, -0.03))
fig.suptitle("UMAP Test Set — Correct vs Misclassified Predictions",
             fontsize=19, fontweight="bold", y=1.02)
save(os.path.join(UMAP_DIR, "umap_test_errors.png"))


# ============================================================
# FIGURE 5: DECISION MARGIN COLORMAP
# ============================================================

print("[5/8] Decision margin colored...")

margin = p - thr_main  # positive = predicted para, negative = predicted non-para

cmap_margin = LinearSegmentedColormap.from_list("margin",
    ["#2E86C1", "#AED6F1", "#FDFEFE", "#F5B7B1", "#E74C3C"])

fig, axes = plt.subplots(1, 3, figsize=(22, 6.5))

for ax, (key, name) in zip(axes, HEAD.items()):
    emb = embeddings[key]
    order = np.argsort(np.abs(margin))  # low margin on top (most interesting)
    sc = ax.scatter(emb[order, 0], emb[order, 1], s=1.2, c=margin[order],
                    cmap=cmap_margin, alpha=0.22, vmin=-0.5, vmax=0.5, rasterized=True)
    ax.set_title(name, fontsize=17, fontweight="bold", pad=10)
    ax.set_xlabel("UMAP-1", fontsize=11)
    if ax == axes[0]:
        ax.set_ylabel("UMAP-2", fontsize=11)
    ax.tick_params(labelsize=8)
    ax.set_aspect("equal", adjustable="datalim")
    for sp in ax.spines.values():
        sp.set_linewidth(0.4)
        sp.set_color("#CCCCCC")

cbar = fig.colorbar(sc, ax=axes, shrink=0.7, aspect=30, pad=0.015)
cbar.set_label(f"Decision Margin (p − τ, τ={thr_main:.3f})", fontsize=11)
fig.suptitle("UMAP Test Set — Decision Margin (Blue=Non-Para, Red=Para, White=Boundary)",
             fontsize=18, fontweight="bold", y=1.02)
save(os.path.join(UMAP_DIR, "umap_test_margin.png"))


# ============================================================
# FIGURE 6: HEAD DISAGREEMENT MAP
# Which samples have the biggest gap between head predictions?
# ============================================================

print("[6/8] Head disagreement map...")

head_stack = np.stack([p_lex, p_syn, p_sem], axis=1)
head_range = head_stack.max(axis=1) - head_stack.min(axis=1)  # max gap across heads

cmap_disagree = LinearSegmentedColormap.from_list("disagree",
    ["#1A5276", "#F4D03F", "#E74C3C"])

fig, axes = plt.subplots(1, 3, figsize=(22, 6.5))

for ax, (key, name) in zip(axes, HEAD.items()):
    emb = embeddings[key]
    order = np.argsort(head_range)  # high disagreement on top
    sc = ax.scatter(emb[order, 0], emb[order, 1], s=1.2, c=head_range[order],
                    cmap=cmap_disagree, alpha=0.25, vmin=0, vmax=1, rasterized=True)
    ax.set_title(name, fontsize=17, fontweight="bold", pad=10)
    ax.set_xlabel("UMAP-1", fontsize=11)
    if ax == axes[0]:
        ax.set_ylabel("UMAP-2", fontsize=11)
    ax.tick_params(labelsize=8)
    ax.set_aspect("equal", adjustable="datalim")
    for sp in ax.spines.values():
        sp.set_linewidth(0.4)
        sp.set_color("#CCCCCC")

cbar = fig.colorbar(sc, ax=axes, shrink=0.7, aspect=30, pad=0.015)
cbar.set_label("Head Disagreement (max − min across lex/syn/sem)", fontsize=11)
fig.suptitle("UMAP Test Set — Auxiliary Head Disagreement",
             fontsize=19, fontweight="bold", y=1.02)
save(os.path.join(UMAP_DIR, "umap_test_head_disagreement.png"))


# ============================================================
# FIGURE 7: INDIVIDUAL SUBSPACE LARGE PLOTS (one per head)
# ============================================================

print("[7/8] Individual large subspace plots...")

for key, name in HEAD.items():
    emb = embeddings[key]
    fig, ax = plt.subplots(figsize=(10, 9))

    m0, m1 = y == 0, y == 1

    # Interleave classes for fair rendering
    order = rng.permutation(len(y))
    for i in order:
        c = COL[int(y[i])]
        ax.scatter(emb[i, 0], emb[i, 1], s=1.5, c=c, alpha=0.15, rasterized=True)

    ax.set_title(f"{name} Subspace — Test Set", fontsize=20, fontweight="bold", pad=12)
    ax.set_xlabel("UMAP-1", fontsize=13)
    ax.set_ylabel("UMAP-2", fontsize=13)
    ax.tick_params(labelsize=10)
    ax.set_aspect("equal", adjustable="datalim")
    for sp in ax.spines.values():
        sp.set_linewidth(0.5)
        sp.set_color("#CCCCCC")

    legend = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor=COL[0], markersize=9,
               label=f"Non-Paraphrase ({int(m0.sum()):,})"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor=COL[1], markersize=9,
               label=f"Paraphrase ({int(m1.sum()):,})"),
    ]
    ax.legend(handles=legend, fontsize=12, loc="upper right", framealpha=0.9)
    save(os.path.join(UMAP_DIR, f"umap_test_{key}_large.png"))


# ============================================================
# FIGURE 8: COMBINED PAPER FIGURE (2×3 grid)
#   Row 1: para vs non-para
#   Row 2: per-head confidence
# ============================================================

print("[8/8] Combined 2×3 paper figure...")

fig = plt.figure(figsize=(21, 13.5))
gs = gridspec.GridSpec(2, 3, hspace=0.25, wspace=0.2,
                       top=0.93, bottom=0.06, left=0.04, right=0.96)

# --- Row 1: label coloring ---
for col, (key, name) in enumerate(HEAD.items()):
    ax = fig.add_subplot(gs[0, col])
    emb = embeddings[key]
    m0, m1 = y == 0, y == 1
    ax.scatter(emb[m0, 0], emb[m0, 1], s=0.8, c=COL[0], alpha=0.10, rasterized=True)
    ax.scatter(emb[m1, 0], emb[m1, 1], s=0.8, c=COL[1], alpha=0.10, rasterized=True)
    ax.set_title(name, fontsize=16, fontweight="bold")
    if col == 0:
        ax.set_ylabel("Para vs Non-Para", fontsize=13, fontweight="bold")
    ax.tick_params(labelsize=7)
    ax.set_aspect("equal", adjustable="datalim")
    for sp in ax.spines.values():
        sp.set_linewidth(0.3)
        sp.set_color("#CCC")

# --- Row 2: per-head confidence ---
for col, (key, name) in enumerate(HEAD.items()):
    ax = fig.add_subplot(gs[1, col])
    emb = embeddings[key]
    hp = head_probs[key]
    order = np.argsort(hp)
    sc = ax.scatter(emb[order, 0], emb[order, 1], s=0.8, c=hp[order],
                    cmap=head_cmaps[key], alpha=0.18, vmin=0, vmax=1, rasterized=True)
    ax.set_title(f"{name} Head Score", fontsize=14, fontweight="bold")
    if col == 0:
        ax.set_ylabel("Aux. Head Confidence", fontsize=13, fontweight="bold")
    ax.tick_params(labelsize=7)
    ax.set_aspect("equal", adjustable="datalim")
    for sp in ax.spines.values():
        sp.set_linewidth(0.3)
        sp.set_color("#CCC")

# Top legend
top_leg = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=COL[0], markersize=7, label="Non-Paraphrase"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor=COL[1], markersize=7, label="Paraphrase"),
]
fig.legend(handles=top_leg, loc="upper center", ncol=2, fontsize=11,
           frameon=True, bbox_to_anchor=(0.5, 0.97))

fig.suptitle("Disentangled Subspace Geometry — Test Set (DiPPer)",
             fontsize=19, fontweight="bold", y=1.0)
save(os.path.join(UMAP_DIR, "umap_test_paper_figure_2x3.png"))


# ============================================================
# SAVE COORDINATES
# ============================================================

print("\n  Saving UMAP coordinates to CSV...")

coord_df = pd.DataFrame({
    "label": y,
    "p_meta": p,
    "p_lex": p_lex,
    "p_syn": p_syn,
    "p_sem": p_sem,
    "pred": pred,
    "correct": correct.astype(int),
    "margin": margin,
    "head_disagreement": head_range,
})
for key in HEAD:
    coord_df[f"{key}_umap1"] = embeddings[key][:, 0]
    coord_df[f"{key}_umap2"] = embeddings[key][:, 1]

coord_df.to_csv(os.path.join(UMAP_DIR, "umap_test_coordinates.csv"), index=False)

# ============================================================
# SUMMARY
# ============================================================

print(f"\n{'='*60}")
print("UMAP TEST SET VISUALIZATION COMPLETE")
print(f"{'='*60}")
print(f"Directory: {UMAP_DIR}")
files = sorted(f for f in os.listdir(UMAP_DIR) if f.startswith("umap_test"))
print(f"Test-set files ({len(files)}):")
for f in files:
    sz = os.path.getsize(os.path.join(UMAP_DIR, f))
    print(f"  {f:50s} {sz/1024:>7.0f} KB")

print(f"\nStats:")
print(f"  Points plotted:      {len(idx):,}")
print(f"  Misclassified:       {n_wrong:,} ({n_wrong/len(idx)*100:.2f}%)")
print(f"  Mean head disagree:  {head_range.mean():.4f}")
print(f"  Threshold used:      {thr_main:.4f}")

UMAP — ORIGINAL TEST DATA (DiPPer)
Total test:  120000
Subsampled:  40000
Pos rate:    0.6648

  Fitting UMAP for Lexical ((40000, 256))... 

/usr/local/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


done (30.7s)

  Fitting UMAP for Syntactic ((40000, 256))... 

/usr/local/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


done (30.2s)

  Fitting UMAP for Semantic ((40000, 256))... 

/usr/local/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


done (30.2s)

[1/8] Para vs Non-Para (3-panel)...
  ✓ umap_test_label.png
[2/8] Meta confidence colored...
  ✓ umap_test_confidence.png
[3/8] Per-head confidence (each panel uses its own head)...
  ✓ umap_test_per_head_confidence.png
[4/8] Correct vs incorrect classification...
  ✓ umap_test_errors.png
[5/8] Decision margin colored...
  ✓ umap_test_margin.png
[6/8] Head disagreement map...
  ✓ umap_test_head_disagreement.png
[7/8] Individual large subspace plots...
  ✓ umap_test_z_lex_large.png
  ✓ umap_test_z_syn_large.png
